# Wyckoff liquidity-sweep scalping on 1-minute ETF bars: rules vs. ML meta-labeling

**Question.** In an established trend, does a Wyckoff-style shake-out inside a pullback carry a tradable edge after realistic costs on liquid US ETFs? The shake-out is a sweep of a recent swing low (high) that is immediately reclaimed and confirmed by volume and order-flow **proxies**. And does an ML meta-labeler improve trade selection over the rules-only version?

**How to read this notebook.** §1 derives the rules from the source text before any code. §2–§6 build the data, features, signals and engine with lookahead and memory constraints. §7 searches parameters on IS data only, logging every trial. §8 fits the meta-labeler. §9 touches OOS once per split. §10 runs the overfitting and integrity checks, each with a PASS/FAIL. §11–§12 break results down by VIX regime and macro-event context. §13 concludes, and its last cell audits every number quoted in the conclusions against the outputs.

**Running it.** Use the `python3` kernel (anaconda base) with pandas, numpy, numba, pyarrow, scipy, scikit-learn, xgboost, optuna, shap, plotly, yfinance, fredapi and psutil. Optional: set `FRED_API_KEY` in the environment for §12; the key is read from the environment and never stored. `WYCKOFF_FAST=1` gives a smoke run (3 ETFs, tiny budgets). `WYCKOFF_REBUILD=1` rebuilds all caches. Intermediates and all historical data live in `cache/`; only cited sources live in `artifacts/`.

**Honesty rule.** A negative result is a valid outcome. Thresholds for every verdict are fixed in the code *before* results are seen, and the strategy is not re-tuned after OOS evaluation.

## 1 Research summary: Wyckoff's course as design input

This section was written **before any strategy code**. Six readers each extracted one page range of the 405-page PDF (`artifacts/wyckyoff_method_book.pdf`), and the extractions were merged into the summary below. Page numbers are PDF pages. The literature tables come from separate searches, and every citation was then checked by an independent adversarial verifier.

### 1.1 Source note

**What the text is.** This is Richard D. Wyckoff's *Course of Instruction in Stock Market Science and Technique*, Sections 1M–25M (about 405 PDF pages). Its worked examples run from 1929 to 1937: the NYT 50-stock average, Bethlehem Steel, U.S. Steel, Anaconda, Electric Power & Light and others. It is built on daily vertical (bar) charts with volume, figure (point-and-figure) charts, and one intraday tool, the Wave Chart of Tape Readings (22M, pp. 312–350), which compares buying and selling waves that last minutes. Every price unit is a point on a daily bar of a stock or average. Many chart plates have no OCR text (PDF 83–85, 102–110, 118–120, 193–200, 231–234, 254–260, 343–349, 351), so anything shown only in figures is not covered here.

Section map:
- 2M: supply and demand
- 4M–6M: chart types and waves
- 7M: the trend chart of the averages
- 8M: group strength
- 9M–10M: the operator's campaign
- 11M–13M: figure charts
- 14M: volume
- 15M: trend lines
- 16M–17M: vertical-chart case studies
- 18M–19M: Position Sheet and technical position
- 20M: Buying and Selling Tests
- 21M: Refinements (terminal shake-out)
- 22M: Wave Chart
- 23M: stop orders
- 24M–25M: general instructions and market philosophy

**Modern Wyckoff vocabulary vs this text**

| Modern term | In this text? | The book's own term or nearest equivalent | PDF pages |
|---|---|---|---|
| Phases A–E | **No** | Campaign Phases I–IV (accumulation, marking up, distribution, marking down). Otherwise "phase" is used loosely. | 201–202, 219 |
| Accumulation/distribution schematics #1/#2 | **No** | Behavioural narratives in 7M, 9M–10M, 16M–17M and 21M | 52–60, 112–125, 201–253, 302–306 |
| PS / PSY | As a phrase only | "preliminary support" (Buying Test 3); "preliminary supply" (Selling Test 3) | 136, 138, 188, 206, 250, 296–298 |
| SC | As a phrase only | "selling climax", "minor selling climax" | 42, 52–53, 57, 67, 168, 188–190, 204–205, 286–287, 297, 303 |
| BC | As a phrase only | "buying climax"; activity of "climactic intensity" | 56–59, 72, 81–82, 167, 208–209, 286, 290, 336 |
| AR | **No** | "technical rally" or "technical rebound" after a climax | 53, 188, 205, 225 |
| ST | **No** | "secondary reaction" that tests the climax low or preliminary support; "test of the support" | 53–55, 71, 79–81, 188, 206, 208 |
| Spring | **No** (only "springboard") | "shake-out", "terminal shake-out" (defined in 21M); a quick dip to support that brings out little stock | 137–139, 158, 205, 218–220, 291, 302–304 |
| UT / UTAD | "upthrust" yes; UTAD **no** | a failed "upthrust", "thrust movement", "terminal thrust", "terminal mark-up", bidding up to a new high to catch shorts | 59, 73–74, 144, 211, 250–252, 289–290, 304, 336 |
| SOS / SOW | **No** | Rally through the supply line or old tops on expanding volume and spread; a heavy-volume reaction that is "breaking the backbone"; the Buying/Selling Tests | 58–62, 222, 243, 296–299, 305, 332 |
| LPS / LPSY | **No** (LPS appears once, in a generic sense, p. 223) | Light-volume dip to support, "second springboard"; rally to a lower top, secondary distribution | 63–65, 124, 140, 161, 222, 336–338 |
| Creek / ice / JAC / BUEC | **No** | Supply and support lines; breakout through old tops; a "second springboard" dip that holds above the range | 140, 180–190, 222, 243 |
| Composite Man | Yes | "Composite Man" (9M), "Composite Operator" (11M) | 112–115, 135 |
| Law of Supply and Demand | **Yes, named** | Sect. 2M; footnote in 11M | 9–11, 134–135, 166 |
| Law of Effort vs Result | **No, the idea only** | "braking power"; price progress "in proportion" to volume | 67–70, 130, 175–176, 217, 223–224, 248, 335 |
| Law of Cause and Effect | **No, the idea only** | Figure-chart horizontal counts; length of the preparation period | 23, 123–125, 202–203, 236–252, 299, 337–338 |
| Nine buying/selling tests | Yes | Sect. 20M | 296–299 |

**Policy.**
- **The book is the primary source.** The Phase A–E schematics and their labelled events (PS, SC, AR, ST, Spring, LPS, SOS, BU / PSY, BC, ST, UTAD, LPSY, SOW), and the "three laws" framing, are not in this text. They come from later Wyckoff-school teaching, for example Stock Market Institute course material and authors such as Hank Pruden, David Weis, Bruce Fraser and Roman Bogomazov. We have not traced which author introduced which label.
- **Modern names are code labels only.** Each rule below is defined by the book behaviour it cites. Where the book is more specific than the schematics, the book wins. Two examples: low volume means different things depending on where it occurs, and abnormal volume on a breakout means continuation while abnormal volume after an extended move means exhaustion.
- **No mechanical rules.** The book warns against them (pp. 14, 18, 23, 51, 396, 403). So every threshold is a tunable parameter, robustness is tested walk-forward across ETFs and regimes, and the ML meta-labeler stands in for the "judgment" the book asks for.
- **The 1-minute transfer is a hypothesis.** The book says the same principles hold at every scale (pp. 53, 57) and applies them to waves of minutes (pp. 312–320, 331–333). Whether they carry over to 1-minute ETF bars is an empirical question, not a claim the book makes.

### 1.2 Concept summaries

**Supply and demand, and trend.**
- The one named law: price rises when demand exceeds supply, falls when supply exceeds demand, and moves sideways in a narrow range when the two are balanced (pp. 9–11, 134).
- The trend is the line of least resistance. Every market is bullish, bearish or neutral. Trade in harmony with the trend and stand aside when it is in doubt (pp. 11, 18, 36, 55–56, 265, 283–285, 390–392).
- Positions 1–4 encode a minor or major swing, up or down, plus Neutral (pp. 264–267, 281–283).
- All judgments use four facts: price movement, volume, the relation between them, and time (pp. 15–16, 42–43).
- Swings are built from buying and selling waves, compared by extent, duration, speed and volume (pp. 44–47, 312–320).

**Accumulation, distribution and phases as the book presents them.**
- Campaign Phases I–IV (p. 201). The operator accumulates in a weak, dull market. He forces the price down to the bottom of the range, takes the stock that comes out, and lets the price advance only once the floating supply is gone (pp. 112–115, 170–172).
- Signs of accumulation:
  - support holds again and again while volume shrinks on each reaction (pp. 121–124);
  - supports rise, or bottoms round upward (pp. 129, 138–140, 213–214, 293);
  - up-waves carry more volume than down-waves (pp. 57–58, 189);
  - swings narrow into a hinge or apex (pp. 127–128, 210, 286, 298);
  - prolonged dullness tires out holders (pp. 204–205, 214, 241).
- An explicit three-part test separates absorption from distribution: light volume on reactions, a narrowing range instead of a deep reaction, and volume building while supports rise (pp. 57–58).
- Signs of distribution: churning on heavy volume without progress, heavier volume on reactions, lower rally tops, failed new highs, and wide, erratic swings propped up by "hypodermics" (pp. 115–117, 130–132, 143, 157–161, 223–224, 248–251, 334–338).
- A long range is not proof of either. Many ranges are equilibrium zones (p. 183), and Safeway is the cautionary false base (p. 306).
- **The bottom sequence as the book narrates it** (pp. 53–59, 71, 79–81, 136, 188–190, 204–222, 296–304):
  1. an oversold acceleration;
  2. selling climax and preliminary support;
  3. technical rally, normally no more than half the decline, on shrinking volume;
  4. secondary reaction on lower volume that holds at or above the climax low;
  5. a range with rising supports and dullness;
  6. possibly a (terminal) shake-out;
  7. a light-volume dip or hinge, then the springboard;
  8. mark-up on expanding volume, with resting spells and "second springboard" dips.
- **The top sequence** (pp. 56–62, 67, 76, 90, 157–161, 223–224, 248–253, 334–338):
  1. a run-up ("whooping up");
  2. buying climax;
  3. a range with churning and failed upthrusts;
  4. a heavy-volume break;
  5. a rally to a lower top (secondary distribution);
  6. a liquidating mark-down.
- Mapping Phases A–E onto these sequences is our own convention. The book stresses that timing, size, order and volume vary from case to case (p. 57).

**Springs, which the book calls shake-outs.**
- A shake-out is a quick sell-off under a well-established line of supports that is recovered promptly (pp. 137–139).
- A **terminal shake-out** (21M, pp. 302–304) is a rapid, steep drop near the end of preparation. It frightens holders, trips the stops under the support line, picks up cheap stock and traps shorts. The recovery can be rapid or gradual, and it reads as poor-quality supply meeting good-quality demand.
- Confirmation: a later dip to support holds and is followed by a higher rebound (p. 138), or higher lows, tops and closes after price re-enters the range (pp. 218–220).
- The book refuses to trust a shake-out while the general market is acutely weak (p. 218).
- Small undercuts are still counted as support holding (29 7/8 under a 30 line, p. 122).
- 11M gives no volume rule for shake-outs. 21M implies an accelerating, frightened sell-off, while the springboard dip "brings out little stock" (p. 291). So two volume patterns are both plausible.

**Upthrusts.**
- The word first appears descriptively on p. 59: "a quick upthrust to a new high on such a heavy turnover" (p. 59).
- Related forms:
  - bidding up to a new high on a narrower range to catch shorts (pp. 73–74);
  - running a point above old tops to catch stops (pp. 128–131, 143, 148);
  - "terminal thrust" (p. 144);
  - "thrust movement" or "terminal mark-up", the reverse of a shake-out (pp. 289–290, 304).
- Failed upthrusts come in two volume patterns: very small volume (p. 211), or heavy volume with a low close (pp. 223, 250–252, 336).
- "Upthrust" also simply means any upward push, as in "shortening of the upthrusts" (pp. 72–73, 141).
- Not every poke above old tops is false. After rising supports, one led to a mark-up (p. 128).

**Tests.**
- A secondary reaction after a selling climax that comes on clearly smaller volume and holds at or above the climax low shows liquidation is complete (pp. 53–55, 71, 79–81, 188, 206, 208). Rising volume and no hold means the test failed (pp. 79–81).
- At a critical support level, shrinking volume plus a quick rebound passes. No rebound means a slump follows (p. 136).
- An almost complete drying-up of volume on a dip to the same low, with a close near the high, ends the reaction (pp. 57–58).
- The final dip, on lighter volume than any earlier reaction to that level, is the buy point. It is confirmed when volume rises on the recovery (p. 124).
- Failure to follow through after a new low (pp. 287–289).
- Low volume is bearish in a downtrend (pp. 75–78, 211–212, 240), so the meaning of a test depends on context.

**Signs of strength and weakness.** The book never names these as events.
- Strength:
  - a rally through the supply line or old tops on expanding volume and spread (pp. 58–59, 222, 243, 247, 305, 332);
  - higher supports and tops (Buying Tests 6–7, p. 296);
  - reactions smaller than half (pp. 126, 169–170);
  - closes near the high;
  - strength relative to the market (pp. 86–101, 296).
- Weakness:
  - a heavy-volume reaction that is "breaking the backbone" (pp. 61–62);
  - the first high-volume reaction in a trend (p. 60);
  - volume rising on reactions (pp. 211, 213);
  - failure to rally where a rally is due (p. 213);
  - rallies smaller than half (pp. 141, 150, 160);
  - lower tops (p. 132).
- The book's own checklist is the nine Buying and nine Selling Tests (pp. 296–299).

**Buying and selling climaxes.**
- **Selling climax:** an accelerating decline, a widening range and abnormal volume (about 2.5x in the 7M example), with much of the volume at and after the low, followed by a technical rally (pp. 42, 52–53, 168, 204–205, 286–287, 297). It is panic selling by untrained holders, absorbed by operators (p. 92).
- **Failed climax:** the supporting stock is thrown back on the market and a break of the climax low resumes liquidation (pp. 53–54, 67).
- **Buying climax:** an abrupt volume jump into a new high that closes near the low and below the prior close (pp. 56–57), usually after a steep run (pp. 72, 208–209, 286, 290, 292).
- **Key principle:** abnormal volume after an extended move marks its end, but abnormal volume as price breaks out of a defined range means continuation (pp. 65–66).
- Which kind of climax it was is only clear from the bars that follow (pp. 63, 65, 68, 228).
- Minor climaxes appear at reaction lows inside trends (pp. 57, 189, 219, 242).

**Effort vs result.** Never named as a law, but used throughout.
- Heavy volume without progress means supply is overcoming demand (pp. 58–60, 130, 223–224, 248, 250, 335). The book calls this "braking power" (pp. 175–176).
- Downward progress slackening on heavy volume means buying (pp. 67–68, 287).
- Progress in proportion to volume is healthy, and at resistance it means absorption rather than a climax (pp. 69–72, 222, 228–229, 243).
- Large progress on light volume means supply is scarce (pp. 221, 242, 296).

**Volume confirming or contradicting price.**
- Volume rising on rallies and shrinking on reactions is bullish; the reverse is bearish (pp. 38, 55, 64, 66, 126, 173–174, 296, 328). An abnormal surge is the exception: it marks a culmination (p. 328).
- Relative change matters, not absolute size (pp. 175–176). Declines normally trade less volume than advances (p. 64).
- Expanding volume into support predicts a break (p. 65).
- Short sessions are normalized (pp. 19, 59, 225), and the book shows raw and normalized values side by side (pp. 288–289).
- The Wave Chart compares waves by volume per unit of time (pp. 331–332).

**Stop placement and targets.**
- Find the danger point, place the stop as soon as the order fills, and require probable profit of at least 3:1 to the stop (pp. 266, 284, 297, 352–353).
- Enter close to the danger point (pp. 55, 352–353).
- Longs: a fraction to a point or two under the shake-out, climax or support low (pp. 42, 54–55, 124, 139, 220). Shorts: 1–3 points above the highs (pp. 60, 73–74, 131).
- Avoid round figures, and go 1 3/8 to 2 5/8 points past obvious levels, at odd fractions (pp. 355, 363–365).
- Never widen a stop. Move to breakeven, and tighten near the objective or on warning signs (pp. 355–359). Allow for the normal 3–5 point counter-move after a first thrust of about 10 points (p. 360).
- Frequent stop-outs mean you are out of step with the trend (pp. 366–368).
- Targets: P&F counts, which the book calls rough (p. 125); the halfway "normal" recovery (pp. 62, 225); and overhead supply (pp. 204–205, 216–217, 333).

### 1.3 Testable rules on 1-minute bars

Notation:
- Bars `t` are 1-minute RTH bars with fields `O H L C V`.
- `ATR` is Wilder ATR(14) on 1-minute bars; `ATR_H` is the same on the higher-timeframe (HTF) bars.
- `RV` is relative volume (P3), `CLV01` is the close location value (P1), and `R = H - L`.
- Legs come from a causal zigzag with reversal `z*ATR`.
- `SL*` / `SH*` are reference swing lows/highs: confirmed pivots, clusters, prior-day high/low or opening-range high/low.
- `s` is the sweep bar, `r` the reclaim bar, and `E = O[r+1]` the entry, filled at the next bar's open. All thresholds on `CLV01` are on the [0, 1] scale.

| Wyckoff concept | Book basis (PDF pp.) | Testable 1m-bar rule | Parameters (search range) | OHLCV proxy caveat |
|---|---|---|---|---|
| **Trend (higher timeframe)** | 11, 18, 36, 45–47, 55–56, 264–267, 281–283, 366–368, 393, 402–404 | Resample to the HTF and use completed bars only. `UP` when the last K HTF pivot highs and pivot lows are both rising; `DOWN` is the mirror; otherwise `NEUTRAL`, which means no new trades. Hysteresis: `UP` holds until an HTF close falls below the last HTF pivot low by ε·ATR_H. Optionally require the SPY/QQQ/IWM/DIA composite to be in the same state (pp. 264, 404). | HTF ∈ {5, 15, 30} min; K ∈ {2, 3}; z_H ∈ [1.5, 3]·ATR_H; ε ∈ [0, 0.3]; composite filter on/off | Pivots confirm with a lag, so use the state as of the last completed HTF bar. Overnight gaps break swing continuity, so decide whether prior-session pivots count. |
| **Pullback with volume drying up** | 38, 55, 57–58, 62, 64, 126, 169–170, 173–174, 209–210, 289, 296–299 | In `UP`, the impulse leg runs from pivot low P0 to pivot high H0 and the pullback is the next down-leg. Require `retr = (H0 - minL)/(H0 - P0) <= R_max` and `PV = meanRV(pullback)/meanRV(impulse) <= q`. Optionally require the RV slope across the pullback to be ≤ 0. The book treats about half as a normal reaction; shallower means strength. | z ∈ [1, 3]·ATR; R_max ∈ [0.38, 0.75]; q ∈ [0.5, 0.9]; minimum pullback bars ∈ [3, 10] | Without minute-of-day normalization, any pullback through the midday lull looks "dry". Declines normally carry less volume than advances (p. 64), so calibrate q per instrument. |
| **Spring / shake-out (sweep below a swing low, then reclaim)** | 69, 78–79, 122, 137–139, 158, 205, 214, 218–220, 291, 297, 302–304 | `SL*` is the latest confirmed pivot low, a cluster of ≥ n_touch lows within tol·ATR, or the prior-day or opening-range low, all within lookback Lb. Sweep: `SL* - d_max*ATR <= L_s <= SL* - d*ATR`. Reclaim within m bars: `C_r > SL* + e*ATR`, with no close below L_s in between. The event fires at the close of r. Tag the sweep volume as `climactic` (RV_s ≥ v_hi; frightened selling and stops run, p. 303) or `quiet` (RV_s ≤ v_lo; little stock comes out, p. 291). Shorts are the mirror. | d ∈ [0.05, 0.5]·ATR; d_max ∈ [1, 3]·ATR; m ∈ {0..5}; e ∈ [0, 0.2]·ATR; Lb ∈ [30, 390]; n_touch ∈ {1, 2, 3}; v_hi ∈ [1.5, 3]; v_lo ∈ [0.5, 0.9] | The book counts tiny undercuts as support holding (p. 122), so d must be tuned, not set to zero. With m = 0 the order is known (the low comes before the close); with m > 0 the path inside each bar is unknown. The book's shake-outs are daily or multi-day and 11M gives no volume rule, so leave the volume tag to the meta-labeler. |
| **Upthrust (mirror)** | 59, 73–74, 128–131, 143–144, 211, 215, 223, 248–252, 289–290, 304, 336, 340 | Sweep: `SH* + d*ATR <= H_s <= SH* + d_max*ATR`. Then `C_r < SH* - e*ATR` within m bars, with `CLV01_r <= c_u`. Volume tag: `no-demand` (RV_s < v_lo, p. 211) or `supply` (RV_s ≥ v_hi with a small result, pp. 252, 336). Use it as a short trigger in `DOWN`, and as a long-exit / no-new-long flag in `UP`. Suppress it when the last k pivot lows are rising and price holds above SH* for ≥ h bars (p. 128). | as for the spring; c_u ∈ [0.2, 0.4]; h ∈ [3, 15]; k ∈ {2, 3} | The book also uses "upthrust" for any upward push (pp. 72–73, 141); only the failed version is this event. Same caveats on the path inside each bar. |
| **Test of the spring (low-volume retest)** | 53–55, 57–58, 71, 79–81, 124, 136, 186, 188, 206, 208, 287–289, 291 | After the event at r, look for a down-leg within N_t bars with its low `L_t` in `[L_s - tol*ATR, SL* + u*ATR]` (L_t > L_s preferred). Require `meanRV(test) <= v_t*RV_s` and `<= q_t*meanRV(reclaim leg)`, and `mean(R/ATR)` over the test leg `<= c_r`. **Entry B:** the open after the first bar that closes above the test leg's high. **Fail:** a close below `L_s - ε*ATR` invalidates the setup; optionally reverse (p. 362). | N_t ∈ [3, 60]; tol ∈ [0, 0.3]; u ∈ [0, 1]; v_t ∈ [0.3, 0.8]; q_t ∈ [0.4, 0.9]; c_r ∈ [0.5, 1.0] | In the book the test comes sessions later; on 1-minute bars it often never comes. Label both entry A (at E) and entry B, and let the meta-labeler choose. |
| **Effort vs result at the sweep** | 67–68, 130, 175–176, 217, 222–224, 228–229, 248, 287, 318, 335 | Over the cluster [s, r]: effort `EF = sum(RV)`, extension `RS = (SL* - minL)/ATR`, net progress `NP = (C_r - C_{s-1})/ATR`. **Absorption:** percentile of EF ≥ p_hi, RS ≤ x and NP ≥ 0 (downward progress slackening on heavy volume). **Shortening thrust:** `RS_now < s_th * RS_prev_push` (p. 287). At highs, the mirror is bearish. | p_hi ∈ [70, 90]th pct; x ∈ [0.2, 1.0]·ATR; s_th ∈ [0.4, 0.8]; percentile window ∈ [5, 20] sessions | Not a named law in the book, and its examples are daily bars and waves. A single 1-minute bar is noisy and quantized, so compute this on the cluster and on 3- and 5-minute aggregates, as percentile ranks. |
| **Climactic vs drying-up volume** | 42, 52–53, 63–68, 71, 75–77, 92, 168, 171–176, 204–205, 211–212, 228, 240, 286–287 | **Climax candidate:** `RV >= v_c` after a leg ≥ D·ATR with no range edge crossed (exhaustion). If the bar instead closes beyond a defined range edge, tag it **breakout volume** (continuation, pp. 65–66). Classify after k bars: bullish if no lower close and higher lows follow. **Drying up:** `meanRV(n) <= v_d`. Treat low volume as bullish only at tested support and only when the market is not acutely weak. | v_c ∈ [2.5, 6]; D ∈ [3, 10]·ATR; k ∈ [3, 15]; v_d ∈ [0.4, 0.8]; n ∈ [3, 15] | The first 5–15 and last 5–10 minutes have structural volume spikes, and scheduled releases (10:00, 14:00 FOMC) produce spikes that are not Wyckoff behaviour. Give those minutes their own baselines or exclude them. |
| **Close location / wick absorption** | 19, 54, 56–59, 63, 186–187, 226–227, 252, 286 | Reclaim bar `CLV01_r >= c_l` (a close near the high ends the reaction, pp. 57–58, 187). Sweep bar `LW_s/R_s >= w`, plus a volume-weighted rejection `RV_s*LW_s/R_s`. Bearish mirror: an N-bar high with `CLV01 <= 0.3` and `C < C_prev` (pp. 56–57). | c_l ∈ [0.5, 0.8]; w ∈ [0.3, 0.7]; aggregation ∈ {1, 3, 5} min | How much volume traded in the wick is unknown. Liquid-ETF 1-minute ranges are only a few ticks, so CLV takes few values; set it to NaN when R < 2 ticks. The book reads daily closes, so also compute it on the aggregated cluster. |
| **Sign of strength after the reclaim** | 58–59, 124, 220–222, 228–229, 243, 247, 287–288, 291, 296–298, 305, 332 | Within N_s bars after r, a close above the pullback's last minor pivot high (the downward stride broken, Buying Test 5), on an up-leg with `meanRV(up) >= g_v*meanRV(sweep or test leg)` and `mean(R/ATR) >= g` (progress in proportion to volume; volume rising on the recovery, p. 124). | N_s ∈ [3, 30]; g_v ∈ [1.0, 2.0]; g ∈ [1.0, 1.8] | SOS as a named event comes from later authors; here it is a confirmation feature. Waiting for it raises the entry price and lowers reward:risk. |
| **Stop beyond the sweep extreme** | 42, 54–55, 124, 139, 206–207, 220, 266, 297–298, 303, 352–360, 363–366 | `stop = L_s - b*ATR`; if that lands near a round price (x.00, x.50, x.25), push it c ticks beyond (pp. 355, 363–365). Entry filter: `(E - stop) <= e_max*ATR`. Management: never widen; move to breakeven plus costs once MFE ≥ m1·R; trail under each new confirmed higher pivot low minus b; tighten on an upthrust or effort-without-result at highs; leave room for the normal counter-move (p. 360). | b ∈ [0.1, 1.0]·ATR; c ∈ [1, 3] ticks; e_max ∈ [1, 4]·ATR; m1 ∈ [0.75, 1.5]·R | If stop and target are both touched in the same bar after entry, assume the stop was hit first. Model at least 1 tick of slippage on stops, more on thin ETFs (p. 358). Round-number logic needs unadjusted prices (see the data note in 1.4). |
| **Target at opposing liquidity** | 23, 62, 73–74, 123–125, 128–131, 169–170, 202–203, 216–217, 225, 237, 266, 284, 297, 333–338, 352–353, 360 | `T` is the nearest level above E among: the pullback origin H0, the range high, the session high, the prior-day high, and optionally a P&F count target. Optionally take a first target at 50% of the down-leg after a climax (pp. 62, 225). Require `(T - E)/(E - stop) >= RR_min`. Optionally exit part of the position at the first target (p. 360). | RR_min ∈ [1.5, 4] (the book says ≥ 3); P&F box ∈ [0.25, 1]·ATR, reversal ∈ {1, 3}; target haircut ∈ [0.7, 1.0] | "Liquidity" (stops resting beyond highs) is a modern framing. The book's version is operators running stops above old tops and at round figures (pp. 73–74, 128–131, 363–365). The book calls P&F counts rough, so weight them low. |
| **Failed sweep / breakdown** | 53–54, 65, 78, 136, 173–174, 211, 362 | A close below `L_s - ε*ATR` with `RV >= v_brk` means supply was not absorbed: invalidate, or optionally stop and reverse. A rising RV slope on the approach to SL* predicts that the low will not hold (p. 65), so record it as a feature. | ε ∈ [0, 0.3]·ATR; v_brk ∈ [1.5, 3]; slope window ∈ [5, 20] | The book reads low-volume breaks both ways (p. 211 bearish, p. 212 not), so use this as a feature, not a hard rule. |
| **Relative strength vs the composite** | 86–101, 208–219, 264, 269–271, 296–298, 306 | During the sweep window, bullish if the ETF holds a higher low while the composite makes a lower low. `rs = ret_ETF - beta*ret_comp` over the pullback. Block shake-out longs while the composite is in acute weakness (p. 218). | beta window ∈ [1, 20] sessions; rs threshold ∈ [0, 0.5]·ATR | Bars must be minute-aligned across symbols. TLT, GLD, USO and VXX need their own comparators, not SPY. |
| **Neutral gate / selectivity** | 265, 277, 283–285, 330, 333–334, 366–368, 382, 390–393 | No new trades when the trend is `NEUTRAL`, when scales conflict, or when `range/ATR_long < d_dull` with median RV < v_dull. Cap trades per session, and stop for the session after k consecutive losses. | trades per session ∈ {1, 2, 3}; k ∈ {2, 3}; d_dull ∈ [0.3, 0.8]; v_dull ∈ [0.4, 0.7] | Apply the caps in evaluation, not in candidate generation, so the meta-labeler still has enough samples. |
| **Time stop (springboard trades should move at once)** | 361, 380, 384 | Exit at market if MFE < x·ATR within N bars. | x ∈ [0.5, 1.5]; N ∈ [5, 30] | Set it equal to the triple-barrier vertical barrier so labels and execution agree. |

**How the pieces fit:**
1. A long candidate requires the trend, pullback and spring rows together: HTF trend `UP`, a qualified pullback, then a sweep and reclaim.
2. The primary rules are tuned for recall.
3. The test, effort-vs-result, climactic-volume, close-location, sign-of-strength, failed-sweep and relative-strength rows, plus P1–P8 below, become meta-labeler features, tuned for precision.
4. Entry is at E (entry A) or after a passed test (entry B).
5. The stop and target rows set the barriers, and the neutral gate and time stop govern execution.
6. Every rule is implemented symmetrically for shorts, and long and short results are checked separately for bias. The book recommends viewing charts upside down for the same reason (p. 293).

### 1.4 Order-flow proxies

OHLCV bars carry no bid/ask, aggressor side or trade sizes. Everything below is a deterministic function of `O H L C V` and is labelled **PROXY**. These proxies are correlated with each other and with returns, so they should not be counted as independent evidence.

- **P1. Close location value (PROXY: where the bar's auction settled).**
  - `CLV01 = (C - L)/R` on [0, 1].
  - `CLV = 2*CLV01 - 1 = ((C-L) - (H-C))/R` on [-1, 1].
  - Set it to NaN when `R < 2*tick`.
  - Basis: the close as the net result of the battle (pp. 19, 54, 56–59).

- **P2. Bar-delta estimate (PROXY: signed aggressor volume).**
  - CLV split: `D_clv = V*(2*CLV01 - 1)`.
  - Body split: `D_body = V*(C - O)/R`.
  - Tick rule on closes: `D_tick = V*sgn(C_t - C_{t-1})`; when the sign is 0, carry the previous sign forward.
  - Bulk volume classification (Easley, López de Prado & O'Hara): `D_bvc = V*(2*Phi(dC/sigma_dC) - 1)`, where `sigma_dC` is the rolling standard deviation of 1-minute close changes.
  - Aggregates: `CVD_N = sum(D)` and `pressure_N = sum(D)/sum(V)` on [-1, 1].
  - Caveat: none of these observes the true aggressor side, and on bars only a few ticks wide they differ mostly by noise.

- **P3. Relative volume vs the same minute of day (PROXY: effort / intensity).**
  - `RV_t = V_t / median(V at the same minute-of-day over the prior D sessions)`, using prior sessions only and a floor on the denominator.
  - Also `RV_local = V_t / median(V[t-n..t-1])`. Use `log(RV)` for ML features.
  - Half-days and auction minutes get separate baselines.
  - The book normalizes short sessions (pp. 19, 59, 225) but says to keep raw and normalized side by side (pp. 288–289), so keep both RV forms.
  - Ranges: D ∈ [10, 30], n ∈ [20, 60].

- **P4. Effort/result ratio (PROXY: progress in proportion to volume).**
  - Per bar: `RE_t = (abs(C_t - C_{t-1})/ATR)/RV_t`, with `ER = 1/RE`.
  - Range form: `(R_t/ATR)/RV_t`.
  - Per wave: `RE_w = (abs(dP_w)/ATR)/sum(RV_w)`.
  - At a level: `(extension beyond SL* or SH*)/ATR / RV`.
  - Use rolling percentile ranks per instrument. Basis: pp. 67–70, 175–176, 223–224, 335.

- **P5. Wick/body ratios (PROXY: rejection and absorption).**
  - `B = abs(C-O)`, `UW = H - max(O,C)`, `LW = min(O,C) - L`, each also divided by R.
  - Sweep-specific: `pen_share = (SL* - L_s)/R_s` and `close_back = (C_s - SL*)/ATR`.
  - Volume-weighted rejection: `RV*LW/R`.
  - Caveat: how much volume traded in the wick is unknown, and a "long wick" on a narrow bar may be a single tick.

- **P6. Sweep-bar vs reclaim-bar volume ratio (PROXY: volume rising on the recovery).**
  - `VR_rs = meanRV[r..r+j]/RV_s`; when r = s, use the bars from r+1 to r+j.
  - `VR_ts = meanRV(test leg)/RV_s`.
  - `VR_sp = RV_s/meanRV(pullback)`.
  - Bullish pattern: `VR_rs >= 1` with `VR_ts < 1` (pp. 124, 220, 291). Range: j ∈ [1, 5].

- **P7. Wave volume rate and up/down ratio (PROXY: the Wave Chart, pp. 312–320, 331–338).**
  - Per zigzag wave: extent `dP/ATR`, duration `T`, speed `dP/T`.
  - `RR_w = sum(V_w)/sum(baseline V over the same minutes)`.
  - `UD = meanRR(up waves)/meanRR(down waves)` over the last N waves, with N ∈ [4, 6].
  - A bearish shift is `UD < 1` near the highs; the bullish mirror is `UD > 1` near the lows.

- **P8. Activity (PROXY).**
  - The book's "activity" is the rate of transactions, and it infers order quality from lot sizes (pp. 324–325). Our files have no trade counts or VWAP.
  - Use only price-based activity: the share of bars in a wave with `R >= 2 ticks`, or `sum(R)/T`.
  - Make no lot-size or quality inferences.

**Data note** (from inspecting `/Users/andrew/main/trading/data/etfs/*_clean_1min.csv`; columns `datetime,Open,High,Low,Close,Volume,source`):

1. **Volume source break.** The `source` column changes from `pitrading` to `iex` on 2022-03-07 (seen in SPY, QQQ and CPER; VXX is IEX-only). SPY's mean 1-minute volume drops about 78x, from roughly 255k to 3.3k. Volume after that date is IEX's venue volume only, not consolidated volume.
   - RV baselines must never span the switch: restart them on 2022-03-07, or normalize per source.
   - Volume-based edges must be validated on each regime separately.
   - IEX-only RV is a proxy of a proxy.
2. **Missing minutes on thin ETFs.** CPER has about 38k IEX bars, roughly 9% of the RTH minutes in that span. Either reindex to the full RTH grid (V = 0, flat bar at the prior close) or exclude those symbols from the volume rules.
3. **Prices appear back-adjusted.** SPY's first bar on 2002-12-30 opens at 61.25, while SPY traded near 88 at the time. Tick-size and round-number logic (the stop row) is only valid on unadjusted prices or on recent data.
4. **Timestamps.** They run 09:30–15:59, which fits bar-start labels for RTH only. The minute-of-day index is 0–389, and "next bar open" means the `Open` of label t+1.

### 1.4b Implementation scope: what §5 actually builds from 1.3

The 1.3 table is the design space. To keep the search budget (and the multiple-testing penalty) bounded, the notebook implements a subset. Each row is marked **implemented**, **simplified**, **feature only** (used by the ML meta-labeler, not as a rule), or **not implemented**:

| 1.3 row | Status in this notebook | How / why |
|---|---|---|
| Trend (HTF) | **simplified** | Last *completed* HTF bar closes above (below) its EMA and the EMA is rising (falling). HTF ∈ {5, 15, 30, 60} min, EMA ∈ {10, 20, 50}. Optional prior-day daily-trend agreement. A pivot-structure trend and the cross-ETF composite filter are not implemented. |
| Pullback, volume drying up | **implemented** | Causal zigzag waves (`z`·ATR). Reaction depth `rr ∈ [rr_min, rr_max]`, impulse ≥ `imp_min` ATR. Reaction/impulse relvol ratio ≤ `wv_max` in the "dry" recipe. |
| Spring / shake-out | **implemented** | Rolling-window low or confirmed fractal pivot. Sweep depth `[sd_min, sd_max]`·ATR. Reclaim close within `m` bars (e = 0). The volume tag is the `confirm` recipe (climax / dry / either). |
| Upthrust | **implemented (mirror)** | Short entries in a down-trend. Not used as a long-exit flag. |
| Test of the spring (entry B) | **not implemented** | Entry A only (next open after the reclaim). On 1-minute bars the test often never comes, and a second entry type would double the search. |
| Effort vs result | **simplified + feature** | Max per-bar relvol/range ratio over the sweep (`er_min` in the climax recipe). The cluster/percentile form is not implemented. |
| Climactic vs drying volume | **implemented + feature** | `confirm` ∈ {none, climax, dry, recovery, either}. The breakout-volume classification is not implemented. |
| Close location / wick | **implemented + feature** | Reclaim-bar CLV ≥ `clv_min`. Wick ratios are ML features. |
| Sign of strength after reclaim | **not implemented** | It would delay entry and reduce reward/risk; left for future work. |
| Stop beyond sweep extreme | **implemented** | Extreme − `stop_buf`·ATR, risk bounded to `[risk_min, risk_max]` ATR, plus a cost-to-risk filter. Round-number offsets are not implemented (prices are back-adjusted), and neither are trailing or breakeven (fixed brackets keep triple-barrier labels aligned with execution). |
| Target at opposing liquidity | **implemented** | `R_mult`·R, or the impulse top if it is ≥ `min_R` away. P&F counts are not implemented. |
| Failed sweep | **implemented** | The setup is cancelled if price runs past `sd_max` ATR before reclaiming. |
| Relative strength vs composite | **not implemented** | It needs several ETFs' minute bars in memory at once, which the one-symbol-at-a-time memory rule forbids. |
| Neutral gate / selectivity | **simplified** | No trades when the HTF trend is neutral. `max_trades_day` per ETF. The portfolio-level −2R daily loss limit replaces the "k consecutive losses" stop. |
| Time stop | **implemented** | `time_stop` bars, identical to the triple-barrier vertical barrier. |
| Proxies P1–P8 | **partly** | P1 CLV, P2 CLV-split delta, P3 relvol (time-of-day and local), P4 effort/result (range form), P5 wicks, P6 reclaim/sweep volume, P7 reaction/impulse volume. Not built: P2 tick-rule/BVC variants, P7 up/down wave rate, P8. |

### 1.5 MIT sources

| Title | Authors | Year | Venue | Link | Technique used in this project |
|---|---|---|---|---|---|
| Foundations of Technical Analysis: Computational Algorithms, Statistical Inference, and Empirical Implementation | Andrew W. Lo, Harry Mamaysky, Jiang Wang | 2000 | Journal of Finance 55(4):1705-1765, Aug 2000 (also NBER Working Paper 7613, Mar 2000) | [DOI](https://doi.org/10.1111/0022-1082.00265) | Finds chart patterns automatically. Prices are smoothed with kernel regression, local extrema are found, and patterns (head-and-shoulders, double tops and bottoms, and others) are defined as rules on those extrema. Tests whether return distributions after a pattern differ from unconditional ones, using goodness-of-fit and Kolmogorov-Smirnov tests. This is the template for coding and testing Wyckoff patterns. Applying it to Wyckoff patterns is this project's extension. |
| Technical analysis : neural network based pattern recognition of technical trading indicators, statistical evaluation of their predictive value and a historical overview of the field | Jasmina Hasanhodzic | 2004 | S.M. thesis, MIT Dept. of Electrical Engineering and Computer Science (advisor Andrew W. Lo) | [hdl:1721.1/28725](http://hdl.handle.net/1721.1/28725) | Re-implements the Lo-Mamaysky-Wang pattern detector as a neural network. The patterns still add predictive value across smoothing levels and pattern definitions. Serves as a robustness check for ML-based pattern detection. |
| Investments unwrapped: demystifying and automating technical analysis and hedge-fund strategies | Jasmina Hasanhodzic | 2007 | Ph.D. thesis, MIT Dept. of Electrical Engineering and Computer Science (advisor Andrew W. Lo) | [hdl:1721.1/38677](http://hdl.handle.net/1721.1/38677) | Detects chart patterns with a neural network and tests whether they add predictive value on stock and FX data. Also builds hedge-fund replication portfolios from linear factor models. |
| Data-Snooping Biases in Tests of Financial Asset Pricing Models | Andrew W. Lo, A. Craig MacKinlay | 1990 | Review of Financial Studies 3(3):431-467 (earlier: MIT Sloan WP 3020-89, rev. Nov 1989; NBER WP 3001, Jun 1989) | [hdl:1721.1/2249](http://hdl.handle.net/1721.1/2249) · [DOI](https://doi.org/10.1093/rfs/3.3.431) | Data snooping: if test portfolios are formed on a characteristic picked from the data, 5% tests can reject a true null with near certainty. The bias is measured analytically and by Monte Carlo. This is the basic reason not to choose signals or patterns on a sample and then test them on the same sample. |
| The Statistics of Sharpe Ratios | Andrew W. Lo | 2002 | Financial Analysts Journal 58(4):36-52 | [DOI](https://doi.org/10.2469/faj.v58.n4.2453) | Standard errors of the Sharpe ratio for IID, serially correlated and time-aggregated returns. Shows that sqrt(q) annualization is wrong when returns are autocorrelated. In the paper's hedge-fund sample, serial correlation overstated annual Sharpe by up to 65%. Used here for confidence intervals on backtest Sharpe and for correct annualization. |
| The Adaptive Markets Hypothesis: Market Efficiency from an Evolutionary Perspective | Andrew W. Lo | 2004 | Journal of Portfolio Management 30(5):15-29 (30th Anniversary Issue) | [DOI](https://doi.org/10.3905/jpm.2004.442611) | Strategy profits change as markets evolve, so edges appear and decay. Supports evaluating signals regime by regime and on rolling windows. The test design is this project's own inference; the paper prescribes no method. |
| What Happened to the Quants in August 2007? Evidence from Factors and Transactions Data | Amir E. Khandani, Andrew W. Lo | 2011 | Journal of Financial Markets 14(1):1-46 | [hdl:1721.1/108275](https://hdl.handle.net/1721.1/108275) · [DOI](https://doi.org/10.1016/j.finmar.2010.07.005) | A simulated high-frequency contrarian (market-making) strategy lost heavily in the week of 6 Aug 2007 but was profitable before and after. A case study in crowding, regime and tail risk for intraday mean-reversion signals. |
| When Do Stop-Loss Rules Stop Losses? | Kathryn M. Kaminski, Andrew W. Lo | 2014 | Journal of Financial Markets 18 (Mar 2014):234-254 (online Jul 2013) | [DSpace](https://dspace.mit.edu/handle/1721.1/114876) · [DOI](https://doi.org/10.1016/j.finmar.2013.07.001) | Values a stop-loss rule by comparing expected return and volatility with and without it. Under a random walk, stops lower expected return; with momentum they can add value. Used to evaluate stop and target exits. The paper studies portfolio-level drawdown stops on daily-to-monthly data, so the link to triple-barrier labels is this project's extension. |
| A computational view of market efficiency | Jasmina Hasanhodzic, Andrew W. Lo, Emanuele Viola | 2011 | Quantitative Finance 11(7):1043-1050; preprint arXiv:0908.4580 (2009) | [hdl:1721.1/75357](http://hdl.handle.net/1721.1/75357) · [DOI](https://doi.org/10.1080/14697688.2010.541487) | Defines efficiency relative to computing resources. "Memory-m" strategies see only the last m returns, and more memory can earn profits that shorter memory cannot. Gives a theoretical reason to treat lookback and feature-window length as tuned parameters. It is not a procedure for choosing them. |
| Developing high-frequency equities trading models | Leandro Rafael Infantino, Savion Itzhaki | 2010 | M.B.A. thesis, MIT Sloan School of Management (advisor Roy E. Welsch) | [hdl:1721.1/59122](http://hdl.handle.net/1721.1/59122) | Uses short-term PCA of US equity returns to predict cumulative returns over holding periods of about 1 second to 5 minutes. Tested on 50 stocks drawn at random from the most liquid S&P 500 names. Relevant to building and backtesting signals at scalping horizons. |
| A time and frequency domain analysis of contrarian trading strategies | Shomesh Ernesto Chaudhuri | 2014 | S.M. thesis, MIT Dept. of Electrical Engineering and Computer Science (advisor Andrew W. Lo) | [hdl:1721.1/91093](http://hdl.handle.net/1721.1/91093) | Applies time- and frequency-domain analysis to a contrarian (market-making) strategy over 1964-2013. In recent decades, shorter intraday horizons were more profitable and less sensitive to turbulence. Turbulence in particular frequency bands moves contrarian returns. Shows how short-term mean reversion depends on horizon and regime. |
| Application of machine learning: automated trading informed by event driven data | Jason W. Leung | 2016 | M.Eng. thesis, MIT Dept. of Electrical Engineering and Computer Science (advisor Jacob K. White) | [hdl:1721.1/105982](http://hdl.handle.net/1721.1/105982) | ML models trained on technical-indicator and sentiment features act as an automated trader on the S&P 500. A worked example of building signals with ML. |
| Trading Volume: Definitions, Data Analysis, and Implications of Portfolio Theory | Andrew W. Lo, Jiang Wang | 2000 | Review of Financial Studies 13(2):257-300 | [DOI](https://doi.org/10.1093/rfs/13.2.257) | Argues that share turnover is the natural measure of volume, and finds a roughly two-factor structure in turnover. Used to normalize the volume inputs of Wyckoff volume/price signals across securities. |
| 18.S096 Topics in Mathematics with Applications in Finance (Fall 2013) | Peter Kempthorne, Choongbum Lee, Vasily Strela, Jake Xia | 2013 | MIT OpenCourseWare, Dept. of Mathematics (first version of 18.642) | [OCW](https://ocw.mit.edu/courses/18-s096-topics-in-mathematics-with-applications-in-finance-fall-2013/) | Background material: regression, time series (3 lectures), volatility modeling, factor models and portfolio theory. Used only as general statistical background; the course does not cover intraday modeling or backtest methodology. |

MIT affiliation basis: every row has at least one MIT author (Lo and Wang at MIT Sloan) or is an MIT thesis or MIT course, and all are on DSpace@MIT, MIT OCW or Lo's MIT pages. Some co-authors were elsewhere: Mamaysky at Yale, MacKinlay at Wharton, and Viola at Northeastern.

### 1.6 Other published work (NOT from MIT)

| Title | Authors | Year | Venue | Link | Technique used in this project |
|---|---|---|---|---|---|
| Advances in Financial Machine Learning | Marcos López de Prado | 2018 | John Wiley & Sons, Hoboken NJ (ISBN 978-1-119-48208-6; eISBN 978-1-119-48211-6) | [Wiley](https://www.wiley-vch.de/en/areas-interest/finance-economics-law/advances-in-financial-machine-learning-978-1-119-48208-6) | Triple-barrier labeling and meta-labeling (Ch. 3), sample uniqueness and weights (Ch. 4), fractional differentiation (Ch. 5), purged and embargoed k-fold CV (Ch. 7), dangers of backtesting (Ch. 11), combinatorial purged CV (Ch. 12). |
| The 10 Reasons Most Machine Learning Funds Fail | Marcos López de Prado | 2018 | The Journal of Portfolio Management 44(6):120-133 | [DOI](https://doi.org/10.3905/jpm.2018.44.6.120) | Peer-reviewed summary of the pitfalls in Advances in Financial Machine Learning: fixed-horizon vs triple-barrier labels, meta-labeling, weights for non-IID samples, CV leakage (fixed by purging and embargo), combinatorial purged CV, and backtest overfitting. |
| Meta-Labeling: Theory and Framework | Jacques Francois Joubert | 2022 | The Journal of Financial Data Science 4(3):31-44 | [DOI](https://doi.org/10.3905/jfds.2022.1.098) | A formal framework for meta-labeling. A secondary model filters false positives from a primary signal and sizes the position, so trade side and trade size are decided separately. It formalizes and extends López de Prado's idea. |
| The Deflated Sharpe Ratio: Correcting for Selection Bias, Backtest Overfitting, and Non-Normality | David H. Bailey; Marcos López de Prado | 2014 | The Journal of Portfolio Management 40(5):94-107 | [DOI](https://doi.org/10.3905/jpm.2014.40.5.094) | Deflated Sharpe Ratio (DSR): the probabilistic Sharpe ratio measured against the expected maximum Sharpe over N trials. It uses a Sharpe standard error adjusted for skew and kurtosis, which comes from Bailey & López de Prado (2012). |
| The Probability of Backtest Overfitting | David H. Bailey; Jonathan M. Borwein; Marcos López de Prado; Qiji Jim Zhu | 2017 | The Journal of Computational Finance 20(4):39-69 (first published online 19 Sep 2016; SSRN 2326253) | [DOI](https://doi.org/10.21314/JCF.2016.322) | Probability of Backtest Overfitting (PBO), estimated by combinatorially symmetric cross-validation (CSCV) on the performance matrix of every strategy configuration tried. |
| Pseudo-Mathematics and Financial Charlatanism: The Effects of Backtest Overfitting on Out-of-Sample Performance | David H. Bailey; Jonathan M. Borwein; Marcos López de Prado; Qiji Jim Zhu | 2014 | Notices of the American Mathematical Society 61(5):458-471 | [DOI](https://doi.org/10.1090/noti1105) | Minimum backtest length (MinBTL) as a function of the number of trials, and the expected maximum Sharpe when the true Sharpe is zero. Explains why heavily optimized backtests fail out of sample. |
| Backtesting | Campbell R. Harvey; Yan Liu | 2015 | The Journal of Portfolio Management 42(1):13-28 | [DOI](https://doi.org/10.3905/jpm.2015.42.1.013) | "Haircut" Sharpe ratios under multiple testing: Bonferroni, Holm and BHY adjustments to p-values and Sharpe ratios, plus a profit hurdle. |
| … and the Cross-Section of Expected Returns | Campbell R. Harvey; Yan Liu; Heqing Zhu | 2016 | The Review of Financial Studies 29(1):5-68 (published online 9 Oct 2015; NBER WP 20592, Oct 2014) | [DOI](https://doi.org/10.1093/rfs/hhv059) | Multiple-testing framework: family-wise error rate (Bonferroni, Holm) and false discovery rate (BHY). Sets a t > 3.0 hurdle for new factors or signals. |
| A Reality Check for Data Snooping | Halbert White | 2000 | Econometrica 68(5):1097-1126 | [DOI](https://doi.org/10.1111/1468-0262.00152) | White's Reality Check: a stationary-bootstrap test of whether the best of many rules beats a benchmark, with data snooping taken into account. |
| A Test for Superior Predictive Ability | Peter Reinhard Hansen | 2005 | Journal of Business & Economic Statistics 23(4):365-380 | [DOI](https://doi.org/10.1198/073500105000000063) | Hansen's SPA test: a studentized, recentered refinement of the Reality Check. It is less conservative and less affected by poor or irrelevant alternatives. |
| The Stationary Bootstrap | Dimitris N. Politis; Joseph P. Romano | 1994 | Journal of the American Statistical Association 89(428):1303-1313 | [DOI](https://doi.org/10.1080/01621459.1994.10476870) | Block bootstrap for dependent return series, with block lengths drawn from a geometric distribution. Here it drives the Reality Check, the SPA test and Sharpe confidence intervals; those uses come from later work, not this paper. |
| A Simple Way to Estimate Bid-Ask Spreads from Daily High and Low Prices | Shane A. Corwin; Paul Schultz | 2012 | The Journal of Finance 67(2):719-760 | [DOI](https://doi.org/10.1111/j.1540-6261.2012.01729.x) | High-low spread estimator: comparing one-day and two-day ranges isolates the spread from volatility. Used for transaction costs in backtests. The paper validated it on daily bars, so using it on intraday OHLC bars is this project's extension. |
| A Simple Implicit Measure of the Effective Bid-Ask Spread in an Efficient Market | Richard Roll | 1984 | The Journal of Finance 39(4):1127-1139 | [DOI](https://doi.org/10.1111/j.1540-6261.1984.tb03897.x) | Roll spread, `2*sqrt(-Cov(dp[t], dp[t-1]))`, used for cost estimates and as a microstructure feature. It is undefined when the covariance is non-negative, as in trending markets. |
| XGBoost: A Scalable Tree Boosting System | Tianqi Chen; Carlos Guestrin | 2016 | Proceedings of the 22nd ACM SIGKDD International Conference on Knowledge Discovery and Data Mining (KDD '16):785-794 | [DOI](https://doi.org/10.1145/2939672.2939785) | Gradient-boosted tree classifier, a candidate for the secondary (meta-label) model. |
| Random Forests | Leo Breiman | 2001 | Machine Learning 45(1):5-32 | [DOI](https://doi.org/10.1023/A:1010933404324) | Random forest (bagging plus random feature selection at each split, with out-of-bag error), used as a baseline or meta-label classifier. Provides out-of-bag permutation feature importance. Impurity (Gini) importance comes from later Breiman software and CART, not from this paper. |
| A Unified Approach to Interpreting Model Predictions | Scott M. Lundberg; Su-In Lee | 2017 | Advances in Neural Information Processing Systems 30 (NIPS 2017), pp. 4765-4774; arXiv:1705.07874 | [NeurIPS](https://proceedings.neurips.cc/paper_files/paper/2017/hash/8a20a8621978632d76c43dfd28b67767-Abstract.html) | SHAP (Shapley additive explanations), used to attribute predictions to the features of the Wyckoff and meta-label models. |
| Optuna: A Next-generation Hyperparameter Optimization Framework | Takuya Akiba; Shotaro Sano; Toshihiko Yanase; Takeru Ohta; Masanori Koyama | 2019 | Proceedings of the 25th ACM SIGKDD International Conference on Knowledge Discovery & Data Mining (KDD '19):2623-2631 | [DOI](https://doi.org/10.1145/3292500.3330701) | Hyperparameter search with the TPE sampler and pruning. The recorded trial count feeds the DSR and PBO multiple-testing corrections; that is this project's use, and the paper does not discuss DSR or PBO. |
| Empirical Asset Pricing via Machine Learning | Shihao Gu; Bryan Kelly; Dacheng Xiu | 2020 | The Review of Financial Studies 33(5):2223-2273 | [DOI](https://doi.org/10.1093/rfs/hhaa009) | Benchmark comparison of ML methods for return prediction: penalized linear models, trees, boosting and neural nets. Uses out-of-sample R² with a non-demeaned denominator and time-ordered train/validation/test splits. |

Note on White (2000): the paper was written at UC San Diego. White earned his Ph.D. at MIT in 1976, so he is an MIT alumnus, but this is not MIT work. "Massachusetts" also appears in some NBER versions of papers in this table; that is NBER's Cambridge address, not an author affiliation.

**How the techniques map to notebook sections.**
- **§8 labels:** triple-barrier labels follow López de Prado (2018, Ch. 3; 2018 JPM). Kaminski & Lo (2014) is the reason the stop is treated as a design choice to test, not one assumed to help: stop buffer and time stop are search parameters. Their with/without-stop comparison itself is not implemented.
- **§8 meta-labeling:** follows López de Prado (2018) and Joubert (2022), with XGBoost (Chen & Guestrin 2016) or Random Forests (Breiman 2001) as the secondary classifier.
- **§8 cross-validation:** purged and embargoed K-fold CV by date blocks follows López de Prado (2018, Ch. 7). Combinatorial purged CV (Ch. 12) is not used; CSCV in §10.5 plays the combinatorial role for the search.
- **§8 interpretation:** SHAP (Lundberg & Lee 2017) explains the features, with Breiman's permutation importance as a cross-check.
- **§9/§10 Sharpe uncertainty:** §10.4b re-annualizes OOS Sharpe with Lo's (2002) autocorrelation-corrected factor. The DSR and PSR use the skew- and kurtosis-adjusted standard error.
- **§10 overfitting:** the Deflated Sharpe Ratio (Bailey & López de Prado 2014, §10.4), minimum backtest length (Bailey et al. 2014, §10.4) and PBO by CSCV (Bailey et al. 2017, §10.5), all using the trial counts and trial-by-day return matrices logged in §7.
- **§10 data snooping:** Lo & MacKinlay (1990) is the basic warning. §10.4b implements White's Reality Check (2000) and Hansen's SPA test (2005). The Harvey & Liu (2015) and Harvey, Liu & Zhu (2016) haircuts and t > 3 hurdle are context only; they are not computed.
- **§10 Monte Carlo:** the stationary bootstrap (Politis & Romano 1994) resamples the dependent daily return series in §10.4b and §10.7.
- **§3 cost model:** spreads are *assumed* by liquidity tier. A Corwin & Schultz (2012) estimate is printed as a sanity check only. Roll (1984) is listed for context and not computed, because its covariance is often non-negative on trending 1-minute data.
- **Wider design choices:** coding a chart pattern as explicit rules on causal extrema, then testing what follows, follows Lo, Mamaysky & Wang (2000) and Hasanhodzic (2004, 2007). Lo & Wang (2000) motivate normalizing volume; we use relative volume against same-minute history, because turnover needs shares outstanding and the data has none. Regime-by-regime evaluation (§11, §12) follows Lo (2004), Khandani & Lo (2011) and Chaudhuri (2014).

> **Access note:** Web search and page fetching worked during this research. Each citation was found by web search, a DSpace@MIT handle fetch, or the Crossref/arXiv APIs, and then checked one by one by a separate adversarial checker. Some publisher landing pages blocked automated fetches with 403 errors or login redirects: Wiley, Taylor & Francis, ACM DL, PM-Research, Springer and SSRN. A few DSpace PDF downloads returned HTTP 429. In those cases the metadata was confirmed through Crossref records, DOI redirect targets, RePEc/NBER listings, or copies hosted by the authors or their institutions.

Candidates dropped because they could not be verified: none.

### 1.7 Sources added for §12 (macro events & uncertainty)

| Source | Provider | Link | Use |
|---|---|---|---|
| Release calendars: CPI, Employment Situation, Personal Income & Outlays, GDP, Advance Retail Sales, PPI | FRED / St. Louis Fed (`fred/release/dates`) | <https://fred.stlouisfed.org/docs/api/fred/release_dates.html> | event-day tagging by release date |
| `USEPUINDXD`, Economic Policy Uncertainty (daily) — Baker, Bloom & Davis (2016), *QJE* 131(4) | FRED | <https://fred.stlouisfed.org/series/USEPUINDXD> | prior-day uncertainty terciles |
| `EMVOVERALLEMV`, Equity Market Volatility tracker (monthly) — Baker, Bloom, Davis & Kost (2019), NBER w25720 | FRED (ALFRED vintages) | <https://fred.stlouisfed.org/series/EMVOVERALLEMV> | point-in-time terciles |
| Geopolitical Risk daily index (GPRD) — Caldara & Iacoviello (2022), *AER* 112(4) | matteoiacoviello.com (**non-FRED**) | <https://www.matteoiacoviello.com/gpr.htm> | prior-day GPR terciles |
| FOMC statement dates, scheduled vs intermeeting | Federal Reserve FOMC calendars (**non-FRED**) | <https://www.federalreserve.gov/monetarypolicy/fomccalendars.htm> | FOMC event days |
| FOMC statement timing (2:00 p.m. from March 2013) | Federal Reserve press release, 13 Mar 2013 | <https://www.federalreserve.gov/newsevents/pressreleases/monetary20130313a.htm> | intraday ±30 min window |

Local copies of every openly available source are in `artifacts/sources/` (see `artifacts/sources/MANIFEST.md`). Paywalled works are listed there by link only.

## 2 Config

Everything that changes results lives here: paths, universe rules, session times, the cost model, risk rules, the split schemes, and the search budgets. Seeds are fixed and recorded with the package versions so a rerun reproduces the same trial log.

**Memory design (4 GB budget).** The notebook never holds more than one symbol's 1-minute bars. Each symbol is cleaned once into a parquet file (one row group per year, so IS windows read only the years they need), `float32`/`int16`/`int8` throughout. Search trials loop symbol by symbol and keep only small outputs (trade lists and daily R-series). `mem()` prints current and peak RSS after every stage.

In [1]:
import os, gc, re, json, time, math, glob, platform, resource, warnings, itertools, hashlib, sys
from pathlib import Path
from dataclasses import dataclass, asdict

import numpy as np
import pandas as pd
import numba as nb
import psutil
import pyarrow as pa
import pyarrow.parquet as pq
import scipy
from scipy import stats
import plotly
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.io as pio

warnings.filterwarnings('ignore', category=FutureWarning)
pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 40)
pio.renderers.default = 'notebook_connected'
pio.templates.default = 'plotly_white'

# ---------------------------------------------------------------- reproducibility
SEED = 20261025
np.random.seed(SEED)
RNG = np.random.default_rng(SEED)

# ---------------------------------------------------------------- paths
PROJECT = Path.cwd()
DATA_DIR = Path('/Users/andrew/main/trading/data/etfs')
FILE_PATTERN = '{sym}_clean_1min.csv'          # columns: datetime,Open,High,Low,Close,Volume,source
CACHE = PROJECT / 'cache'
CACHE_VERSION = 'v6'             # bump whenever cleaning/feature/signal/engine code changes results -> fresh derived caches
DIRS = {k: CACHE / CACHE_VERSION / k for k in ['bars', 'feats', 'daily', 'trials', 'trades', 'ml', 'results']}
DIRS['external'] = CACHE / 'external'    # downloaded data (yfinance, FRED, GPR): version-independent
for d in DIRS.values():
    d.mkdir(parents=True, exist_ok=True)

# Set WYCKOFF_FAST=1 for a quick smoke run (3 symbols, small budgets). Default is the full study.
FAST = os.environ.get('WYCKOFF_FAST', '0') == '1'
REBUILD_CACHE = os.environ.get('WYCKOFF_REBUILD', '0') == '1'

# ---------------------------------------------------------------- universe
# VIX-linked products are never traded. VIX itself is used only for regime labels.
VIX_LINKED = re.compile(r'^(VXX|VXXB|VIXY|VIXM|VXZ|UVXY|UVIX|SVXY|SVIX|TVIX|XIV|ZIV|VIIX|VMAX|VMIN|VIX\w*)$')

# ---------------------------------------------------------------- session (minutes after midnight, America/New_York)
OPEN_MIN, LAST_BAR_MIN = 9 * 60 + 30, 15 * 60 + 59      # bars labelled 09:30 .. 15:59
FLAT_MIN = 15 * 60 + 55                                  # flat by 15:55 -> exit at close of the 15:54 bar
BARS_PER_DAY = LAST_BAR_MIN - OPEN_MIN + 1               # 390

# ---------------------------------------------------------------- execution & costs
RISK_PER_TRADE = 0.005           # fixed fractional risk, 0.5% of equity
DAILY_LOSS_LIMIT_R = 2.0         # portfolio-level: no new entries once realized day P&L <= -2R
MAX_POS_LEVERAGE = 4.0           # notional cap per position; caps risk when stops are tiny
MAX_GROSS_LEVERAGE = 4.0         # cap on the SUM of open positions' notional (Reg-T intraday buying power)
SLIP_CENTS = {1: 1.0, 2: 1.0, 3: 2.0}          # per share per side, by liquidity tier (tier 3 = illiquid)
HALF_SPREAD_CENTS = {1: 0.5, 2: 1.0, 3: 2.5}   # half of an assumed typical quoted spread
TIER_DOLLAR_VOL = (1.0e9, 1.0e8)               # median daily $ volume (consolidated era) >= 1e9 -> tier 1, >= 1e8 -> tier 2
COST_MULTS = (1.0, 2.0)          # report every result at 1x and 2x costs
MIN_COVERAGE = 0.80              # trade a symbol-day only if the PRIOR 5 sessions had >= 80% of minutes with real prints

# ---------------------------------------------------------------- splits (IS starts at the global data start)
SPLITS = ['IS1y', 'IS2y', 'IS3y', 'IS4y', 'IS5y', 'HALF']
WF_FIRST_IS_YEARS, WF_STEP_YEARS = 5, 3          # anchored walk-forward: IS grows, next 3y is OOS
N_TRIALS = {'split': 250, 'wf': 100}
MIN_TRADES_PER_IS_YEAR = 25                      # trials below this are logged as failures
PRIMARY_SPLIT = 'HALF'                           # split used for the detailed diagnostics

# ---------------------------------------------------------------- ML
ML_CV_FOLDS, ML_EMBARGO_DAYS = 5, 5
N_PLACEBO = 500
N_LABEL_SHUFFLES = 20
N_BOOT = 2000

if FAST:
    N_TRIALS = {'split': 12, 'wf': 6}
    N_PLACEBO, N_LABEL_SHUFFLES, N_BOOT = 60, 4, 300

# ---------------------------------------------------------------- memory instrumentation
_PROC = psutil.Process()
_MAXRSS_DIV = 2**20 if platform.system() == 'Darwin' else 2**10   # ru_maxrss: bytes on macOS, KiB on Linux
MEM_LOG = []

def mem(stage: str) -> None:
    rss = _PROC.memory_info().rss / 2**20
    peak = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / _MAXRSS_DIV
    MEM_LOG.append({'stage': stage, 'rss_mb': round(rss, 1), 'peak_mb': round(peak, 1)})
    print(f'[mem] {stage:<42s} rss={rss:8.1f} MB   peak={peak:8.1f} MB')

def free(*names):
    for n in names:
        globals().pop(n, None)
    gc.collect()

RUN_INFO = {
    'seed': SEED, 'fast_mode': FAST, 'python': sys.version.split()[0], 'platform': platform.platform(),
    **{m.__name__: m.__version__ for m in [np, pd, nb, pa, scipy, plotly]},
    'started': time.strftime('%Y-%m-%d %H:%M:%S'),
}
print(json.dumps(RUN_INFO, indent=1))
mem('config')

{
 "seed": 20261025,
 "fast_mode": false,
 "python": "3.13.9",
 "platform": "macOS-27.0.1-arm64-arm-64bit-Mach-O",
 "numpy": "2.3.5",
 "pandas": "2.3.3",
 "numba": "0.62.1",
 "pyarrow": "21.0.0",
 "scipy": "1.16.3",
 "plotly": "6.3.0",
 "started": "2026-10-25 23:00:38"
}
[mem] config                                     rss=   277.0 MB   peak=   277.0 MB


## 3 Data loading & cleaning

**Source.** `/Users/andrew/main/trading/data/etfs/{SYMBOL}_clean_1min.csv`. Timestamps are naive and start at 09:30, so they are taken as America/New_York bar-open labels. A bar labelled 10:00 covers 10:00:00–10:00:59, and its close is known at 10:01.

**Known data issues, handled explicitly:**
1. **Volume source break on 2022-03-07.** Every file switches from `pitrading` (consolidated volume) to `iex` (IEX-only prints, roughly 1–3% of consolidated volume). Raw volume is not comparable across the break, so every volume feature here is *relative within the same source segment* (§4). `src` is also an ML feature.
2. **Sparse minutes.** Minutes with no prints are absent from the files, especially in the IEX era and for thin ETFs (CPER, SHY). The bars are re-gridded to one row per minute from 09:30 to the day's last real bar. Missing minutes become flat bars at the prior close with zero volume, flagged `real=0`. Signals are never taken on filled bars. A symbol-day is tradable only if the **prior** 5 sessions had at least 80% real minutes, because today's coverage is not known until the close.
3. **Adjusted prices.** Pre-2022 prices are split- and dividend-adjusted (SPY trades at about 61 in Dec 2002, and USO is about 8x its raw price because of the 2020 1:8 reverse split). Costs are quoted in *real* cents per share, so each day gets a factor `adj = data_close / raw_close`. Raw close is rebuilt from yfinance split-adjusted closes and split history. Cost in data units = real cents × `adj`. Without this, USO costs would be understated about 8x.
4. **Bad prints.** If a wick extends more than 3% beyond the bar's own open/close, it is clipped to the body. The rule uses only that bar's data. Counts are reported.
5. **Session.** Only regular-hours bars are kept (09:30–15:59 labels). Half-days end at the day's last real bar.

6. **Price seam at the source switch.** The pitrading segment is dividend-adjusted to a later date and the IEX segment is raw. That leaves a fake overnight gap on 2022-03-07 (several percent for dividend payers like TLT/IEF/SHY). The pre-switch segment is rescaled by one constant per symbol so the seam-day return equals the raw yfinance return. `seam_factor` and the gap before and after are reported in the QC table.

**Survivorship.** The dataset has 12 files. All of the ETFs are still listed, and none was delisted during the sample. VXX is excluded as VIX-linked. Delisted ETFs would be included automatically if their files were present.

In [2]:
def discover_symbols():
    syms = sorted(Path(p).name.split('_')[0] for p in glob.glob(str(DATA_DIR / FILE_PATTERN.format(sym='*'))))
    excluded = [s for s in syms if VIX_LINKED.match(s)]
    return [s for s in syms if s not in excluded], excluded

SYMBOLS, EXCLUDED = discover_symbols()
if FAST:
    SYMBOLS = [s for s in ['SPY', 'TLT', 'GLD'] if s in SYMBOLS]
print('universe:', SYMBOLS)
print('excluded (VIX-linked):', EXCLUDED)

universe: ['CPER', 'DIA', 'GLD', 'IEF', 'IWM', 'QQQ', 'SHY', 'SLV', 'SPY', 'TLT', 'USO']
excluded (VIX-linked): ['VXX']


In [3]:
YF_CACHE = DIRS['external'] / 'yfinance_daily.parquet'

def fetch_yf_daily(tickers):
    """Daily split-adjusted (not dividend-adjusted) closes + split history for cost adjustment, and ^VIX OHLC."""
    import yfinance as yf
    if YF_CACHE.exists() and not REBUILD_CACHE:
        cached = pd.read_parquet(YF_CACHE)
        if set(tickers) <= set(cached['ticker'].unique()):
            return cached
    rows = []
    for t in tickers:
        try:
            tk = yf.Ticker(t)
            h = tk.history(start='2002-01-01', auto_adjust=False, actions=True)
            if h.empty:
                print(f'  yfinance: no data for {t}')
                continue
            h.index = pd.to_datetime(h.index.tz_localize(None).date)
            splits = h['Stock Splits'].replace(0, np.nan)
            # raw = split-adjusted close * product of all split ratios that happen AFTER that date
            fwd = splits.fillna(1.0)[::-1].cumprod()[::-1].shift(-1).fillna(1.0)
            df = pd.DataFrame({'date': h.index, 'open': h['Open'], 'high': h['High'], 'low': h['Low'],
                               'close': h['Close'], 'raw_close': h['Close'] * fwd, 'ticker': t})
            rows.append(df.reset_index(drop=True))
        except Exception as e:  # network failures are reported, not hidden
            print(f'  yfinance failed for {t}: {e!r}')
    out = pd.concat(rows, ignore_index=True) if rows else pd.DataFrame(columns=['date', 'ticker'])
    out['ticker'] = out['ticker'].astype('category')
    out.to_parquet(YF_CACHE, index=False)
    return out

YF = fetch_yf_daily(SYMBOLS + ['^VIX'])
VIX = YF[YF['ticker'] == '^VIX'][['date', 'open', 'high', 'low', 'close']].sort_values('date').reset_index(drop=True)
print(f'VIX daily rows: {len(VIX)}  {VIX.date.min().date()} -> {VIX.date.max().date()}')
mem('yfinance daily')

VIX daily rows: 6245  2002-01-02 -> 2026-10-23
[mem] yfinance daily                             rss=   319.9 MB   peak=   319.9 MB


In [4]:
def _date_int(dt):
    return (dt.dt.year * 10000 + dt.dt.month * 100 + dt.dt.day).astype('int32')

def clean_symbol(sym: str) -> dict:
    """CSV -> cleaned, re-gridded, typed 1m bars in parquet (one row group per year). Returns QC stats."""
    out_path = DIRS['bars'] / f'{sym}.parquet'
    daily_path = DIRS['daily'] / f'{sym}_daily.parquet'
    stats_path = DIRS['bars'] / f'{sym}_qc.json'
    if out_path.exists() and daily_path.exists() and stats_path.exists() and not REBUILD_CACHE:
        return json.loads(stats_path.read_text())

    raw = pd.read_csv(DATA_DIR / FILE_PATTERN.format(sym=sym),
                      dtype={'Open': 'float32', 'High': 'float32', 'Low': 'float32', 'Close': 'float32',
                             'Volume': 'float64', 'source': 'category'})
    qc = {'symbol': sym, 'raw_rows': len(raw)}
    dt = pd.to_datetime(raw.pop('datetime'))
    minute = (dt.dt.hour * 60 + dt.dt.minute).astype('int16')
    raw['date'] = _date_int(dt)
    raw['minute'] = minute
    del dt
    raw = raw.rename(columns={'Open': 'o', 'High': 'h', 'Low': 'l', 'Close': 'c', 'Volume': 'v'})
    raw['src'] = (raw.pop('source').astype(str) == 'iex').astype('int8')

    # session filter, duplicates, impossible values
    rth = (raw.minute >= OPEN_MIN) & (raw.minute <= LAST_BAR_MIN)
    qc['dropped_outside_rth'] = int((~rth).sum())
    raw = raw[rth]
    n0 = len(raw)
    raw = raw.drop_duplicates(['date', 'minute'], keep='last').sort_values(['date', 'minute'], kind='stable')
    qc['dropped_duplicates'] = n0 - len(raw)
    bad = (raw[['o', 'h', 'l', 'c']] <= 0).any(axis=1) | raw[['o', 'h', 'l', 'c']].isna().any(axis=1) | (raw.v < 0)
    qc['dropped_bad_values'] = int(bad.sum())
    raw = raw[~bad]
    # OHLC consistency, then same-bar bad-print wick clipping
    body_hi = np.maximum(raw.o.values, raw.c.values)
    body_lo = np.minimum(raw.o.values, raw.c.values)
    h = np.maximum(raw.h.values, body_hi)
    l = np.minimum(raw.l.values, body_lo)
    spike_h = h > body_hi * 1.03
    spike_l = l < body_lo * 0.97
    qc['clipped_spike_wicks'] = int(spike_h.sum() + spike_l.sum())
    raw['h'] = np.where(spike_h, body_hi, h).astype('float32')
    raw['l'] = np.where(spike_l, body_lo, l).astype('float32')
    raw['v'] = raw['v'].astype('float32')

    # ---- re-grid to one row per minute, 09:30 .. last real bar of each day
    day_codes, day_idx = np.unique(raw.date.values, return_inverse=True)
    last_min = raw.groupby('date', sort=True).minute.max().values.astype(np.int32)
    n_slots = last_min - OPEN_MIN + 1
    offsets = np.concatenate([[0], np.cumsum(n_slots)[:-1]])
    total = int(n_slots.sum())
    pos = offsets[day_idx] + (raw.minute.values.astype(np.int64) - OPEN_MIN)
    grid = {
        'date': np.repeat(day_codes.astype(np.int32), n_slots),
        'minute': (np.arange(total) - np.repeat(offsets, n_slots) + OPEN_MIN).astype(np.int16),
    }
    real = np.zeros(total, np.int8); real[pos] = 1
    for col in ['o', 'h', 'l', 'c']:
        a = np.full(total, np.nan, np.float32); a[pos] = raw[col].values; grid[col] = a
    v = np.zeros(total, np.float32); v[pos] = raw.v.values; grid['v'] = v
    src = np.full(total, -1, np.int8); src[pos] = raw.src.values
    grid['real'] = real
    g = pd.DataFrame(grid)
    del raw, grid
    g['c'] = g['c'].ffill()
    for col in ['o', 'h', 'l']:
        g[col] = g[col].fillna(g['c'])
    g['src'] = pd.Series(np.where(src < 0, np.nan, src)).ffill().bfill().astype('int8').values
    g = g.dropna(subset=['c']).reset_index(drop=True)

    # ---- price seam at the 2022-03-07 source switch: the pitrading segment is dividend-adjusted to a later date while
    # the IEX segment is raw, which leaves a fake overnight gap. Rescale the pre-switch prices by one constant so the
    # seam-day close-to-close return equals the raw (yfinance) return. Constant scaling leaves every pre-seam return,
    # R-multiple and signal unchanged; costs stay right because adj = data/raw is computed after the rescale.
    yf_sym = YF[YF['ticker'] == sym][['date', 'raw_close', 'close']].rename(columns={'date': 'dt', 'close': 'yf_close'})
    qc['seam_factor'] = 1.0
    sw_days = np.unique(g.date.values[g.src.values == 1])
    pre_days = np.unique(g.date.values[g.src.values == 0])
    if len(sw_days) and len(pre_days) and len(yf_sym):
        d0, d1 = int(pre_days.max()), int(sw_days.min())
        c0 = g.c.values[g.date.values == d0][-1]; c1 = g.c.values[g.date.values == d1][-1]
        yy = yf_sym.set_index('dt')['raw_close']
        t0_, t1_ = pd.Timestamp(str(d0)), pd.Timestamp(str(d1))
        if t0_ in yy.index and t1_ in yy.index and d1 > d0:
            k = float((c1 / c0) / (yy[t1_] / yy[t0_]))
            qc['seam_factor'] = k
            qc['seam_gap_before_pct'] = float((c1 / c0 - 1) * 100)
            qc['seam_gap_raw_pct'] = float((yy[t1_] / yy[t0_] - 1) * 100)
            pre = g.date.values <= d0
            for col in ['o', 'h', 'l', 'c']:
                g.loc[pre, col] = (g.loc[pre, col].astype(np.float64) * k).astype(np.float32)

    # ---- daily table (real bars only) + cost adjustment factor
    rb = g[g.real == 1]
    d = rb.groupby('date').agg(open=('o', 'first'), high=('h', 'max'), low=('l', 'min'), close=('c', 'last'),
                               volume=('v', 'sum'), n_real=('real', 'size'), src=('src', 'last'))
    d['n_slots'] = g.groupby('date').size()
    d['coverage'] = (d.n_real / d.n_slots).astype('float32')
    d['dollar_vol'] = (d.volume * d.close).astype('float64')
    d = d.reset_index()
    d['dt'] = pd.to_datetime(d.date.astype(str))
    if len(yf_sym):
        d = d.merge(yf_sym, on='dt', how='left')
        f = (d.close / d.raw_close).where(lambda x: (x > 0.02) & (x < 50))
        # smooth day-to-day noise only WITHIN a split regime (split dates are known in advance), so the factor
        # steps immediately at a split instead of lagging it
        fwd = (d.raw_close / d.yf_close).ffill().bfill()
        regime = np.r_[0, np.cumsum(np.abs(np.diff(np.log(fwd.values))) > 1e-6)]
        d['adj'] = (f.groupby(regime).transform(lambda x: x.rolling(5, min_periods=1).median())
                    .ffill().bfill().fillna(1.0).astype('float32'))
        qc['adj_source'] = 'yfinance'
    else:
        print(f'  WARNING {sym}: no yfinance prices -> cost factor adj=1.0 (not cached, will retry next run)')
        d['adj'] = np.float32(1.0)
        qc['adj_source'] = 'none (factor=1)'
    # ---- causal liquidity tier per session -> round-trip cost in real cents (2 x (slippage + half spread))
    # trailing 252-session median dollar volume of PRIOR sessions, consolidated era only, frozen once IEX starts;
    # the first 60 sessions of a symbol default to the illiquid tier (conservative)
    med = d.dollar_vol.where(d.src == 0).rolling(252, min_periods=60).median().shift(1).ffill()
    d['tier'] = np.where(med >= TIER_DOLLAR_VOL[0], 1, np.where(med >= TIER_DOLLAR_VOL[1], 2, 3)).astype('int8')
    d['cost_rt_c'] = (2.0 * (d.tier.map(SLIP_CENTS) + d.tier.map(HALF_SPREAD_CENTS))).astype('float32')
    g = g.merge(d[['date', 'adj', 'cost_rt_c']], on='date', how='left')
    g['adj'] = g['adj'].astype('float32')
    g['cost_rt_c'] = g['cost_rt_c'].astype('float32')

    for col in ['o', 'h', 'l', 'c']:
        g[col] = g[col].astype('float32')
    g['year'] = (g.date // 10000).astype('int16')
    table = pa.Table.from_pandas(g, preserve_index=False)
    with pq.ParquetWriter(out_path, table.schema, compression='zstd') as w:
        for y in np.unique(g.year.values):
            w.write_table(table.filter(pa.compute.equal(table['year'], int(y))))
    d.drop(columns=['dt']).to_parquet(daily_path, index=False)

    qc.update({'rows_gridded': int(len(g)), 'real_rows': int(g.real.sum()), 'days': int(len(d)),
               'first_date': int(d.date.min()), 'last_date': int(d.date.max()),
               'adj_median_pre2022': float(d.loc[d.date < 20220307, 'adj'].median()) if (d.date < 20220307).any() else None})
    if qc['adj_source'] == 'yfinance':        # never cache a transient download failure
        stats_path.write_text(json.dumps(qc))
    del g, d, table, rb
    gc.collect()
    return qc

QC = []
for sym in SYMBOLS:
    t0 = time.time()
    QC.append(clean_symbol(sym))
    print(f'{sym:5s} cleaned in {time.time() - t0:5.1f}s')
    mem(f'clean {sym}')
QC = pd.DataFrame(QC).set_index('symbol')
QC

CPER  cleaned in   0.0s
[mem] clean CPER                                 rss=   321.8 MB   peak=   321.8 MB
DIA   cleaned in   0.0s
[mem] clean DIA                                  rss=   321.8 MB   peak=   321.8 MB
GLD   cleaned in   0.0s
[mem] clean GLD                                  rss=   321.8 MB   peak=   321.8 MB
IEF   cleaned in   0.0s
[mem] clean IEF                                  rss=   321.8 MB   peak=   321.8 MB
IWM   cleaned in   0.0s
[mem] clean IWM                                  rss=   321.8 MB   peak=   321.8 MB
QQQ   cleaned in   0.0s
[mem] clean QQQ                                  rss=   321.8 MB   peak=   321.8 MB
SHY   cleaned in   0.0s
[mem] clean SHY                                  rss=   321.8 MB   peak=   321.8 MB
SLV   cleaned in   0.0s
[mem] clean SLV                                  rss=   321.8 MB   peak=   321.8 MB
SPY   cleaned in   0.0s
[mem] clean SPY                                  rss=   321.8 MB   peak=   321.8 MB
TLT   cleaned in   0.0s
[mem

,raw_rows,dropped_outside_rth,dropped_duplicates,dropped_bad_values,clipped_spike_wicks,seam_factor,seam_gap_before_pct,seam_gap_raw_pct,adj_source,rows_gridded,real_rows,days,first_date,last_date,adj_median_pre2022
symbol,,,,,,,,,,,,,,,
CPER,109059,0,0,0,3,0.999332,-3.375667,-3.311036,yfinance,1060787,109059,3315,20111115,20260921,0.999332
DIA,2067170,0,0,0,2,0.929778,-9.232903,-2.377627,yfinance,2316179,2067170,5966,20021230,20260921,0.765596
GLD,1924936,0,0,0,0,1.000106,1.497006,1.486287,yfinance,2132206,1924936,5490,20041118,20260921,1.000106
IEF,1560831,0,0,0,0,0.882588,-12.116110,-0.424782,yfinance,2315010,1560831,5966,20021230,20260921,0.760660
IWM,2217669,0,0,0,1,0.947666,-7.589924,-2.486662,yfinance,2316669,2217669,5966,20021230,20260921,0.845302
QQQ,2245015,0,0,0,2,0.972573,-6.329679,-3.688112,yfinance,2317587,2245015,5968,20021230,20260921,0.896051
SHY,1523666,0,0,0,0,0.882297,-11.885077,-0.130117,yfinance,2314685,1523666,5966,20021230,20260921,0.818597
SLV,1752691,0,0,0,1,1.000000,NaN,NaN,yfinance,1991491,1752691,5128,20060428,20260921,1.000000
SPY,2250959,0,0,0,4,0.943138,-8.466446,-2.947919,yfinance,2317394,2250959,5968,20021230,20260921,0.792282


### 3.1 Coverage, liquidity tiers and cost model

Liquidity tiers are **causal and per session**: the trailing 252-session median daily dollar volume of *prior* sessions in the consolidated-volume era, frozen at its 2022-03-07 value once volume becomes IEX-only. A symbol's first 60 sessions default to the illiquid tier. Tier 3 (illiquid) gets the higher slippage setting. Costs per side = slippage + half spread, in real cents, converted to adjusted price units by `adj`. One round trip costs `2 × (slip + half_spread) × adj` per share. In R units that is divided by the trade's risk per share, so tight stops pay proportionally more.

**Why the spreads are assumed, not estimated.** The Corwin–Schultz high-low estimator, computed on consecutive 1-minute bars, is printed as a diagnostic. On 1-minute bars it is dominated by intrabar volatility: it is far too high for SPY/QQQ, whose quoted spread is about 1¢, and near zero for thin, flat-printing ETFs. It is therefore *not* used. With no quote data, a tiered assumption plus a 2× cost stress test is the honest choice.

In [5]:
def load_daily(sym):
    return pd.read_parquet(DIRS['daily'] / f'{sym}_daily.parquet')

def corwin_schultz(hi, lo):
    """Corwin & Schultz (2012) two-period high-low spread estimator (proportional spread)."""
    b = np.log(hi / lo) ** 2
    beta = b + np.roll(b, -1)
    gamma = np.log(np.maximum(hi, np.roll(hi, -1)) / np.minimum(lo, np.roll(lo, -1))) ** 2
    k = 3 - 2 * np.sqrt(2)
    alpha = (np.sqrt(2 * beta) - np.sqrt(beta)) / k - np.sqrt(gamma / k)
    s = 2 * (np.exp(alpha) - 1) / (1 + np.exp(alpha))
    return np.clip(s[:-1], 0, None)

rows = []
for sym in SYMBOLS:
    d = load_daily(sym)
    pre, post = d[d.date < 20220307], d[d.date >= 20220307]
    mdv = pre.dollar_vol.median() if len(pre) else np.nan
    tier = int(d.tier.iloc[-1])                                   # tier in force at the end of the sample
    tier_share = d.tier.value_counts(normalize=True).reindex([1, 2, 3], fill_value=0)
    # CS estimate from consecutive real 1m bars over the last 60 sessions (cents, raw price): a DIAGNOSTIC only
    b = pd.read_parquet(DIRS['bars'] / f'{sym}.parquet', columns=['date', 'h', 'l', 'c', 'real', 'adj'],
                        filters=[('year', '>=', int(d.date.max() // 10000) - 1)])
    b = b[(b.real == 1) & (b.date >= np.unique(b.date)[-60])]
    cs_day = [np.mean(corwin_schultz(x.h.values.astype(float), x.l.values.astype(float))) * (x.c / x.adj).mean() * 100
              for _, x in b.groupby('date') if len(x) >= 30]
    cs = float(np.median(cs_day)) if cs_day else np.nan
    del b
    rows.append({'symbol': sym, 'days': len(d), 'first': d.date.min(), 'last': d.date.max(),
                 'coverage_consolidated': pre.coverage.mean() if len(pre) else np.nan,
                 'coverage_iex': post.coverage.mean() if len(post) else np.nan,
                 'tradable_days_frac': (d.coverage.rolling(5).mean().shift(1) >= MIN_COVERAGE).mean(),
                 'median_dollar_vol_m': mdv / 1e6, 'tier_now': tier, 'days_tier1': tier_share[1], 'days_tier2': tier_share[2],
                 'days_tier3': tier_share[3], 'roundtrip_c_now': float(d.cost_rt_c.iloc[-1]),
                 'CS_est_1m_c (diagnostic)': cs})
UNIVERSE = pd.DataFrame(rows).set_index('symbol')
TIER = UNIVERSE['tier_now'].to_dict()      # display only; costs use the causal per-session tier
display(UNIVERSE.round(3))
mem('universe stats')

,days,first,last,coverage_consolidated,coverage_iex,tradable_days_frac,median_dollar_vol_m,tier_now,days_tier1,days_tier2,days_tier3,roundtrip_c_now,CS_est_1m_c (diagnostic)
symbol,,,,,,,,,,,,,
CPER,3315,20111115,20260921,0.104,0.092,0.000,0.065,3,0.000,0.000,1.000,9.0,0.006
DIA,5966,20021230,20260921,0.965,0.586,0.834,569.395,1,0.245,0.745,0.010,3.0,0.524
GLD,5490,20041118,20260921,0.957,0.697,0.848,789.554,1,0.453,0.480,0.067,3.0,1.331
IEF,5966,20021230,20260921,0.701,0.560,0.357,54.383,2,0.000,0.453,0.547,4.0,0.021
IWM,5966,20021230,20260921,0.957,0.960,0.974,2302.330,1,0.870,0.094,0.036,3.0,2.143
QQQ,5968,20021230,20260921,0.969,0.966,0.999,2827.000,1,0.990,0.000,0.010,3.0,5.125
SHY,5966,20021230,20260921,0.750,0.268,0.444,57.829,2,0.000,0.336,0.664,4.0,0.004
SLV,5128,20060428,20260921,0.945,0.653,0.768,134.906,2,0.000,0.721,0.279,4.0,0.371
SPY,5968,20021230,20260921,0.971,0.973,0.999,11475.634,1,0.990,0.000,0.010,3.0,4.428


[mem] universe stats                             rss=   399.6 MB   peak=   399.6 MB


## 4 Feature engineering

Features that don't depend on search parameters are computed once per symbol and cached as parquet (one row group per year, LZ4). Parameter-dependent quantities (swing levels, pullback extremes, the sweep/reclaim state machine) are computed per trial in §5.

**Lookahead rules enforced here** (and tested automatically in §10.6):
* Every rolling statistic is computed over past bars. Normalizers (ATR, mean volume) use values through bar *t−1* when they scale bar *t*.
* Higher-timeframe (HTF) trend: a 1-minute bar inside HTF bucket *b* sees only HTF bucket *b−1*, the last **completed** HTF bar. It never sees the bucket it sits in.
* Relative volume vs. time of day uses the same minute over the **previous** 20 sessions, *within the same volume-source segment* (consolidated vs IEX), so the 2022 volume break cannot leak into the ratios.
* Daily trend, coverage gate, gap, and VIX use only **prior-day** values.

**All "order flow" here is an OHLCV proxy.** With 1-minute bars there is no bid/ask, tape or footprint. `clv`, `delta_proxy`, `effort_result`, and the wick ratios are inferences from where the bar closed within its range and how much volume traded. They are not measurements of aggressor flow.

In [6]:
TREND_TFS = (5, 15, 30, 60)          # HTF candidates in minutes (search parameter)
TREND_LENS = (10, 20, 50)            # EMA lengths on the HTF closes (search parameter)
ATR_N = 14

def wilder(x, n):
    return pd.Series(x).ewm(alpha=1.0 / n, adjust=False, min_periods=n).mean().values

def htf_trend(date, minute, c, h, l, tf):
    """Trend direction & strength per 1m bar from the last COMPLETED tf-minute bar. Returns dict of arrays."""
    bucket_in_day = (minute.astype(np.int32) - OPEN_MIN) // tf
    key = date.astype(np.int64) * 1000 + bucket_in_day
    change = np.r_[True, key[1:] != key[:-1]]
    bid = np.cumsum(change) - 1                                  # sequential HTF bar id per 1m bar
    last_idx = np.r_[np.nonzero(change)[0][1:] - 1, len(c) - 1]  # last 1m bar of each HTF bar
    hc = c[last_idx].astype(np.float64)
    hh = pd.Series(h).groupby(bid).max().values
    hl = pd.Series(l).groupby(bid).min().values
    prev_c = np.r_[hc[0], hc[:-1]]
    tr = np.maximum(hh - hl, np.maximum(np.abs(hh - prev_c), np.abs(hl - prev_c)))
    hatr = wilder(tr, ATR_N)
    out = {}
    for n in TREND_LENS:
        ema = pd.Series(hc).ewm(span=n, adjust=False, min_periods=n).mean().values
        slope = (ema - np.r_[np.full(3, np.nan), ema[:-3]]) / hatr          # EMA change over 3 HTF bars, in HTF ATRs
        d = np.where((hc > ema) & (slope > 0), 1, np.where((hc < ema) & (slope < 0), -1, 0)).astype(np.int8)
        d[np.isnan(slope)] = 0
        # shift by one HTF bar: 1m bar in HTF bar b uses HTF bar b-1
        d_prev = np.r_[np.int8(0), d[:-1]]
        s_prev = np.r_[np.nan, slope[:-1]]
        out[f'tdir_{tf}_{n}'] = d_prev[bid]
        out[f'tslope_{tf}_{n}'] = s_prev[bid].astype(np.float32)
    return out

def relvol_time_of_day(date, minute, v, src, lookback=20):
    """v / mean(v at same minute over previous `lookback` sessions in the same source segment)."""
    days, di = np.unique(date, return_inverse=True)
    slot = (minute.astype(np.int32) - OPEN_MIN)
    M = np.full((len(days), BARS_PER_DAY), np.nan, np.float32)
    M[di, slot] = v
    day_src = pd.Series(src).groupby(di).last().values
    base = np.full_like(M, np.nan)
    for s in np.unique(day_src):
        rows = np.nonzero(day_src == s)[0]
        sub = pd.DataFrame(M[rows])
        base[rows] = sub.rolling(lookback, min_periods=5).mean().shift(1).values   # previous sessions only
    b = base[di, slot]
    return (v / np.maximum(b, 1.0)).astype(np.float32)

def daily_from_bars(g):
    """Daily OHLC/coverage from 1m bars (same logic as §3), so features can be rebuilt from truncated data."""
    rb = g[g.real == 1]
    d = rb.groupby('date').agg(open=('o', 'first'), high=('h', 'max'), low=('l', 'min'), close=('c', 'last'))
    d['coverage'] = rb.groupby('date').size() / g.groupby('date').size()
    return d.reset_index()

def compute_features(g: pd.DataFrame) -> pd.DataFrame:
    """All parameter-independent features from a symbol's gridded bars. Pure function of `g` (and prior-day VIX)."""
    date, minute = g.date.values, g.minute.values
    o, h, l, c, v = (g[k].values.astype(np.float64) for k in ['o', 'h', 'l', 'c', 'v'])
    src = g.src.values
    prev_c = np.r_[c[0], c[:-1]]
    tr = np.maximum(h - l, np.maximum(np.abs(h - prev_c), np.abs(l - prev_c)))
    atr = wilder(tr, ATR_N)
    atr_prev = np.r_[np.nan, atr[:-1]]
    rng = h - l
    f = {}
    f['atr'] = atr.astype(np.float32)                      # includes bar t (known at t close)
    f['atr_prev'] = atr_prev.astype(np.float32)            # through t-1: normaliser for bar t
    f['clv'] = np.where(rng > 0, ((c - l) - (h - c)) / np.where(rng > 0, rng, 1), 0).astype(np.float32)   # PROXY
    vmean30 = pd.Series(v).rolling(30, min_periods=10).mean().shift(1).values
    f['relvol_bar'] = (v / np.maximum(vmean30, 1.0)).astype(np.float32)                                  # PROXY
    f['relvol_tod'] = relvol_time_of_day(date, minute, v.astype(np.float32), src)                         # PROXY
    f['delta_proxy'] = (f['relvol_bar'] * f['clv']).astype(np.float32)                                    # PROXY: signed effort
    f['range_atr'] = (rng / atr_prev).astype(np.float32)
    f['body_atr'] = (np.abs(c - o) / atr_prev).astype(np.float32)
    f['lower_wick'] = np.where(rng > 0, (np.minimum(o, c) - l) / np.where(rng > 0, rng, 1), 0).astype(np.float32)
    f['upper_wick'] = np.where(rng > 0, (h - np.maximum(o, c)) / np.where(rng > 0, rng, 1), 0).astype(np.float32)
    # effort vs result: relative volume per unit of price progress (high = absorption)                     # PROXY
    f['effort_result'] = (f['relvol_tod'] / np.maximum(f['range_atr'], 0.25)).astype(np.float32)
    # session VWAP (uses bars <= t; IEX-era VWAP is IEX-volume weighted)
    tp = (h + l + c) / 3
    day_start = np.r_[True, date[1:] != date[:-1]]
    gid = np.cumsum(day_start)
    cpv = pd.Series(tp * v).groupby(gid).cumsum().values
    cv = pd.Series(v).groupby(gid).cumsum().values
    vwap = np.where(cv > 0, cpv / np.maximum(cv, 1e-9), tp)
    f['vwap_dist'] = ((c - vwap) / atr).astype(np.float32)
    lr = np.r_[0, np.diff(np.log(c))]
    lr[day_start] = 0
    f['rv30'] = (pd.Series(lr).rolling(30, min_periods=10).std().values * np.sqrt(BARS_PER_DAY)).astype(np.float32)
    for tf in TREND_TFS:
        f.update(htf_trend(date, minute, c, h, l, tf))

    # ---- prior-day information (daily table)
    d = daily_from_bars(g).sort_values('date')
    d['cov_prev5'] = d.coverage.rolling(5, min_periods=3).mean().shift(1)
    sma20 = d.close.rolling(20).mean()
    d['dtrend_prev'] = np.sign(d.close - sma20).shift(1).fillna(0).astype('int8')
    dtr = np.maximum(d.high - d.low, np.maximum((d.high - d.close.shift()).abs(), (d.low - d.close.shift()).abs()))
    d['datr_prev'] = dtr.ewm(alpha=1 / 14, adjust=False).mean().shift(1)
    d['prev_close'] = d.close.shift(1)
    # prior-day VIX: last VIX close strictly before this session; percentile vs the 252 VIX closes before that
    vx = VIX[['date', 'close']].copy()
    vx['pct'] = vx.close.rolling(252, min_periods=60).apply(lambda a: (a[:-1] < a[-1]).mean(), raw=True)
    vx['vdate'] = (vx.date.dt.year * 10000 + vx.date.dt.month * 100 + vx.date.dt.day).astype('int32')
    d = pd.merge_asof(d.sort_values('date'), vx[['vdate', 'close', 'pct']].rename(columns={'close': 'vix_prev', 'pct': 'vix_pct_prev'}),
                      left_on='date', right_on='vdate', direction='backward', allow_exact_matches=False)
    dm = d.set_index('date')
    di = pd.Index(dm.index).get_indexer(date)
    f['tradable'] = (dm.cov_prev5.values[di] >= MIN_COVERAGE).astype(np.int8)
    f['dtrend'] = dm.dtrend_prev.values[di].astype(np.int8)
    f['gap_datr'] = ((dm.open.values[di] - dm.prev_close.values[di]) / dm.datr_prev.values[di]).astype(np.float32)
    f['vix_prev'] = dm.vix_prev.values[di].astype(np.float32)
    f['vix_pct_prev'] = dm.vix_pct_prev.values[di].astype(np.float32)

    feats = pd.DataFrame(f)
    return pd.concat([g[['date', 'minute', 'o', 'h', 'l', 'c', 'v', 'real', 'src', 'adj', 'cost_rt_c', 'year']].reset_index(drop=True), feats], axis=1)

def build_features(sym: str) -> dict:
    out_path = DIRS['feats'] / f'{sym}.parquet'
    if out_path.exists() and not REBUILD_CACHE:
        return {'symbol': sym, 'cached': True}
    g = pd.read_parquet(DIRS['bars'] / f'{sym}.parquet')
    out = compute_features(g)
    table = pa.Table.from_pandas(out, preserve_index=False)
    with pq.ParquetWriter(out_path, table.schema, compression='lz4') as w:
        for y in np.unique(out.year.values):
            w.write_table(table.filter(pa.compute.equal(table['year'], int(y))))
    n_cols = out.shape[1]
    del g, out, table
    gc.collect()
    return {'symbol': sym, 'cached': False, 'columns': n_cols}

for sym in SYMBOLS:
    t0 = time.time()
    info = build_features(sym)
    print(f'{sym:5s} features in {time.time() - t0:5.1f}s {info}')
    mem(f'features {sym}')

FEATURE_COLUMNS = pq.read_schema(DIRS['feats'] / f'{SYMBOLS[0]}.parquet').names
print(len(FEATURE_COLUMNS), 'cached columns:', FEATURE_COLUMNS)

CPER  features in   0.0s {'symbol': 'CPER', 'cached': True}


[mem] features CPER                              rss=   399.9 MB   peak=   399.9 MB
DIA   features in   0.0s {'symbol': 'DIA', 'cached': True}
[mem] features DIA                               rss=   399.9 MB   peak=   399.9 MB
GLD   features in   0.0s {'symbol': 'GLD', 'cached': True}
[mem] features GLD                               rss=   399.9 MB   peak=   399.9 MB
IEF   features in   0.0s {'symbol': 'IEF', 'cached': True}
[mem] features IEF                               rss=   399.9 MB   peak=   399.9 MB
IWM   features in   0.0s {'symbol': 'IWM', 'cached': True}
[mem] features IWM                               rss=   399.9 MB   peak=   399.9 MB
QQQ   features in   0.0s {'symbol': 'QQQ', 'cached': True}
[mem] features QQQ                               rss=   399.9 MB   peak=   399.9 MB
SHY   features in   0.0s {'symbol': 'SHY', 'cached': True}
[mem] features SHY                               rss=   399.9 MB   peak=   399.9 MB
SLV   features in   0.0s {'symbol': 'SLV', 'cached': True

In [7]:
def load_feats(sym, columns=None, date_from=None, date_to=None):
    """Read one symbol's cached bars+features for [date_from, date_to] (yyyymmdd ints, inclusive)."""
    filters = []
    if date_from is not None:
        filters.append(('year', '>=', date_from // 10000))
    if date_to is not None:
        filters.append(('year', '<=', date_to // 10000))
    df = pq.read_table(DIRS['feats'] / f'{sym}.parquet', columns=columns, filters=filters or None).to_pandas()
    m = np.ones(len(df), bool)
    if date_from is not None:
        m &= df.date.values >= date_from
    if date_to is not None:
        m &= df.date.values <= date_to
    return df[m].reset_index(drop=True) if not m.all() else df

_s = load_feats(SYMBOLS[0], date_from=20200301, date_to=20200310)
display(_s[['date', 'minute', 'c', 'v', 'real', 'atr', 'relvol_tod', 'clv', 'tdir_15_20', 'tradable', 'vix_prev', 'vix_pct_prev']].iloc[::400].head(12))
print(_s.memory_usage().sum() / 1e6, 'MB for', len(_s), 'rows')
free('_s'); mem('features done')

,date,minute,c,v,real,atr,relvol_tod,clv,tdir_15_20,tradable,vix_prev,vix_pct_prev
0,20200302,570,16.009293,323.0,1,0.019141,0.146638,-1.0,-1,0,40.110001,1.000000
400,20200303,586,16.039270,0.0,0,0.008904,0.000000,0.0,0,0,33.419998,0.992032
800,20200304,597,16.149199,0.0,0,0.001096,0.000000,0.0,1,0,36.820000,0.992032
1200,20200305,607,16.079245,0.0,0,0.000819,0.000000,0.0,-1,0,31.990000,0.984064
1600,20200306,642,15.889371,0.0,0,0.002301,0.000000,0.0,-1,0,39.619999,0.996016
2000,20200309,664,15.489639,0.0,0,0.004017,0.000000,0.0,-1,0,41.939999,1.000000
2400,20200310,681,15.759459,0.0,0,0.001008,0.000000,0.0,1,0,54.459999,1.000000


0.435716 MB for 2656 rows
[mem] features done                              rss=   429.8 MB   peak=   429.8 MB


## 5 Signal generation

The setup implements the rules derived in §1.3. **Long side** (shorts mirror it):

| Step | Rule (1-minute bars) | Book basis | Parameters |
|---|---|---|---|
| Trend | HTF bar closed above its EMA with a rising EMA (last *completed* HTF bar). Optional prior-day daily trend agreement | "Trade in harmony with the trend", line of least resistance (Sect. 7M) | `trend_tf`, `trend_len`, `daily_align` |
| Waves | Causal zigzag with reversal threshold `z`·ATR gives the latest up-wave (impulse: base → top) and the current reaction from the top | Buying and selling waves (5M), Wave Chart (22M) | `z`, `imp_min` |
| Pullback | Reaction depth `rr = (top − low)/(top − base)` within `[rr_min, rr_max]` | Normal reaction about half-way; past 50% is weakness | `rr_min`, `rr_max` |
| Sweep (shake-out) | Low trades below the swing-low level (the "line of supports") by `sd_min`–`sd_max` ATR | Shake-out under supports; stop-running drives below old bottoms | `swing_mode`, `swing_lb`, `sd_min`, `sd_max` |
| Reclaim | A close back above the level within `m` bars. If price runs deeper than `sd_max` ATR first, the setup is cancelled as genuine liquidation | "Immediate sharp recovery" marks a shake-out | `m_reclaim` |
| Volume (PROXY) | Sweep volume either climactic (`relvol ≥ v_hi`: stops run and absorbed) or dried up (`relvol ≤ v_lo`: test, no supply), per `vol_mode` | Volume surge = culmination; shrinking volume on a test of support | `vol_mode`, `v_hi`, `v_lo` |
| Effort vs result (PROXY) | Relative volume per ATR of range at the sweep ≥ `er_min` (much effort, little downside result) | Momentum vs braking power; volume out of proportion to progress | `er_min` |
| Wave volume (PROXY) | Mean relvol of the reaction / mean relvol of the impulse ≤ `wv_max` | Volume shrinks on reactions in a healthy uptrend | `wv_max` |
| Close location (PROXY) | Reclaim-bar CLV ≥ `clv_min` | Close near the high ends the reaction | `clv_min` |
| Bar delta (PROXY) | Σ relvol·CLV over sweep→reclaim > 0 | Demand overcoming supply in the struggle | `delta_conf` |
| Recovery volume (PROXY) | Reclaim relvol / mean sweep relvol ≥ `rv_ratio_min` | Volume increasing on the recovery confirms | `rv_ratio_min` |
| Stop | Sweep extreme − `stop_buf`·ATR. Risk bounded to `[risk_min, risk_max]` ATR | Stops just under the danger point / extreme low | `stop_buf`, `risk_min`, `risk_max` |
| Cost filter | Skip the setup if round-trip cost / risk per share > `max_cost_R` (both known at the signal close) | Wyckoff's insistence on minimum risk vs objective | `max_cost_R` |
| Target | `R_mult`·R, or the impulse top (opposing liquidity) if it is ≥ `min_R` away | Objective at the prior top; reward/risk filter | `target_mode`, `R_mult`, `min_R` |
| Exits | Time stop `time_stop` bars; flat 15:55; at most `max_trades_day` per symbol | — | `time_stop`, `max_trades_day` |

**Lookahead.** The kernel walks forward one bar at a time. A decision at bar *t* reads only arrays at indices ≤ *t*, and the zigzag state is updated with bar *t* only after bar *t*'s measurements are taken. Swing levels end at least 2 bars before *t*. A pivot low is used only after its `k` confirming bars have closed. The signal fires at the reclaim bar's close, and the engine enters at the next open. §10.6 checks all of this by recomputing on truncated data.

In [8]:
@nb.njit
def wyckoff_kernel(h, l, c, atr, rv, clv, delta, er, tdir, dtr, ok, real, minute, day, lvl_lo, lvl_hi, cum_rv, cost_ps, max_cost_R,
                   z, imp_min, rr_min, rr_max, sd_min, sd_max, m_reclaim,
                   vol_mode, v_hi, v_lo, er_min, clv_min, wv_max, delta_conf, rv_ratio_min,
                   stop_buf, risk_min, risk_max, side, daily_align, first_min, last_min):
    n = len(c)
    sig = np.zeros(n, np.int8)
    stop = np.full(n, np.nan)
    tgt = np.full(n, np.nan)
    cap = n // 20 + 16
    info = np.full((cap, 12), np.nan, np.float64)   # see INFO_COLS
    info_idx = np.empty(cap, np.int64)
    ni = 0
    # ---- zigzag state
    mode = 0                      # 0 = undefined, 1 = up-wave, -1 = down-wave
    rh = h[0]; rh_i = 0; rl = l[0]; rl_i = 0
    lo_since_hi = l[0]; lo_since_hi_i = 0
    hi_since_lo = h[0]; hi_since_lo_i = 0
    B = np.nan; B_i = -1          # last confirmed bottom
    T = np.nan; T_i = -1          # last confirmed top
    top_base = np.nan; top_base_i = -1     # bottom that started the up-wave ending at T
    bot_base = np.nan; bot_base_i = -1     # top that started the down-wave ending at B
    # ---- pending sweep state (long = 0, short = 1)
    p_on = np.zeros(2, np.bool_); p_lvl = np.zeros(2); p_ext = np.zeros(2); p_start = np.zeros(2, np.int64)
    p_atr = np.zeros(2); p_ref = np.zeros(2); p_rr = np.zeros(2); p_imp = np.zeros(2); p_wv = np.zeros(2)
    p_rvmax = np.zeros(2); p_rvsum = np.zeros(2); p_ermax = np.zeros(2); p_dsum = np.zeros(2)

    for t in range(1, n):
        a = atr[t]
        if day[t] != day[t - 1]:          # a sweep can only be reclaimed within its own session
            p_on[0] = False
            p_on[1] = False
        # ================= measurements from state through t-1 (plus bar t's own range) =================
        if mode == 1:
            top, top_i, base, base_i = rh, rh_i, B, B_i
            bot, bot_i, bbase, bbase_i = B, B_i, bot_base, bot_base_i
        elif mode == -1:
            top, top_i, base, base_i = T, T_i, top_base, top_base_i
            bot, bot_i, bbase, bbase_i = rl, rl_i, T, T_i
        else:
            top = np.nan; top_i = -1; base = np.nan; base_i = -1
            bot = np.nan; bot_i = -1; bbase = np.nan; bbase_i = -1

        emitted = False
        for s in range(2):
            d = 1 if s == 0 else -1
            if side == 1 and d == -1:
                continue
            if side == 2 and d == 1:
                continue
            new = False
            if not p_on[s]:
                # ---------- detect a sweep at bar t ----------
                if not (a > 0) or ok[t] == 0 or real[t] == 0 or tdir[t] != d:
                    continue
                if daily_align == 1 and dtr[t] != d:
                    continue
                if minute[t] < first_min or minute[t] > last_min:
                    continue
                if d == 1:
                    lvl = lvl_lo[t]
                    if not (lvl == lvl) or base_i < 0 or top_i <= base_i:
                        continue
                    depth = (lvl - l[t]) / a
                    imp = (top - base) / a
                    if not (depth >= sd_min and depth <= sd_max and depth > 0 and imp >= imp_min):
                        continue
                    rr = (top - l[t]) / (top - base)
                    if not (rr >= rr_min and rr <= rr_max):
                        continue
                    i_len = top_i - base_i
                    p_len = (t - 1) - top_i
                    wv = np.nan
                    if i_len > 0 and p_len > 0:
                        wv = ((cum_rv[t - 1] - cum_rv[top_i]) / p_len) / max((cum_rv[top_i] - cum_rv[base_i]) / i_len, 1e-6)
                    ref = top
                else:
                    lvl = lvl_hi[t]
                    if not (lvl == lvl) or bbase_i < 0 or bot_i <= bbase_i:
                        continue
                    depth = (h[t] - lvl) / a
                    imp = (bbase - bot) / a
                    if not (depth >= sd_min and depth <= sd_max and depth > 0 and imp >= imp_min):
                        continue
                    rr = (h[t] - bot) / (bbase - bot)
                    if not (rr >= rr_min and rr <= rr_max):
                        continue
                    i_len = bot_i - bbase_i
                    p_len = (t - 1) - bot_i
                    wv = np.nan
                    if i_len > 0 and p_len > 0:
                        wv = ((cum_rv[t - 1] - cum_rv[bot_i]) / p_len) / max((cum_rv[bot_i] - cum_rv[bbase_i]) / i_len, 1e-6)
                    ref = bot
                if wv_max > 0 and not (wv <= wv_max):
                    continue
                p_on[s] = True; new = True
                p_lvl[s] = lvl; p_ext[s] = l[t] if d == 1 else h[t]; p_start[s] = t; p_atr[s] = a
                p_ref[s] = ref; p_rr[s] = rr; p_imp[s] = imp; p_wv[s] = wv
                p_rvmax[s] = rv[t]; p_rvsum[s] = rv[t]; p_ermax[s] = er[t]; p_dsum[s] = delta[t]
            if p_on[s] and not new:
                # ---------- update a pending sweep with bar t ----------
                if d == 1:
                    p_ext[s] = min(p_ext[s], l[t])
                    deep = (p_lvl[s] - p_ext[s]) / p_atr[s]
                else:
                    p_ext[s] = max(p_ext[s], h[t])
                    deep = (p_ext[s] - p_lvl[s]) / p_atr[s]
                if deep > sd_max:            # genuine liquidation, not a shake-out
                    p_on[s] = False
                    continue
                if (c[t] - p_lvl[s]) * d <= 0:   # not reclaimed yet: this bar is part of the sweep
                    p_rvmax[s] = max(p_rvmax[s], rv[t]); p_rvsum[s] += rv[t]
                    p_ermax[s] = max(p_ermax[s], er[t])
                p_dsum[s] += delta[t]
            if p_on[s]:
                # ---------- reclaim check at bar t ----------
                if (c[t] - p_lvl[s]) * d > 0:
                    p_on[s] = False
                    if emitted or ok[t] == 0 or real[t] == 0 or minute[t] > last_min:
                        continue
                    k = t - p_start[s]
                    sweep_bars = k if k > 0 else 1
                    rvm = p_rvmax[s]
                    vol_ok = True
                    if vol_mode == 1:
                        vol_ok = rvm >= v_hi
                    elif vol_mode == 2:
                        vol_ok = rvm <= v_lo
                    elif vol_mode == 3:
                        vol_ok = (rvm >= v_hi) or (rvm <= v_lo)
                    if not vol_ok:
                        continue
                    if er_min > 0 and not (p_ermax[s] >= er_min):
                        continue
                    if not (clv[t] * d >= clv_min):
                        continue
                    if delta_conf == 1 and not (p_dsum[s] * d > 0):
                        continue
                    rec_ratio = rv[t] / max(p_rvsum[s] / sweep_bars, 1e-3) if k > 0 else 1.0
                    if rv_ratio_min > 0 and not (rec_ratio >= rv_ratio_min):
                        continue
                    st = p_ext[s] - d * stop_buf * p_atr[s]
                    risk = (c[t] - st) * d / p_atr[s]
                    if not (risk >= risk_min and risk <= risk_max):
                        continue
                    if max_cost_R > 0 and not (cost_ps[t] / ((c[t] - st) * d) <= max_cost_R):
                        continue          # cost-aware filter: round-trip cost known at signal time
                    sig[t] = d; stop[t] = st; tgt[t] = p_ref[s]; emitted = True
                    if ni < cap:
                        info_idx[ni] = t
                        info[ni, 0] = (p_lvl[s] - p_ext[s]) * d / p_atr[s]   # sweep depth (ATR)
                        info[ni, 1] = p_rr[s]                                 # reaction depth (fraction of impulse)
                        info[ni, 2] = p_imp[s]                                # impulse size (ATR)
                        info[ni, 3] = p_wv[s]                                 # reaction/impulse volume ratio
                        info[ni, 4] = rvm                                     # max relvol during sweep
                        info[ni, 5] = rec_ratio                               # reclaim volume vs sweep volume
                        info[ni, 6] = p_ermax[s]                              # effort/result at sweep
                        info[ni, 7] = p_dsum[s] * d                           # signed delta proxy, sweep->reclaim
                        info[ni, 8] = k                                       # bars to reclaim
                        info[ni, 9] = (p_ref[s] - c[t]) * d / p_atr[s]        # distance to opposing liquidity (ATR)
                        info[ni, 10] = p_lvl[s]                               # swept level (price; plots only)
                        info[ni, 11] = p_ext[s]                               # sweep extreme (price; plots only)
                        ni += 1
                elif t - p_start[s] >= m_reclaim:
                    p_on[s] = False

        # ================= update zigzag with bar t =================
        if mode == 0:
            if a > 0:
                if h[t] - rl > z * a:
                    mode = 1; B = rl; B_i = rl_i; rh = h[t]; rh_i = t; lo_since_hi = h[t]; lo_since_hi_i = t
                elif rh - l[t] > z * a:
                    mode = -1; T = rh; T_i = rh_i; rl = l[t]; rl_i = t; hi_since_lo = l[t]; hi_since_lo_i = t
            if mode == 0:
                if h[t] > rh:
                    rh = h[t]; rh_i = t
                if l[t] < rl:
                    rl = l[t]; rl_i = t
        elif mode == 1:
            if h[t] > rh:
                rh = h[t]; rh_i = t; lo_since_hi = h[t]; lo_since_hi_i = t   # its own low does not count
            elif l[t] < lo_since_hi:
                lo_since_hi = l[t]; lo_since_hi_i = t
            if a > 0 and rh - lo_since_hi > z * a:      # top confirmed
                T = rh; T_i = rh_i; top_base = B; top_base_i = B_i
                mode = -1; rl = lo_since_hi; rl_i = lo_since_hi_i
                hi_since_lo = l[rl_i]; hi_since_lo_i = rl_i            # the low bar's own high does not count
                for q in range(rl_i + 1, t + 1):
                    if h[q] > hi_since_lo:
                        hi_since_lo = h[q]; hi_since_lo_i = q
        else:
            if l[t] < rl:
                rl = l[t]; rl_i = t; hi_since_lo = l[t]; hi_since_lo_i = t   # its own high does not count
            elif h[t] > hi_since_lo:
                hi_since_lo = h[t]; hi_since_lo_i = t
            if a > 0 and hi_since_lo - rl > z * a:      # bottom confirmed
                B = rl; B_i = rl_i; bot_base = T; bot_base_i = T_i
                mode = 1; rh = hi_since_lo; rh_i = hi_since_lo_i
                lo_since_hi = h[rh_i]; lo_since_hi_i = rh_i            # the high bar's own low does not count
                for q in range(rh_i + 1, t + 1):
                    if l[q] < lo_since_hi:
                        lo_since_hi = l[q]; lo_since_hi_i = q
    return sig, stop, tgt, info_idx[:ni], info[:ni]

INFO_COLS = ['sweep_depth_atr', 'reaction_rr', 'impulse_atr', 'wave_vol_ratio', 'sweep_relvol',
             'reclaim_vol_ratio', 'sweep_effort_result', 'delta_proxy_sum', 'bars_to_reclaim', 'dist_liq_atr',
             'sweep_level', 'sweep_extreme']
INFO_ML = INFO_COLS[:10]          # price-level columns are for charts, never model inputs

In [9]:
def swing_levels(low, high, mode, lb, k):
    """Liquidity levels that bar t may sweep, built only from bars <= t-2 (rolling) or confirmed pivots."""
    lo, hi = pd.Series(low), pd.Series(high)
    if mode == 'rolling':
        gap = 2
        lvl_lo = lo.rolling(lb, min_periods=lb).min().shift(gap).values
        lvl_hi = hi.rolling(lb, min_periods=lb).max().shift(gap).values
    else:   # fractal pivot with k bars each side, usable only after the k right-hand bars have closed
        w = 2 * k + 1
        piv_lo = lo == lo.rolling(w, center=True, min_periods=w).min()
        piv_hi = hi == hi.rolling(w, center=True, min_periods=w).max()
        lvl_lo = lo.where(piv_lo).shift(k).ffill(limit=lb).values
        lvl_hi = hi.where(piv_hi).shift(k).ffill(limit=lb).values
    return lvl_lo, lvl_hi

SIGNAL_COLS = ['date', 'minute', 'o', 'h', 'l', 'c', 'real', 'adj', 'cost_rt_c', 'atr', 'atr_prev', 'relvol_tod', 'clv',
               'delta_proxy', 'effort_result', 'tradable', 'dtrend']
ML_BAR_COLS = ['src', 'relvol_bar', 'range_atr', 'body_atr', 'lower_wick', 'upper_wick', 'vwap_dist', 'rv30',
               'gap_datr', 'vix_prev', 'vix_pct_prev']

def needed_columns(param_list, with_ml=False):
    cols = list(SIGNAL_COLS)
    for p in param_list:
        cols += [f"tdir_{p['trend_tf']}_{p['trend_len']}", f"tslope_{p['trend_tf']}_{p['trend_len']}"]
    if with_ml:
        cols += ML_BAR_COLS
    return list(dict.fromkeys(cols))

def round_trip_cost_ps(sym, F):
    """Round-trip cost per share at 1x, in adjusted price units, per bar."""
    return F['cost_rt_c'].values.astype(np.float64) / 100.0 * F['adj'].values.astype(np.float64)

def generate_signals(sym, F, p):
    """Vectorized prep + numba kernel. F: one symbol's feature frame. Returns (sig, stop, tgt, info_idx, info)."""
    lvl_lo, lvl_hi = swing_levels(F['l'].values, F['h'].values, p['swing_mode'], p['swing_lb'], p.get('pivot_k', 3))
    rv = np.nan_to_num(F['relvol_tod'].values.astype(np.float64), nan=1.0)
    cum_rv = np.cumsum(rv)
    atr = F['atr_prev'].values.astype(np.float64)
    cost_ps = round_trip_cost_ps(sym, F)
    return wyckoff_kernel(
        F['h'].values.astype(np.float64), F['l'].values.astype(np.float64), F['c'].values.astype(np.float64),
        atr, rv, F['clv'].values.astype(np.float64), np.nan_to_num(F['delta_proxy'].values.astype(np.float64)),
        np.nan_to_num(F['effort_result'].values.astype(np.float64)),
        F[f"tdir_{p['trend_tf']}_{p['trend_len']}"].values, F['dtrend'].values, F['tradable'].values, F['real'].values,
        F['minute'].values.astype(np.int64), F['date'].values.astype(np.int64), lvl_lo, lvl_hi, cum_rv, cost_ps, p['max_cost_R'],
        p['z'], p['imp_min'], p['rr_min'], p['rr_max'], p['sd_min'], p['sd_max'], p['m_reclaim'],
        p['vol_mode'], p['v_hi'], p['v_lo'], p['er_min'], p['clv_min'], p['wv_max'], p['delta_conf'], p['rv_ratio_min'],
        p['stop_buf'], p['risk_min'], p['risk_max'], p['side'], p['daily_align'],
        OPEN_MIN + p['open_skip'], p['last_entry_min'])

def run_symbol(sym, F, p, with_ml=False):
    """Signals + simulation for one symbol and one parameter set -> trade DataFrame (gross R, cost_R at 1x)."""
    sig, stop, tgt, info_idx, info = generate_signals(sym, F, p)
    use_R = p['target_mode'] == 'R'
    cps = round_trip_cost_ps(sym, F)
    res = simulate(F['o'].values.astype(np.float64), F['h'].values.astype(np.float64), F['l'].values.astype(np.float64),
                   F['c'].values.astype(np.float64), F['date'].values.astype(np.int64), F['minute'].values.astype(np.int64),
                   sig, stop, tgt, use_R, p['R_mult'], p['min_R'], p['time_stop'], p['max_trades_day'],
                   p['last_entry_min'] + 1, FLAT_MIN, F['real'].values.astype(np.int8), F['atr_prev'].values.astype(np.float64),
                   p['risk_min'], cps, p['max_cost_R'])
    t = trades_frame(sym, F, res, cps)
    t['n_signals'] = int((sig != 0).sum())
    if with_ml and len(t):
        pos = pd.Series(np.arange(len(info_idx)), index=info_idx)
        rows = pos.reindex(t['sig_idx'].values).values
        for j, col in enumerate(INFO_COLS):
            t[col] = info[rows.astype(np.int64), j] if len(info_idx) else np.nan
        si = t['sig_idx'].values
        for col in ['relvol_tod', 'clv', 'delta_proxy', 'effort_result'] + ML_BAR_COLS:
            t[f'f_{col}'] = F[col].values[si]
        t['f_trend_slope'] = F[f"tslope_{p['trend_tf']}_{p['trend_len']}"].values[si] * t['dir'].values
        t['f_dtrend_agree'] = (F['dtrend'].values[si] == t['dir'].values).astype(np.int8)
        t['f_atr_pct'] = F['atr'].values[si] / F['c'].values[si]
        # risk & cost measured at the SIGNAL close (the trade log's risk_ps/cost_R use the next open -> not features)
        est_risk = (F['c'].values[si] - t['stop'].values) * t['dir'].values
        t['f_risk_atr_sig'] = est_risk / F['atr'].values[si]
        t['f_cost_R_sig'] = cps[si] / est_risk
        t['f_tgt_R_sig'] = (t['target'].values - F['c'].values[si]) * t['dir'].values / est_risk if p['target_mode'] == 'liq' else p['R_mult']
        t['f_sym'] = SYMBOLS.index(sym) if sym in SYMBOLS else -1
        t['f_minute'] = F['minute'].values[si]
        t['f_dow'] = pd.to_datetime(t['date'].astype(str)).dt.dayofweek.values
    return t

DEFAULT_PARAMS = dict(
    trend_tf=15, trend_len=20, daily_align=0, side=0, z=2.5, imp_min=3.0, rr_min=0.25, rr_max=0.6,
    swing_mode='rolling', swing_lb=20, pivot_k=3, sd_min=0.05, sd_max=1.0, m_reclaim=3,
    vol_mode=0, v_hi=2.0, v_lo=0.7, er_min=0.0, clv_min=0.3, wv_max=0.0, delta_conf=0, rv_ratio_min=0.0,
    stop_buf=0.1, risk_min=0.3, risk_max=3.0, target_mode='R', R_mult=1.5, min_R=1.0, time_stop=30,
    max_trades_day=3, open_skip=15, max_cost_R=0.0, confirm='none', last_entry_min=15 * 60 + 30, sizing='fixed')

## 6 Backtest engine

The trade simulation is path-dependent: one position per symbol, fills depend on intrabar order, plus time stops and daily caps. It is a single numba loop.

**Fill rules (conservative):**
* Signal on bar *i* is known at *i*'s close. **Entry is at the open of bar *i+1*.** No same-bar fills.
* Risk per share is recomputed from the *actual* entry price: `risk = |entry − stop|`. If the next open is already beyond the stop, or the liquidity target is closer than `min_R` × risk, the trade is skipped and counted.
* The stop is checked from the entry bar onward, with the target. **If both are inside the same bar, the stop is assumed hit first.** If a later bar opens beyond the stop, the fill is at that (worse) open. Targets fill at the target level and never at a better gap price.
* Time stop: exit at the close of the `time_stop`-th bar after entry. End of day: exit at the close of the 15:54 bar (flat by 15:55), or at the day's last bar on half-days.
* Costs: round trip = 2 × (slippage + half spread) real cents × `adj`, charged in R as `cost / risk_per_share` × cost multiplier.

**Portfolio layer (after the per-symbol loop, on the small trade lists):** 0.5% of equity is risked per trade. Each position's notional is capped at 4× equity, and the **sum of open positions' notional is capped at 4× equity** (Reg-T intraday buying power): a new trade is scaled down to fit or skipped. The daily loss limit is portfolio-wide and **latches**: once realized P&L of the day's closed trades reaches −2R (2 × 0.5% of equity, risk-weighted), no further entries that day, even if an open winner later recovers it. The daily return is Σ(effective risk × net R), and days with no trades count as zero-return days in Sharpe/Sortino.

**Entry checks at the actual fill:** no fill on a minute without a print, and the risk floor (`risk_min`·ATR) and cost filter are re-checked against the real next-bar open, not only the signal close.

In [10]:
EXIT_REASONS = np.array(['stop', 'target', 'time', 'eod'])

@nb.njit
def simulate(o, h, l, c, day, minute, sig, stop_lvl, tgt_lvl, use_R_target, R_mult, min_R,
             time_stop, max_trades_day, last_entry_min, flat_bar_min, real, atr_ref, risk_min, cost_ps, max_cost_R):
    """Path-dependent single-symbol simulation. Returns arrays of trade fields (gross; costs applied later)."""
    n = len(o)
    cap = n // 2 + 1
    t_sig = np.empty(cap, np.int64); t_ent = np.empty(cap, np.int64); t_ext = np.empty(cap, np.int64)
    t_dir = np.empty(cap, np.int8); t_reason = np.empty(cap, np.int8)
    t_entry = np.empty(cap, np.float64); t_exit = np.empty(cap, np.float64)
    t_stop = np.empty(cap, np.float64); t_tgt = np.empty(cap, np.float64); t_risk = np.empty(cap, np.float64)
    nt = 0
    skipped = 0
    i = 0
    cur_day = -1
    trades_today = 0
    while i < n - 1:
        if day[i] != cur_day:
            cur_day = day[i]
            trades_today = 0
        s = sig[i]
        if s == 0 or trades_today >= max_trades_day:
            i += 1
            continue
        j = i + 1
        if day[j] != day[i] or minute[j] > last_entry_min or minute[j] >= flat_bar_min:
            i += 1
            continue
        if real[j] == 0:            # no print in the entry minute: a filled bar's open is the signal close -> no fill
            skipped += 1
            i += 1
            continue
        entry = o[j]
        stop = stop_lvl[i]
        risk = (entry - stop) * s
        # re-check the risk floor and the cost filter at the ACTUAL entry (a gap toward the stop shrinks risk)
        if not (risk > 0) or risk < risk_min * atr_ref[i] or (max_cost_R > 0 and cost_ps[j] / risk > max_cost_R):
            skipped += 1
            i += 1
            continue
        if use_R_target:
            tgt = entry + s * R_mult * risk
        else:
            tgt = tgt_lvl[i]
            if not ((tgt - entry) * s >= min_R * risk):
                skipped += 1
                i += 1
                continue
        # walk forward from the entry bar
        k = j
        reason = -1
        px = 0.0
        while True:
            if s == 1:
                hit_stop = l[k] <= stop
                hit_tgt = h[k] >= tgt
            else:
                hit_stop = h[k] >= stop
                hit_tgt = l[k] <= tgt
            if hit_stop:                                    # stop first when both touched
                reason = 0
                if k > j and (o[k] - stop) * s < 0:        # gapped through the stop: fill at the open
                    px = o[k]
                else:
                    px = stop
                break
            if hit_tgt:
                reason = 1
                px = tgt
                break
            last_of_day = (k == n - 1) or (day[k + 1] != day[k])
            if minute[k] >= flat_bar_min - 1 or last_of_day:
                reason = 3
                px = c[k]
                break
            if k - j + 1 >= time_stop:
                reason = 2
                px = c[k]
                break
            k += 1
        t_sig[nt] = i; t_ent[nt] = j; t_ext[nt] = k; t_dir[nt] = s; t_reason[nt] = reason
        t_entry[nt] = entry; t_exit[nt] = px; t_stop[nt] = stop; t_tgt[nt] = tgt; t_risk[nt] = risk
        nt += 1
        trades_today += 1
        i = k          # a new signal may fire on the exit bar's close (entry on the following bar)
        if i == j - 1:
            i += 1
    return (t_sig[:nt], t_ent[:nt], t_ext[:nt], t_dir[:nt], t_reason[:nt], t_entry[:nt], t_exit[:nt],
            t_stop[:nt], t_tgt[:nt], t_risk[:nt], skipped)


def trades_frame(sym, bars, res, cost_ps_bar):
    """Assemble a trade DataFrame from simulate() output; costs in R at 1x (scale by COST_MULTS later)."""
    t_sig, t_ent, t_ext, t_dir, t_reason, t_entry, t_exit, t_stop, t_tgt, t_risk, skipped = res
    cost_ps = cost_ps_bar[t_ent]                                    # round trip per share at the entry session, data units
    gross = (t_exit - t_entry) * t_dir / t_risk
    df = pd.DataFrame({
        'sym': sym, 'date': bars['date'].values[t_ent], 'sig_idx': t_sig, 'entry_idx': t_ent, 'exit_idx': t_ext,
        'entry_min': bars['minute'].values[t_ent].astype(np.int16), 'exit_min': bars['minute'].values[t_ext].astype(np.int16),
        'dir': t_dir, 'reason': t_reason, 'entry': t_entry, 'exit': t_exit, 'stop': t_stop, 'target': t_tgt,
        'risk_ps': t_risk, 'risk_atr': t_risk / bars['atr'].values[t_sig], 'R_gross': gross,
        'cost_R': cost_ps / t_risk, 'lev_req': RISK_PER_TRADE * t_entry / t_risk,
        'bars_held': (t_ext - t_ent + 1).astype(np.int32),
    })
    df['sym'] = df['sym'].astype('category')
    df.attrs['skipped'] = int(skipped)
    return df


@nb.njit
def _portfolio_pass(day_key, entry_t, exit_t, R, rf0, notional0, limit, gross_cap, risk_unit):
    """Trades sorted by (day, entry_t). Path-dependent portfolio rules, per day:
    * daily loss LATCH: once the running realized P&L of kept trades (risk-weighted, in R = RISK_PER_TRADE of equity)
      reaches -limit at any exit before this entry, no further entries that day;
    * gross notional cap: open notional + new notional <= gross_cap (x equity); a new trade is scaled down to fit, or skipped."""
    n = len(R)
    keep = np.zeros(n, np.bool_)
    rf = np.zeros(n)
    start = 0
    while start < n:
        end = start
        while end < n and day_key[end] == day_key[start]:
            end += 1
        for a in range(start, end):
            # realized path of kept trades closed before this entry (exit bar closes before entry bar opens)
            m = 0
            ex = np.empty(end - start)
            pl = np.empty(end - start)
            for b in range(start, a):
                if keep[b] and exit_t[b] < entry_t[a]:
                    ex[m] = exit_t[b]
                    pl[m] = R[b] * rf[b] / risk_unit
                    m += 1
            latched = False
            if m:
                order = np.argsort(ex[:m])
                cum = 0.0
                for q in range(m):
                    cum += pl[:m][order[q]]
                    if cum <= -limit:
                        latched = True
                        break
            if latched:
                continue
            open_notional = 0.0
            for b in range(start, a):
                if keep[b] and exit_t[b] >= entry_t[a]:
                    open_notional += notional0[b] * rf[b] / rf0[b] if rf0[b] > 0 else 0.0
            room = gross_cap - open_notional
            if room <= 1e-9:
                continue
            scale = min(1.0, room / notional0[a]) if notional0[a] > 0 else 1.0
            keep[a] = True
            rf[a] = rf0[a] * scale
        start = end
    return keep, rf


def apply_portfolio(trades, cost_mult=1.0, sizing='fixed'):
    """Net R at the given cost multiple, risk sizing with per-position and gross notional caps, daily loss latch."""
    if trades is None or len(trades) == 0:
        return pd.DataFrame(columns=['date', 'entry_min', 'exit_min', 'R', 'ret', 'risk_frac'])
    t = trades.sort_values(['date', 'entry_min', 'sym'], kind='stable').reset_index(drop=True)
    t['R'] = t['R_gross'] - cost_mult * t['cost_R']
    risk = RISK_PER_TRADE * np.ones(len(t))
    if sizing == 'atr_scaled' and 'risk_atr' in t:
        risk *= np.clip(1.0 / np.maximum(t['risk_atr'].values, 1e-6), 0.5, 1.5)
    lev = risk / RISK_PER_TRADE * t['lev_req'].values                    # notional / equity at the base risk
    rf0 = risk * np.minimum(1.0, MAX_POS_LEVERAGE / np.maximum(lev, 1e-12))
    notional0 = rf0 / RISK_PER_TRADE * t['lev_req'].values
    keep, rf = _portfolio_pass(t['date'].values.astype(np.int64), t['entry_min'].values.astype(np.int64),
                               t['exit_min'].values.astype(np.int64), t['R'].values.astype(np.float64), rf0, notional0,
                               DAILY_LOSS_LIMIT_R, MAX_GROSS_LEVERAGE, RISK_PER_TRADE)
    t = t[keep].reset_index(drop=True)
    t['risk_frac'] = rf[keep]
    t['ret'] = t['risk_frac'] * t['R']
    return t


def daily_returns(t, calendar):
    """Daily portfolio return on every calendar trading day (zeros on no-trade days)."""
    r = t.groupby('date')['ret'].sum() if len(t) else pd.Series(dtype=float)
    return r.reindex(calendar, fill_value=0.0).astype(float)


def metrics(t, calendar, label=None):
    """Standard trade + equity metrics. `t` from apply_portfolio, `calendar` = trading days of the period."""
    n_days = len(calendar)
    r = daily_returns(t, calendar)
    eq = (1 + r).cumprod()
    dd = eq / np.maximum.accumulate(np.r_[1.0, eq.values])[1:] - 1        # peak includes the starting equity of 1
    R = t['R'].values if len(t) else np.array([])
    wins, losses = R[R > 0], R[R <= 0]
    down = r[r < 0]
    sd = r.std(ddof=1)
    dsd = np.sqrt((np.minimum(r, 0) ** 2).mean())
    years = n_days / 252
    out = {
        'trades': int(len(R)),
        'win_rate': float(len(wins) / len(R)) if len(R) else np.nan,
        'expectancy_R': float(R.mean()) if len(R) else np.nan,
        'avg_R_gross': float(t['R_gross'].mean()) if len(R) else np.nan,
        'avg_win_R': float(wins.mean()) if len(wins) else np.nan,
        'avg_loss_R': float(losses.mean()) if len(losses) else np.nan,
        'profit_factor': float(wins.sum() / -losses.sum()) if len(losses) and losses.sum() < 0 else np.nan,
        'sharpe': float(r.mean() / sd * np.sqrt(252)) if sd > 0 else np.nan,
        'sortino': float(r.mean() / dsd * np.sqrt(252)) if dsd > 0 else np.nan,
        'max_dd': float(dd.min()) if n_days else np.nan,
        'cagr': float(eq.iloc[-1] ** (1 / years) - 1) if n_days and years > 0 else np.nan,
        'total_R': float(R.sum()) if len(R) else 0.0,
        'trades_per_day': float(len(R) / n_days) if n_days else np.nan,
        'days': n_days,
    }
    if label is not None:
        out = {'variant': label, **out}
    return out

### 6.1 Engine unit tests on synthetic bars
Each test builds a few bars by hand and checks one fill rule. These run before any real data is touched.

In [11]:
def _bars(rows, minute0=600):
    a = np.array(rows, dtype=np.float64)
    n = len(a)
    return dict(o=a[:, 0], h=a[:, 1], l=a[:, 2], c=a[:, 3], day=np.zeros(n, np.int64),
                minute=(minute0 + np.arange(n)).astype(np.int64))

def _run(b, sig_at, direction, stop, tgt=np.nan, use_R=True, R_mult=2.0, time_stop=50, min_R=0.0):
    n = len(b['o'])
    sig = np.zeros(n, np.int8); sig[sig_at] = direction
    st = np.full(n, np.nan); st[sig_at] = stop
    tg = np.full(n, np.nan); tg[sig_at] = tgt
    n_ = len(b['o'])
    return simulate(b['o'], b['h'], b['l'], b['c'], b['day'], b['minute'], sig, st, tg, use_R, R_mult, min_R,
                    time_stop, 5, 15 * 60 + 50, FLAT_MIN, b.get('real', np.ones(n_, np.int8)), np.ones(n_), b.get('risk_min', 0.0),
                    np.zeros(n_), 0.0)

# 1) entry at next open, target at 2R
b = _bars([[100, 100.2, 99.8, 100], [100, 100.1, 99.9, 100.0], [100, 101.1, 99.95, 101], [101, 101, 101, 101]])
r = _run(b, 0, 1, 99.5)
assert r[1][0] == 1 and r[5][0] == 100.0 and r[4][0] == 1 and abs(r[6][0] - 101.0) < 1e-12, r
# 2) both stop and target inside one bar -> stop first
b = _bars([[100, 100, 100, 100], [100, 101.5, 99.0, 100.5], [100.5, 100.5, 100.5, 100.5]])
r = _run(b, 0, 1, 99.5)
assert r[4][0] == 0 and r[6][0] == 99.5, r
# 3) gap through stop -> fill at the worse open
b = _bars([[100, 100, 100, 100], [100, 100.2, 99.8, 100], [99.0, 99.2, 98.8, 99.0]])
r = _run(b, 0, 1, 99.5)
assert r[4][0] == 0 and r[6][0] == 99.0, r
# 4) next open already beyond stop -> skipped
b = _bars([[100, 100, 100, 100], [99.4, 99.6, 99.3, 99.5], [99.5, 99.5, 99.5, 99.5]])
r = _run(b, 0, 1, 99.5)
assert len(r[0]) == 0 and r[10] == 1, r
# 5) time stop exits at the close of the time_stop-th bar after entry
b = _bars([[100, 100, 100, 100]] + [[100, 100.1, 99.9, 100.05]] * 10)
r = _run(b, 0, 1, 99.0, time_stop=3)
assert r[4][0] == 2 and r[2][0] == 3 and r[6][0] == 100.05, r
# 6) end-of-day: exit at the close of the 15:54 bar
b = _bars([[100, 100, 100, 100]] + [[100, 100.1, 99.9, 100.02]] * 10, minute0=15 * 60 + 48)
r = _run(b, 0, 1, 99.0, time_stop=100)
assert r[4][0] == 3 and b['minute'][r[2][0]] == 15 * 60 + 54, r
# 7) short side mirror + liquidity target with min_R filter
b = _bars([[100, 100, 100, 100], [100, 100.1, 99.0, 99.2], [99.2, 99.2, 99.2, 99.2]])
r = _run(b, 0, -1, 100.5, tgt=99.1, use_R=False, min_R=1.5)
assert r[4][0] == 1 and r[6][0] == 99.1, r
r = _run(b, 0, -1, 100.5, tgt=99.6, use_R=False, min_R=1.5)   # target only 0.8R away -> skipped
assert len(r[0]) == 0, r
# 8) portfolio daily loss limit
_t = pd.DataFrame({'sym': 'X', 'date': 1, 'entry_min': [600, 610, 620, 630], 'exit_min': [605, 615, 625, 640],
                   'R_gross': [-1.0, -1.0, 2.0, 1.0], 'cost_R': 0.0, 'lev_req': 1.0, 'risk_atr': 1.0})
_p = apply_portfolio(_t)
assert list(_p.entry_min) == [600, 610], _p
# 9) the loss limit LATCHES: a later winner that closes does not re-arm entries
_t = pd.DataFrame({'sym': 'X', 'date': 1, 'entry_min': [600, 601, 606, 701], 'exit_min': [700, 605, 610, 720],
                   'R_gross': [3.0, -1.0, -1.0, 1.0], 'cost_R': 0.0, 'lev_req': 1.0, 'risk_atr': 1.0})
assert list(apply_portfolio(_t).entry_min) == [600, 601, 606]
# 10) gross notional cap: 4 open 1x positions fill the 4x cap, a 5th is skipped; a 2x trade with 1x room is halved
_t = pd.DataFrame({'sym': list('ABCDE'), 'date': 1, 'entry_min': [600, 601, 602, 603, 604], 'exit_min': [650] * 5,
                   'R_gross': 1.0, 'cost_R': 0.0, 'lev_req': [1.0, 1.0, 1.0, 1.0, 1.0], 'risk_atr': 1.0})
assert len(apply_portfolio(_t)) == 4
_t = pd.DataFrame({'sym': list('ABCD'), 'date': 1, 'entry_min': [600, 601, 602, 603], 'exit_min': [650] * 4,
                   'R_gross': 1.0, 'cost_R': 0.0, 'lev_req': [1.0, 1.0, 1.0, 2.0], 'risk_atr': 1.0})
_p = apply_portfolio(_t)
assert np.isclose(_p.risk_frac.iloc[-1], RISK_PER_TRADE * 0.5), _p
# 11) no fill when the entry minute has no print (filled bar)
b = _bars([[100, 100, 100, 100], [100, 100, 100, 100], [100, 101.1, 99.9, 101], [101, 101, 101, 101]])
b['real'] = np.array([1, 0, 1, 1], np.int8)
r = _run(b, 0, 1, 99.5)
assert len(r[0]) == 0 and r[10] == 1, r
# 12) risk floor re-checked at the actual entry: next open gaps to within 0.1 ATR of the stop -> skipped
b = _bars([[100, 100, 100, 100], [99.55, 99.9, 99.5, 99.8], [99.8, 99.8, 99.8, 99.8]])
b['risk_min'] = 0.3
r = _run(b, 0, 1, 99.5)
assert len(r[0]) == 0 and r[10] == 1, r
# 13) max drawdown counts a loss on the very first day
assert np.isclose(metrics(pd.DataFrame({'date': [1, 2], 'R': [-1.0, 1.0], 'R_gross': [-1.0, 1.0], 'ret': [-0.01, 0.005]}),
                          np.array([1, 2]))['max_dd'], -0.01)
print('engine unit tests: all 13 passed')
free('_t', '_p', 'b', 'r')

engine unit tests: all 13 passed


### 6.2 Stage check: one ETF, short date range
Before scaling up, the full signal → engine → portfolio path runs on SPY for 2019–2020 with the default (un-optimized) parameters. The checks: timing, trade counts, exit-reason mix, and invariants that must hold for every trade (entry after the signal bar, stops on the correct side, the flat-time rule).

In [12]:
CALENDAR = sorted(set().union(*[set(load_daily(s).date.values.tolist()) for s in SYMBOLS]))
CALENDAR = np.array(CALENDAR, dtype=np.int64)

def cal_between(d0, d1):
    return CALENDAR[(CALENDAR >= d0) & (CALENDAR <= d1)]

t0 = time.time()
_F = load_feats('SPY', columns=needed_columns([DEFAULT_PARAMS], with_ml=True), date_from=20190101, date_to=20201231)
_t = run_symbol('SPY', _F, DEFAULT_PARAMS, with_ml=True)
print(f'SPY 2019-2020: {len(_F):,} bars, {_t.n_signals.iloc[0] if len(_t) else 0} signals, {len(_t)} trades, '
      f'{_t.attrs.get("skipped", 0)} skipped entries, {time.time() - t0:.2f}s (incl. numba compile)')
t0 = time.time(); _ = run_symbol('SPY', _F, DEFAULT_PARAMS); print(f'warm run: {time.time() - t0:.3f}s')

# invariants
assert (_t.entry_idx == _t.sig_idx + 1).all(), 'entry must be the bar after the signal'
assert ((_t.stop - _t.entry) * _t.dir < 0).all(), 'stop must be on the losing side of the entry'
assert (_t.exit_min <= FLAT_MIN - 1).all(), 'positions must be flat by 15:55'
assert (_t.exit_idx >= _t.entry_idx).all()
assert (_t.R_gross >= -1.0 - 1e-9)[_t.reason != 0].all(), 'only gap-through stops may lose more than 1R gross'
assert (_F.date.values[_t.exit_idx] == _F.date.values[_t.entry_idx]).all(), 'no overnight holds'
_p = apply_portfolio(_t)
display(pd.DataFrame([metrics(_p, cal_between(20190101, 20201231), 'SPY default 2019-20 (1x costs)')]).round(3))
display(_t.groupby(EXIT_REASONS[_t.reason]).agg(n=('R_gross', 'size'), mean_R_gross=('R_gross', 'mean'),
                                                mean_cost_R=('cost_R', 'mean'), bars=('bars_held', 'mean')).round(3))
display(_t[['date', 'entry_min', 'dir', 'entry', 'stop', 'target', 'exit', 'R_gross', 'cost_R', 'bars_held'] + INFO_COLS[:5]].head(8).round(3))
free('_F', '_t', '_p', '_'); mem('single-ETF stage check')

SPY 2019-2020: 195,976 bars, 614 signals, 540 trades, 9 skipped entries, 1.50s (incl. numba compile)
warm run: 0.018s


,variant,trades,win_rate,expectancy_R,avg_R_gross,avg_win_R,avg_loss_R,profit_factor,sharpe,sortino,max_dd,cagr,total_R,trades_per_day,days
0,SPY default 2019-20 (1x costs),537,0.466,-0.11,0.157,1.228,-1.276,0.838,0.005,0.008,-0.058,-0.001,-59.236,1.063,505


,n,mean_R_gross,mean_cost_R,bars
stop,288,-1.009,0.270,4.653
target,246,1.500,0.265,5.935
time,6,0.526,0.171,30.000


,date,entry_min,dir,entry,stop,target,exit,R_gross,cost_R,bars_held,sweep_depth_atr,reaction_rr,impulse_atr,wave_vol_ratio,sweep_relvol
0,20190102,693,1,224.420,224.200,224.749,224.200,-1.0,0.123,5,0.659,0.400,12.750,1.338,0.562
1,20190102,701,1,224.505,224.058,225.175,224.058,-1.0,0.060,18,0.633,0.448,12.228,1.258,1.158
2,20190107,810,1,230.041,229.878,230.285,229.878,-1.0,0.166,1,0.081,0.506,6.548,0.867,1.481
3,20190108,593,1,230.569,230.422,230.790,230.422,-1.0,0.183,2,0.152,0.357,11.701,0.899,0.796
4,20190110,603,-1,230.531,230.642,230.366,230.366,1.5,0.245,2,0.237,0.319,13.728,1.106,0.960
5,20190110,614,-1,230.588,230.763,230.325,230.325,1.5,0.154,3,0.817,0.375,14.577,0.988,0.735
6,20190110,880,1,232.286,232.169,232.460,232.460,1.5,0.232,3,0.526,0.353,15.316,0.728,0.630
7,20190114,595,-1,231.512,231.709,231.217,231.709,-1.0,0.137,10,0.793,0.418,12.906,0.943,1.052


[mem] single-ETF stage check                     rss=   616.0 MB   peak=   616.0 MB


## 7 Parameter search

**Design.** Search is a budgeted Optuna TPE run (multivariate, seeded), **one independent search per split, on that split's IS window only**. It does not use an exhaustive grid. The space has about 30 dimensions, so a full grid would be astronomically large, and every extra configuration adds to the multiple-testing burden that §10 charges for.

**Every trial is logged**, including failures (too few trades, exceptions), with its parameters, status, IS metrics and IS daily-return series. The trial count and the cross-trial Sharpe variance feed the Deflated Sharpe Ratio. The trials × days return matrix feeds CSCV/PBO.

**Memory.** Trials are evaluated in batches of `BATCH` configurations. Each symbol's IS bars are loaded once per batch, the whole batch is run on them, and then they're freed. One symbol's bars are in memory at a time, and IO is amortized across the batch.

**Objective.** Annualized Sharpe of daily portfolio returns over the IS window: 1× costs, the portfolio daily-loss limit, the leverage cap, zero-return days included. Trials with fewer than `MIN_TRADES_PER_IS_YEAR` trades per IS year get a fixed score of −3 and are logged as `too_few_trades`, so the sampler learns to avoid them.

In [13]:
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)
RUN_INFO['optuna'] = optuna.__version__

def _ymd(d):
    return int(d.year * 10000 + d.month * 100 + d.day)

DATA_START, DATA_END = int(CALENDAR[0]), int(CALENDAR[-1])
_y0 = DATA_START // 10000 + (1 if DATA_START % 10000 > 1215 else 0)       # 2002-12-30 start -> years counted from 2003
def _is_end(n_years):
    return _ymd(pd.Timestamp(year=_y0 + n_years, month=1, day=1) - pd.Timedelta(days=1))

def _next_day(d):
    return int(CALENDAR[CALENDAR > d][0])

SPLIT_DEF = {f'IS{n}y': (DATA_START, _is_end(n), _next_day(_is_end(n)), DATA_END) for n in range(1, 6)}
_mid = int(CALENDAR[len(CALENDAR) // 2])
SPLIT_DEF['HALF'] = (DATA_START, int(CALENDAR[CALENDAR < _mid][-1]), _mid, DATA_END)
WF_FOLDS = {}
_y = _y0 + WF_FIRST_IS_YEARS
while _ymd(pd.Timestamp(year=_y, month=1, day=1)) <= DATA_END:
    oos_end = min(DATA_END, _ymd(pd.Timestamp(year=_y + WF_STEP_YEARS, month=1, day=1) - pd.Timedelta(days=1)))
    if (DATA_END // 10000) - (_y + WF_STEP_YEARS) < WF_STEP_YEARS - 1:
        oos_end = DATA_END       # the last fold absorbs a short remainder
    _ie = _ymd(pd.Timestamp(year=_y, month=1, day=1) - pd.Timedelta(days=1))
    WF_FOLDS[f'WF{_y}'] = (DATA_START, _ie, _next_day(_ie), oos_end)
    if oos_end >= DATA_END:
        break
    _y += WF_STEP_YEARS
if FAST:
    SPLITS = ['IS3y', 'HALF']
    WF_FOLDS = dict(list(WF_FOLDS.items())[:2])
SPLIT_TABLE = pd.DataFrame({k: dict(zip(['is_start', 'is_end', 'oos_start', 'oos_end'], v)) for k, v in {**{s: SPLIT_DEF[s] for s in SPLITS}, **WF_FOLDS}.items()}).T
SPLIT_TABLE['is_days'] = [len(cal_between(r.is_start, r.is_end)) for r in SPLIT_TABLE.itertuples()]
SPLIT_TABLE['oos_days'] = [len(cal_between(r.oos_start, r.oos_end)) for r in SPLIT_TABLE.itertuples()]
SPLIT_TABLE

,is_start,is_end,oos_start,oos_end,is_days,oos_days
IS1y,20021230,20031231,20040102,20260921,254,5715
IS2y,20021230,20041231,20050103,20260921,506,5463
IS3y,20021230,20051231,20060103,20260921,758,5211
IS4y,20021230,20061231,20070103,20260921,1009,4960
IS5y,20021230,20071231,20080102,20260921,1260,4709
HALF,20021230,20141104,20141105,20260921,2984,2985
WF2008,20021230,20071231,20080102,20101231,1260,757
WF2011,20021230,20101231,20110103,20131231,2017,754
WF2014,20021230,20131231,20140102,20161231,2771,756
WF2017,20021230,20161231,20170103,20191231,3527,754


In [14]:
def suggest_params(trial):
    p = {}
    p['trend_tf'] = trial.suggest_categorical('trend_tf', list(TREND_TFS))
    p['trend_len'] = trial.suggest_categorical('trend_len', list(TREND_LENS))
    p['daily_align'] = trial.suggest_categorical('daily_align', [0, 1])
    p['side'] = trial.suggest_categorical('side', [0, 1, 2])                       # both / long / short
    p['z'] = trial.suggest_float('z', 1.5, 5.0, step=0.25)
    p['imp_min'] = trial.suggest_float('imp_min', 1.0, 8.0, step=0.5)
    p['rr_min'] = trial.suggest_float('rr_min', 0.1, 0.4, step=0.05)
    p['rr_max'] = trial.suggest_float('rr_max', 0.45, 1.0, step=0.05)
    p['swing_mode'] = trial.suggest_categorical('swing_mode', ['rolling', 'pivot'])
    p['swing_lb'] = trial.suggest_int('swing_lb', 5, 60, step=5)
    p['pivot_k'] = trial.suggest_int('pivot_k', 2, 5) if p['swing_mode'] == 'pivot' else 3
    p['sd_min'] = trial.suggest_float('sd_min', 0.0, 0.4, step=0.05)
    p['sd_max'] = trial.suggest_float('sd_max', 0.5, 2.5, step=0.25)
    p['m_reclaim'] = trial.suggest_int('m_reclaim', 0, 10)
    # Volume confirmation recipes = the book's distinct readings of a shake-out / test (all OHLCV PROXIES)
    p['confirm'] = trial.suggest_categorical('confirm', ['none', 'climax', 'dry', 'recovery', 'either'])
    p.update(vol_mode=0, v_hi=2.0, v_lo=0.7, er_min=0.0, wv_max=0.0, delta_conf=0, rv_ratio_min=0.0)
    if p['confirm'] == 'climax':        # stops run on heavy volume, absorbed (effort without downside result)
        p['vol_mode'] = 1
        p['v_hi'] = trial.suggest_float('v_hi', 1.25, 3.0, step=0.25)
        p['er_min'] = trial.suggest_categorical('er_min', [0.0, 1.0, 1.5])
    elif p['confirm'] == 'dry':         # test of support on shrinking volume; reaction volume < impulse volume
        p['vol_mode'] = 2
        p['v_lo'] = trial.suggest_float('v_lo', 0.5, 1.0, step=0.1)
        p['wv_max'] = trial.suggest_categorical('wv_max', [0.0, 0.8, 1.0, 1.2])
    elif p['confirm'] == 'recovery':    # volume increases on the recovery; net (proxy) buying sweep->reclaim
        p['rv_ratio_min'] = trial.suggest_categorical('rv_ratio_min', [0.75, 1.0, 1.25])
        p['delta_conf'] = trial.suggest_categorical('delta_conf', [0, 1])
    elif p['confirm'] == 'either':      # climactic OR dried-up, never the ambiguous middle
        p['vol_mode'] = 3
        p['v_hi'] = trial.suggest_float('v_hi', 1.25, 3.0, step=0.25)
        p['v_lo'] = trial.suggest_float('v_lo', 0.5, 1.0, step=0.1)
    p['clv_min'] = trial.suggest_float('clv_min', 0.0, 0.7, step=0.1)
    p['stop_buf'] = trial.suggest_float('stop_buf', 0.0, 0.5, step=0.05)
    p['risk_min'] = trial.suggest_float('risk_min', 0.1, 1.0, step=0.1)
    p['risk_max'] = p['risk_min'] + trial.suggest_float('risk_span', 1.0, 4.0, step=0.5)
    p['max_cost_R'] = trial.suggest_categorical('max_cost_R', [0.0, 0.2, 0.3, 0.4, 0.5])
    p['target_mode'] = trial.suggest_categorical('target_mode', ['R', 'liq'])
    p['R_mult'] = trial.suggest_float('R_mult', 1.0, 4.0, step=0.25)
    p['min_R'] = trial.suggest_float('min_R', 0.75, 2.0, step=0.25)
    p['time_stop'] = trial.suggest_int('time_stop', 5, 120, step=5)
    p['max_trades_day'] = trial.suggest_categorical('max_trades_day', [1, 2, 3, 5])
    p['open_skip'] = trial.suggest_categorical('open_skip', [0, 5, 15, 30])
    p['last_entry_min'] = trial.suggest_categorical('last_entry_min', [840, 870, 900, 930, 945])
    p['sizing'] = trial.suggest_categorical('sizing', ['fixed', 'atr_scaled'])
    return p

def evaluate_params(param_list, d0, d1, symbols=None, with_ml=False):
    """Run every config in param_list over [d0, d1], loading each symbol once. Returns list of trade frames (or Exception)."""
    symbols = symbols or SYMBOLS
    per = [[] for _ in param_list]
    errs = [None] * len(param_list)
    cols = needed_columns(param_list, with_ml)
    for sym in symbols:
        F = load_feats(sym, cols, d0, d1)
        if len(F) < 2000:
            continue
        for k, p in enumerate(param_list):
            if errs[k] is not None:
                continue
            try:
                per[k].append(run_symbol(sym, F, p, with_ml))
            except Exception as e:
                errs[k] = e
        del F
        gc.collect()
    out = []
    for k in range(len(param_list)):
        if errs[k] is not None:
            out.append(errs[k])
        else:
            frames = [x for x in per[k] if len(x)]
            out.append(pd.concat(frames, ignore_index=True) if frames else pd.DataFrame())
    return out

def score_trades(trades, cal, p, cost_mult=1.0):
    port = apply_portfolio(trades, cost_mult, p.get('sizing', 'fixed'))
    m = metrics(port, cal)
    r = daily_returns(port, cal).values.astype(np.float32)
    return port, m, r

def run_search(name, d0, d1, n_trials, batch=8):
    """Optuna TPE over the IS window [d0, d1]. Logs every trial. Cached by a hash of everything that affects it."""
    cal = cal_between(d0, d1)
    key = hashlib.md5(json.dumps([name, d0, d1, n_trials, SYMBOLS, SEED, SLIP_CENTS, HALF_SPREAD_CENTS, RISK_PER_TRADE,
                                  MAX_POS_LEVERAGE, MAX_GROSS_LEVERAGE, DAILY_LOSS_LIMIT_R, MIN_TRADES_PER_IS_YEAR, MIN_COVERAGE,
                                  TIER_DOLLAR_VOL, TREND_TFS, TREND_LENS, ATR_N, FLAT_MIN, CACHE_VERSION, 'space-v5']).encode()).hexdigest()[:10]
    log_path, mat_path = DIRS['trials'] / f'{name}_{key}_log.parquet', DIRS['trials'] / f'{name}_{key}_daily.npy'
    if log_path.exists() and mat_path.exists() and not REBUILD_CACHE:
        return pd.read_parquet(log_path), np.load(mat_path), cal
    min_trades = MIN_TRADES_PER_IS_YEAR * len(cal) / 252
    sampler = optuna.samplers.TPESampler(seed=SEED, multivariate=True, constant_liar=True, n_startup_trials=min(30, n_trials // 3))
    study = optuna.create_study(direction='maximize', sampler=sampler, study_name=name)
    rows, mats = [], []
    t_start = time.time()
    while len(rows) < n_trials:
        trials = [study.ask() for _ in range(min(batch, n_trials - len(rows)))]
        params = [suggest_params(tr) for tr in trials]
        t0 = time.time()
        results = evaluate_params(params, d0, d1)
        for tr, p, res in zip(trials, params, results):
            row = {'split': name, 'trial': tr.number, 'params': json.dumps(p), **{f'p_{k}': v for k, v in p.items()}}
            if isinstance(res, Exception):
                row.update(status='error', error=repr(res)[:300], objective=-3.0)
                r = np.zeros(len(cal), np.float32)
            else:
                port, m, r = score_trades(res, cal, p)
                row.update({f'is_{k}': v for k, v in m.items()})
                if m['trades'] < min_trades:
                    row.update(status='too_few_trades', objective=-3.0)
                elif not np.isfinite(m['sharpe']):
                    row.update(status='degenerate', objective=-3.0)
                else:
                    row.update(status='ok', objective=m['sharpe'])
            study.tell(tr, row['objective'])
            row['elapsed_s'] = (time.time() - t0) / len(trials)
            rows.append(row)
            mats.append(r)
        best = max(r_['objective'] for r_ in rows)
        print(f'  {name}: {len(rows):4d}/{n_trials} trials  best IS Sharpe {best:6.2f}  ({time.time() - t_start:6.0f}s)', end='\r')
    print()
    log = pd.DataFrame(rows)
    mat = np.vstack(mats).astype(np.float32)
    log.to_parquet(log_path, index=False)
    np.save(mat_path, mat)
    return log, mat, cal

In [15]:
SEARCH = {}
for name in SPLITS:
    d0, d1, _, _ = SPLIT_DEF[name]
    SEARCH[name] = run_search(name, d0, d1, N_TRIALS['split'])
    mem(f'search {name}')
for name, (d0, d1, _, _) in WF_FOLDS.items():
    SEARCH[name] = run_search(name, d0, d1, N_TRIALS['wf'])
    mem(f'search {name}')

TRIAL_LOG = pd.concat([v[0] for v in SEARCH.values()], ignore_index=True)
TRIAL_LOG.to_parquet(DIRS['results'] / 'trial_log_all.parquet', index=False)
summary = TRIAL_LOG.groupby('split').agg(trials=('trial', 'size'), ok=('status', lambda s: (s == 'ok').sum()),
                                         too_few=('status', lambda s: (s == 'too_few_trades').sum()),
                                         errors=('status', lambda s: (s == 'error').sum()),
                                         best_is_sharpe=('objective', 'max'),
                                         median_ok_sharpe=('is_sharpe', lambda s: s[TRIAL_LOG.loc[s.index, 'status'] == 'ok'].median()))
print(f'TOTAL TRIALS LOGGED: {len(TRIAL_LOG)}')
summary.round(3)

[mem] search IS1y                                rss=   616.3 MB   peak=   616.3 MB
[mem] search IS2y                                rss=   616.6 MB   peak=   616.6 MB
[mem] search IS3y                                rss=   616.7 MB   peak=   616.7 MB
[mem] search IS4y                                rss=   616.8 MB   peak=   616.8 MB
[mem] search IS5y                                rss=   617.1 MB   peak=   617.1 MB
[mem] search HALF                                rss=   617.2 MB   peak=   617.2 MB
[mem] search WF2008                              rss=   617.5 MB   peak=   617.5 MB
[mem] search WF2011                              rss=   617.5 MB   peak=   617.5 MB
[mem] search WF2014                              rss=   617.5 MB   peak=   617.5 MB
[mem] search WF2017                              rss=   617.7 MB   peak=   617.7 MB
[mem] search WF2020                              rss=   617.8 MB   peak=   617.8 MB
[mem] search WF2023                              rss=   617.8 MB   peak=   6

,trials,ok,too_few,errors,best_is_sharpe,median_ok_sharpe
split,,,,,,
HALF,250,190,60,0,0.288,-0.365
IS1y,250,198,52,0,2.111,0.315
IS2y,250,161,89,0,0.837,-0.790
IS3y,250,194,56,0,1.138,0.071
IS4y,250,180,70,0,0.365,-0.382
IS5y,250,167,83,0,0.617,-0.275
WF2008,100,60,40,0,0.228,-0.918
WF2011,100,63,37,0,0.295,-0.601
WF2014,100,64,36,0,0.040,-0.768


In [16]:
def best_params(name):
    log = SEARCH[name][0]
    ok = log[log.status == 'ok']
    if ok.empty:
        return None, None
    row = ok.loc[ok.objective.idxmax()]
    return json.loads(row['params']), row

BEST = {name: best_params(name) for name in SEARCH}
_missing = [k for k, v in BEST.items() if v[0] is None]
if _missing:
    print('NO valid trial (all failed) for:', _missing)
_bp = pd.DataFrame({k: v[0] for k, v in BEST.items() if v[0] is not None}).T
if len(_bp):
    display(_bp[['trend_tf', 'trend_len', 'side', 'z', 'rr_min', 'rr_max', 'swing_mode', 'swing_lb', 'sd_min', 'sd_max',
                 'm_reclaim', 'confirm', 'clv_min', 'stop_buf', 'max_cost_R', 'target_mode', 'R_mult', 'time_stop', 'max_trades_day']])

fig = make_subplots(rows=1, cols=len(SPLITS), subplot_titles=SPLITS, shared_yaxes=True)
for j, name in enumerate(SPLITS, 1):
    lg = SEARCH[name][0]
    fig.add_trace(go.Scatter(x=lg.trial, y=lg.objective, mode='markers', name=name, showlegend=False,
                             marker=dict(size=4, color=np.where(lg.status == 'ok', '#2a6fdb', '#bbbbbb')),
                             hovertemplate='trial %{x}<br>IS Sharpe %{y:.2f}<extra>' + name + '</extra>'), 1, j)
    fig.update_xaxes(title_text='trial #', row=1, col=j)
fig.update_yaxes(title_text='IS Sharpe (annualized; −3 = failed)', row=1, col=1)
fig.update_layout(title='Search log: IS objective per trial (grey = failed/too few trades)', height=380)
fig.show()

,trend_tf,trend_len,side,z,rr_min,rr_max,swing_mode,swing_lb,sd_min,sd_max,m_reclaim,confirm,clv_min,stop_buf,max_cost_R,target_mode,R_mult,time_stop,max_trades_day
IS1y,60,50,0,3.75,0.4,0.9,pivot,60,0.1,2.5,6,none,0.7,0.2,0.3,liq,2.5,20,5
IS2y,60,10,0,3.75,0.4,0.75,pivot,45,0.1,1.75,1,none,0.5,0.05,0.3,liq,3.25,40,5
IS3y,30,50,2,4.75,0.4,0.9,rolling,45,0.25,1.5,0,recovery,0.4,0.25,0.5,liq,3.0,45,5
IS4y,15,50,2,5.0,0.4,1.0,rolling,30,0.35,2.25,3,dry,0.7,0.25,0.5,liq,2.0,95,5
IS5y,5,50,2,3.75,0.15,1.0,pivot,55,0.0,1.5,8,dry,0.3,0.45,0.3,R,1.5,15,1
HALF,30,50,2,4.0,0.15,1.0,pivot,45,0.4,2.25,10,dry,0.2,0.45,0.3,R,3.25,20,5
WF2008,15,50,2,4.25,0.3,0.9,pivot,60,0.05,2.25,7,climax,0.7,0.45,0.3,R,1.25,15,1
WF2011,60,50,2,4.75,0.35,0.85,rolling,55,0.2,0.5,2,recovery,0.7,0.45,0.5,liq,2.25,35,3
WF2014,30,50,2,3.75,0.2,0.9,pivot,55,0.05,0.75,4,either,0.5,0.4,0.2,R,2.5,15,5
WF2017,30,50,2,4.75,0.4,0.65,rolling,15,0.3,0.5,5,recovery,0.6,0.35,0.3,liq,1.75,55,5


## 8 ML meta-labeling

**Role.** The ML layer does not predict price. It is a *meta-label filter* (López de Prado, 2018): for each rules-based signal it estimates the probability that the trade ends profitable after costs, and low-probability signals are skipped.

**Labels: triple barrier, aligned with the strategy.** The horizontal barriers are the strategy's own stop and target. The vertical barrier is the time stop or 15:55 flat, whichever comes first. The engine already resolves which barrier is touched first (stop first on ties), so the label is `y = 1 if net R at 1× costs > 0`. Time-stop exits are labelled by the sign of their net return.

**Features (all known at the signal-bar close).** Sweep structure: depth, reaction depth, impulse size, bars to reclaim, distance to opposing liquidity. Volume/order-flow PROXIES: sweep relvol, reclaim/sweep volume ratio, effort/result, delta proxy, reaction/impulse volume ratio, CLV, wick ratios. Volatility: ATR % of price, 30-bar realized vol. Context: trend slope, daily-trend agreement, gap, VWAP distance, minute of day, weekday, symbol, volume-source era. Risk: risk and cost in R measured from the signal close, not the next open. Regime: **prior-day** VIX close and VIX percentile.

**Validation.** Purged, embargoed K-fold **inside the IS window only**. Folds are contiguous blocks of trading *dates*, so all symbols' trades on a date share a fold. Every training date within `ML_EMBARGO_DAYS` of a test block is dropped on both sides. Trades never span overnight, so this removes every overlapping label horizon plus a serial-correlation buffer. Out-of-sample predictions average the K fold models, so OOS probabilities are on the same scale as the out-of-fold IS probabilities that set the threshold. The probability threshold is the quantile of OOF probabilities, chosen from {keep 100%, 80%, 60%, 50%, 40%}, that maximizes OOF mean net R. The model is chosen by the same OOF criterion among logistic regression (baseline), random forest and XGBoost. Everything is fit on IS trades only.

In [17]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score
from sklearn.inspection import permutation_importance
import xgboost as xgb
import sklearn
from sklearn.base import clone
RUN_INFO.update({'sklearn': sklearn.__version__, 'xgboost': xgb.__version__})

ML_FEATURES = INFO_ML + ['f_relvol_tod', 'f_clv', 'f_delta_proxy', 'f_effort_result', 'f_src', 'f_relvol_bar', 'f_range_atr',
               'f_body_atr', 'f_lower_wick', 'f_upper_wick', 'f_vwap_dist', 'f_rv30', 'f_gap_datr', 'f_vix_prev',
               'f_vix_pct_prev', 'f_trend_slope', 'f_dtrend_agree', 'f_atr_pct', 'f_minute', 'f_dow', 'f_risk_atr_sig',
               'f_cost_R_sig', 'f_tgt_R_sig', 'f_sym', 'dir']
KEEP_FRACS = (1.0, 0.8, 0.6, 0.5, 0.4)
MIN_ML_TRADES = 150

def make_models(seed=SEED):
    return {
        'logreg': make_pipeline(SimpleImputer(strategy='median'), StandardScaler(), LogisticRegression(C=0.1, max_iter=2000)),
        'rf': make_pipeline(SimpleImputer(strategy='median'),
                            RandomForestClassifier(n_estimators=300, max_depth=6, min_samples_leaf=40, max_features='sqrt',
                                                   n_jobs=4, random_state=seed)),
        'xgb': xgb.XGBClassifier(n_estimators=300, max_depth=3, learning_rate=0.04, subsample=0.8, colsample_bytree=0.7,
                                 min_child_weight=20, reg_lambda=5.0, n_jobs=4, random_state=seed, verbosity=0,
                                 eval_metric='logloss'),
    }

def purged_folds(dates, k=ML_CV_FOLDS, embargo=ML_EMBARGO_DAYS):
    """Contiguous date-block folds; drop training dates within `embargo` trading days of the test block (both sides)."""
    ud = np.unique(dates)
    blocks = np.array_split(np.arange(len(ud)), k)
    pos = {d: i for i, d in enumerate(ud)}
    di = np.array([pos[d] for d in dates])
    for b in blocks:
        lo, hi = b[0], b[-1]
        test = (di >= lo) & (di <= hi)
        train = (di < lo - embargo) | (di > hi + embargo)
        yield np.nonzero(train)[0], np.nonzero(test)[0]

def ml_frame(trades):
    X = trades[ML_FEATURES].astype(np.float32).replace([np.inf, -np.inf], np.nan)
    y = ((trades['R_gross'] - trades['cost_R']) > 0).astype(np.int8).values
    return X, y

def fit_meta(trades_is, seed=SEED, labels=None):
    """Purged CV on IS trades. Returns dict with chosen model name, threshold, fold models and OOF diagnostics."""
    X, y = ml_frame(trades_is)
    if labels is not None:
        y = labels
    Rnet = (trades_is['R_gross'] - trades_is['cost_R']).values
    dates = trades_is['date'].values
    folds = list(purged_folds(dates))
    report, fitted = {}, {}
    for name, proto in make_models(seed).items():
        oof = np.full(len(y), np.nan)
        models = []
        for tr, te in folds:
            if len(np.unique(y[tr])) < 2 or len(te) == 0:
                continue
            m = clone(proto)
            m.fit(X.iloc[tr], y[tr])
            oof[te] = m.predict_proba(X.iloc[te])[:, 1]
            models.append(m)
        ok = ~np.isnan(oof)
        best = (-np.inf, 1.0, -np.inf)
        for f in KEEP_FRACS:
            thr = np.quantile(oof[ok], 1 - f) if f < 1 else -np.inf
            keep = ok & (oof >= thr)
            if keep.sum() >= 30 and Rnet[keep].mean() > best[0]:
                best = (Rnet[keep].mean(), f, thr)
        auc = roc_auc_score(y[ok], oof[ok]) if len(np.unique(y[ok])) == 2 else np.nan
        report[name] = {'oof_auc': auc, 'oof_exp_all': Rnet[ok].mean(), 'oof_exp_kept': best[0], 'keep_frac': best[1],
                        'threshold': best[2], 'n_oof': int(ok.sum())}
        fitted[name] = {'models': models, 'oof': oof, 'threshold': best[2]}
    chosen = max(report, key=lambda k: (report[k]['oof_exp_kept'], -list(report).index(k)))
    return {'chosen': chosen, 'report': pd.DataFrame(report).T, 'fitted': fitted, 'X_cols': list(X.columns), 'folds': folds}

def predict_meta(meta, trades, model_name=None):
    model_name = model_name or meta['chosen']
    X, _ = ml_frame(trades)
    ms = meta['fitted'][model_name]['models']
    return np.mean([m.predict_proba(X)[:, 1] for m in ms], axis=0) if ms and len(X) else np.full(len(X), np.nan)

def ml_filter(trades, meta, probs=None):
    probs = predict_meta(meta, trades) if probs is None else probs
    thr = meta['fitted'][meta['chosen']]['threshold']
    return trades[probs >= thr].copy(), probs

### 8.1 IS trade logs for every selected configuration, then meta-label fitting
The IS trades of each split's best rules configuration are regenerated with the full ML feature set. The meta-labeler is then fit on those trades only. IS "ML-filtered" performance below uses **out-of-fold** probabilities, so even the IS ML numbers are not fit-on-self.

In [18]:
EVAL = {}
for name in SEARCH:
    p = BEST[name][0]
    if p is None:
        print(f'{name}: no valid configuration, skipped')
        continue
    is0, is1, oos0, oos1 = (SPLIT_DEF[name] if name in SPLIT_DEF else WF_FOLDS[name])
    t_is = evaluate_params([p], is0, is1, with_ml=True)[0]
    EVAL[name] = {'p': p, 'is_cal': cal_between(is0, is1), 'oos_cal': cal_between(oos0, oos1), 'is_raw': t_is,
                  'is_port': {}, 'oos_port': {}}
    if isinstance(t_is, Exception) or len(t_is) < MIN_ML_TRADES:
        EVAL[name]['meta'] = None
        print(f'{name}: {0 if isinstance(t_is, Exception) else len(t_is)} IS trades < {MIN_ML_TRADES} -> ML layer not fit')
        continue
    meta = fit_meta(t_is)
    EVAL[name]['meta'] = meta
    oof = meta['fitted'][meta['chosen']]['oof']
    EVAL[name]['is_ml_mask'] = (~np.isnan(oof)) & (oof >= meta['fitted'][meta['chosen']]['threshold'])
    print(f"{name}: {len(t_is)} IS trades | chosen={meta['chosen']} keep={meta['report'].loc[meta['chosen'], 'keep_frac']:.0%} "
          f"| OOF AUC " + ', '.join(f"{k}={v:.3f}" for k, v in meta['report'].oof_auc.items()))
    mem(f'ML fit {name}')

ML_REPORT = pd.concat({k: v['meta']['report'] for k, v in EVAL.items() if v.get('meta')}, names=['split', 'model'])
ML_REPORT.round(4)

IS1y: 38 IS trades < 150 -> ML layer not fit


IS2y: 67 IS trades < 150 -> ML layer not fit


IS3y: 175 IS trades | chosen=logreg keep=100% | OOF AUC logreg=0.517, rf=0.461, xgb=0.423
[mem] ML fit IS3y                                rss=  1007.1 MB   peak=  1015.6 MB


IS4y: 128 IS trades < 150 -> ML layer not fit


IS5y: 129 IS trades < 150 -> ML layer not fit


HALF: 488 IS trades | chosen=logreg keep=80% | OOF AUC logreg=0.477, rf=0.466, xgb=0.478
[mem] ML fit HALF                                rss=  1449.0 MB   peak=  1698.5 MB


... (3 outputs left out)

WF2017: 537 IS trades | chosen=xgb keep=40% | OOF AUC logreg=0.495, rf=0.507, xgb=0.514
[mem] ML fit WF2017                              rss=   917.5 MB   peak=  1698.5 MB


WF2020: 469 IS trades | chosen=xgb keep=40% | OOF AUC logreg=0.482, rf=0.469, xgb=0.516
[mem] ML fit WF2020                              rss=   951.9 MB   peak=  1698.5 MB


WF2023: 585 IS trades | chosen=logreg keep=40% | OOF AUC logreg=0.550, rf=0.526, xgb=0.503
[mem] ML fit WF2023                              rss=   939.8 MB   peak=  1698.5 MB


oof_auc  oof_exp_all  oof_exp_kept  keep_frac  threshold  n_oof
split  model                                                                  
IS3y   logreg   0.5169       0.2419        0.2419        1.0       -inf  175.0
       rf       0.4607       0.2419        0.2419        1.0       -inf  175.0
       xgb      0.4231       0.2419        0.2419        1.0       -inf  175.0
HALF   logreg   0.4768       0.0587        0.0694        0.8     0.3352  488.0
       rf       0.4664       0.0587        0.0587        1.0       -inf  488.0
       xgb      0.4784       0.0587        0.0587        1.0       -inf  488.0
WF2008 logreg   0.5510       0.0006        0.1095        0.5     0.5115  170.0
       rf       0.5317       0.0006        0.0501        0.8     0.5211  170.0
       xgb      0.4796       0.0006        0.0006        1.0       -inf  170.0
WF2011 logreg   0.4572       0.0345        0.0345        1.0       -inf  344.0
       rf       0.5074       0.0345        0.1256        0.6     0.3334  344.0
       xgb      0.5274       0.0345        0.1086        0.8     0.2286  344.0
WF2014 logreg   0.5147      -0.0299        0.0146        0.4     0.3939  445.0
       rf       0.5536      -0.0299        0.1174        0.6     0.3826  445.0
       xgb      0.5222      -0.0299        0.0956        0.4     0.4146  445.0
WF2017 logreg   0.4951       0.0920        0.1473        0.8     0.2571  537.0
       rf       0.5067       0.0920        0.1695        0.6     0.3263  537.0
       xgb      0.5137       0.0920        0.2670        0.4     0.3554  537.0
WF2020 logreg   0.4821       0.0882        0.1313        0.6     0.2436  469.0
       rf       0.4693       0.0882        0.0882        1.0       -inf  469.0
       xgb      0.5160       0.0882        0.2319        0.4     0.3176  469.0
WF2023 logreg   0.5498       0.0429        0.2265        0.4     0.4041  585.0
       rf       0.5263       0.0429        0.0792        0.4     0.3958  585.0
       xgb      0.5034       0.0429        0.0844        0.5     0.3626  585.0

### 8.2 Feature importance (primary split)
SHAP (TreeExplainer) for XGBoost on out-of-fold rows, permutation importance (ΔAUC on each fold's held-out block) for the random forest, and absolute standardized coefficients for logistic regression. If importance is spread thinly across proxies, with no feature standing out, the meta-labeler has little real structure to exploit.

In [19]:
import shap
RUN_INFO['shap'] = shap.__version__
IMPORTANCE = None
if PRIMARY_SPLIT in EVAL and EVAL[PRIMARY_SPLIT].get('meta'):
    meta = EVAL[PRIMARY_SPLIT]['meta']
    X, y = ml_frame(EVAL[PRIMARY_SPLIT]['is_raw'])
    imp = {}
    sv = []
    for m, (tr, te) in zip(meta['fitted']['xgb']['models'], meta['folds']):
        sv.append(np.abs(shap.TreeExplainer(m).shap_values(X.iloc[te])).mean(axis=0))
    imp['xgb_mean_abs_shap'] = np.mean(sv, axis=0)
    pi = []
    for m, (tr, te) in zip(meta['fitted']['rf']['models'], meta['folds']):
        if len(np.unique(y[te])) == 2:
            pi.append(permutation_importance(m, X.iloc[te], y[te], scoring='roc_auc', n_repeats=5, random_state=SEED, n_jobs=1).importances_mean)
    imp['rf_perm_dAUC'] = np.mean(pi, axis=0) if pi else np.nan
    imp['logreg_abs_coef'] = np.mean([np.abs(m[-1].coef_[0]) for m in meta['fitted']['logreg']['models']], axis=0)
    IMPORTANCE = pd.DataFrame(imp, index=X.columns).sort_values('xgb_mean_abs_shap', ascending=False)
    top = IMPORTANCE.head(20)[::-1]
    fig = make_subplots(rows=1, cols=3, subplot_titles=['XGBoost mean |SHAP| (log-odds)', 'RF permutation ΔAUC', 'LogReg |std. coef|'], shared_yaxes=True)
    for j, col in enumerate(['xgb_mean_abs_shap', 'rf_perm_dAUC', 'logreg_abs_coef'], 1):
        fig.add_trace(go.Bar(x=top[col], y=top.index, orientation='h', showlegend=False,
                             hovertemplate='%{y}: %{x:.4f}<extra></extra>'), 1, j)
    fig.update_xaxes(title_text='importance (model-specific units)')
    fig.update_yaxes(title_text='feature (f_ = bar feature at signal close; PROXY = OHLCV order-flow proxy)', row=1, col=1)
    fig.update_layout(title=f'Meta-label feature importance ({PRIMARY_SPLIT} IS, out-of-fold)', height=620)
    fig.show()
    display(IMPORTANCE.head(15).round(4))
mem('ML importance')

,xgb_mean_abs_shap,rf_perm_dAUC,logreg_abs_coef
f_upper_wick,0.2329,0.0205,0.1143
f_minute,0.2136,-0.0119,0.2266
sweep_depth_atr,0.1347,-0.0039,0.0744
delta_proxy_sum,0.1301,0.0162,0.0949
f_vix_pct_prev,0.1208,-0.0028,0.1796
sweep_relvol,0.1152,-0.0043,0.1366
sweep_effort_result,0.1144,-0.0011,0.1403
f_clv,0.1019,0.0009,0.1737
f_risk_atr_sig,0.0997,-0.0038,0.1118
f_rv30,0.0941,-0.0161,0.1025


[mem] ML importance                              rss=   140.6 MB   peak=  1698.5 MB


## 9 IS/OOS evaluation

**OOS discipline.** Each split's single selected candidate (the best IS rules configuration and its IS-fit meta-labeler) is run on that split's OOS window **once**. Nothing below feeds back into §7 or §8. Results are reported at 1× and 2× costs. The ML-filtered variant removes trades from the rules-only trade list. (A skipped trade would in reality free the symbol for another signal; that second-order effect is ignored, which slightly understates ML trade counts.)

In [20]:
def portfolio_set(raw, mask_ml, p):
    out = {}
    for cm in COST_MULTS:
        out[cm] = {'rules': apply_portfolio(raw, cm, p['sizing']) if len(raw) else apply_portfolio(None),
                   'ml': apply_portfolio(raw[mask_ml], cm, p['sizing']) if mask_ml is not None and len(raw) else None}
    return out

rows = []
for name, E in EVAL.items():
    p = E['p']
    is0, is1, oos0, oos1 = (SPLIT_DEF[name] if name in SPLIT_DEF else WF_FOLDS[name])
    t_oos = evaluate_params([p], oos0, oos1, with_ml=True)[0]                 # <- the one OOS touch for this candidate
    E['oos_raw'] = t_oos
    meta = E.get('meta')
    if meta is not None and len(t_oos):
        E['oos_prob'] = predict_meta(meta, t_oos)
        E['oos_ml_mask'] = E['oos_prob'] >= meta['fitted'][meta['chosen']]['threshold']
    else:
        E['oos_ml_mask'] = None
    E['is_port'] = portfolio_set(E['is_raw'], E.get('is_ml_mask'), p)
    E['oos_port'] = portfolio_set(t_oos, E['oos_ml_mask'], p)
    for per, cal, ports in [('IS', E['is_cal'], E['is_port']), ('OOS', E['oos_cal'], E['oos_port'])]:
        for cm in COST_MULTS:
            for var in ['rules', 'ml']:
                if ports[cm][var] is None:
                    continue
                rows.append({'split': name, 'variant': 'rules-only' if var == 'rules' else f"ML-filtered ({meta['chosen']})",
                             'period': per, 'costs': f'{cm:g}x', **metrics(ports[cm][var], cal)})
    t_oos.to_parquet(DIRS['trades'] / f'{name}_oos_raw.parquet', index=False)
    E['is_raw'].to_parquet(DIRS['trades'] / f'{name}_is_raw.parquet', index=False)
    print(f'{name}: OOS {len(t_oos)} raw trades')
    mem(f'OOS eval {name}')

METRICS = pd.DataFrame(rows)
METRICS.to_parquet(DIRS['results'] / 'metrics_all.parquet', index=False)
_show = ['split', 'variant', 'period', 'costs', 'trades', 'win_rate', 'expectancy_R', 'profit_factor', 'sharpe', 'sortino',
         'max_dd', 'avg_R_gross', 'trades_per_day']
display(METRICS[METRICS.split.isin(SPLITS)][_show].round(3))

IS1y: OOS 5277 raw trades
[mem] OOS eval IS1y                              rss=   917.5 MB   peak=  1698.5 MB


IS2y: OOS 6592 raw trades
[mem] OOS eval IS2y                              rss=  1151.2 MB   peak=  1698.5 MB


IS3y: OOS 4808 raw trades
[mem] OOS eval IS3y                              rss=  1100.4 MB   peak=  1698.5 MB


IS4y: OOS 1875 raw trades
[mem] OOS eval IS4y                              rss=  1128.4 MB   peak=  1698.5 MB


IS5y: OOS 1653 raw trades
[mem] OOS eval IS5y                              rss=  1261.8 MB   peak=  1698.5 MB


HALF: OOS 729 raw trades
[mem] OOS eval HALF                              rss=  1077.0 MB   peak=  1698.5 MB


... (3 outputs left out)

WF2017: OOS 213 raw trades
[mem] OOS eval WF2017                            rss=   712.1 MB   peak=  1698.5 MB


WF2020: OOS 477 raw trades
[mem] OOS eval WF2020                            rss=   519.4 MB   peak=  1698.5 MB


WF2023: OOS 114 raw trades
[mem] OOS eval WF2023                            rss=   712.8 MB   peak=  1698.5 MB


,split,variant,period,costs,trades,win_rate,expectancy_R,profit_factor,sharpe,sortino,max_dd,avg_R_gross,trades_per_day
0,IS1y,rules-only,IS,1x,38,0.632,0.566,2.383,2.111,5.999,-0.017,0.813,0.150
1,IS1y,rules-only,IS,2x,38,0.526,0.318,1.632,1.292,2.694,-0.023,0.813,0.150
2,IS1y,rules-only,OOS,1x,4776,0.317,-0.158,0.803,-1.170,-1.707,-0.878,0.019,0.836
3,IS1y,rules-only,OOS,2x,4677,0.306,-0.329,0.643,-2.445,-3.184,-0.986,0.023,0.818
4,IS2y,rules-only,IS,1x,67,0.537,0.272,1.480,0.837,1.475,-0.026,0.510,0.132
5,IS2y,rules-only,IS,2x,67,0.537,0.035,1.051,-0.054,-0.077,-0.054,0.510,0.132
6,IS2y,rules-only,OOS,1x,5626,0.329,-0.115,0.857,-0.851,-1.332,-0.823,0.061,1.030
7,IS2y,rules-only,OOS,2x,5482,0.326,-0.288,0.688,-2.233,-3.061,-0.986,0.062,1.003
8,IS3y,rules-only,IS,1x,167,0.467,0.257,1.399,1.138,2.094,-0.035,0.612,0.220
9,IS3y,ML-filtered (logreg),IS,1x,167,0.467,0.257,1.399,1.138,2.094,-0.035,0.612,0.220


### 9.1 Anchored walk-forward (cross-check)
Each fold re-runs the search on an IS window anchored at the data start, then trades the next three years. The stitched OOS segments form one continuous track record with no overlap.

In [21]:
WF_OOS = {}
for var in ['rules', 'ml']:
    for cm in COST_MULTS:
        parts = [E['oos_port'][cm][var] for k, E in EVAL.items() if k in WF_FOLDS and E['oos_port'][cm][var] is not None]
        WF_OOS[(var, cm)] = pd.concat(parts, ignore_index=True) if parts else None
wf_cal = np.concatenate([EVAL[k]['oos_cal'] for k in WF_FOLDS if k in EVAL]) if any(k in EVAL for k in WF_FOLDS) else np.array([])
wf_rows = [{'split': 'WF stitched', 'variant': var, 'period': 'OOS', 'costs': f'{cm:g}x', **metrics(t, wf_cal)}
           for (var, cm), t in WF_OOS.items() if t is not None]
WF_METRICS = pd.DataFrame(wf_rows)
display(pd.concat([METRICS[METRICS.split.isin(WF_FOLDS) & (METRICS.period == 'OOS')], WF_METRICS])[_show].round(3))

# How sure are we about the stitched walk-forward numbers? Trade-level bootstrap CI of expectancy, and a fold-level
# sign test of "ML-filtered beats rules-only" (evaluation of existing results only; nothing is re-selected).
wf_ci = []
for (var, cm), t in WF_OOS.items():
    if t is not None and len(t):
        bt = np.random.default_rng(SEED).choice(t['R'].values, size=(N_BOOT, len(t)), replace=True).mean(axis=1)
        wf_ci.append({'variant': var, 'costs': f'{cm:g}x', 'trades': len(t), 'expectancy_R': t['R'].mean(),
                      'ci_lo': np.quantile(bt, 0.025), 'ci_hi': np.quantile(bt, 0.975), 'P(exp<=0)': float((bt <= 0).mean())})
WF_CI = pd.DataFrame(wf_ci)
display(WF_CI.round(3))
fold_cmp = []
for k in WF_FOLDS:
    if k in EVAL and EVAL[k]['oos_port'][1.0]['ml'] is not None:
        r_, m_ = EVAL[k]['oos_port'][1.0]['rules'], EVAL[k]['oos_port'][1.0]['ml']
        fold_cmp.append({'fold': k, 'rules_exp': r_['R'].mean(), 'ml_exp': m_['R'].mean() if len(m_) else np.nan,
                         'ml_minus_rules': (m_['R'].mean() if len(m_) else np.nan) - r_['R'].mean()})
WF_FOLD_CMP = pd.DataFrame(fold_cmp)
WF_ML_WINS = int((WF_FOLD_CMP.ml_minus_rules > 0).sum()) if len(WF_FOLD_CMP) else 0
WF_ML_SIGN_P = stats.binomtest(WF_ML_WINS, len(WF_FOLD_CMP), 0.5, alternative='greater').pvalue if len(WF_FOLD_CMP) else np.nan
display(WF_FOLD_CMP.round(3))
print(f'ML-filtered beat rules-only in {WF_ML_WINS}/{len(WF_FOLD_CMP)} walk-forward folds (one-sided sign test p = {WF_ML_SIGN_P:.3f})')

,split,variant,period,costs,trades,win_rate,expectancy_R,profit_factor,sharpe,sortino,max_dd,avg_R_gross,trades_per_day
36,WF2008,rules-only,OOS,1x,313,0.435,-0.209,0.647,-2.010,-2.409,-0.307,-0.035,0.413
37,WF2008,ML-filtered (logreg),OOS,1x,91,0.429,-0.241,0.586,-1.263,-1.541,-0.110,-0.062,0.120
38,WF2008,rules-only,OOS,2x,313,0.406,-0.383,0.449,-3.360,-3.644,-0.462,-0.035,0.413
39,WF2008,ML-filtered (logreg),OOS,2x,91,0.374,-0.421,0.398,-1.996,-2.223,-0.176,-0.062,0.120
44,WF2011,rules-only,OOS,1x,164,0.451,0.052,1.074,0.428,0.751,-0.045,0.367,0.218
45,WF2011,ML-filtered (rf),OOS,1x,120,0.450,0.063,1.088,0.392,0.732,-0.035,0.363,0.159
46,WF2011,rules-only,OOS,2x,162,0.407,-0.250,0.720,-0.640,-0.961,-0.082,0.384,0.215
47,WF2011,ML-filtered (rf),OOS,2x,118,0.415,-0.219,0.754,-0.421,-0.683,-0.062,0.386,0.156
52,WF2014,rules-only,OOS,1x,95,0.453,0.082,1.131,0.429,0.742,-0.026,0.240,0.126
53,WF2014,ML-filtered (rf),OOS,1x,58,0.500,0.232,1.398,0.832,1.560,-0.032,0.391,0.077


,variant,costs,trades,expectancy_R,ci_lo,ci_hi,P(exp<=0)
0,rules,1x,1319,-0.067,-0.160,0.025,0.920
1,rules,2x,1314,-0.244,-0.336,-0.148,1.000
2,ml,1x,519,0.105,-0.044,0.259,0.100
3,ml,2x,517,-0.080,-0.239,0.083,0.846


,fold,rules_exp,ml_exp,ml_minus_rules
0,WF2008,-0.209,-0.241,-0.032
1,WF2011,0.052,0.063,0.011
2,WF2014,0.082,0.232,0.150
3,WF2017,-0.156,-0.025,0.131
4,WF2020,0.020,0.452,0.432
5,WF2023,-0.162,-0.071,0.091


ML-filtered beat rules-only in 5/6 walk-forward folds (one-sided sign test p = 0.109)


### 9.2 Equity and drawdown curves
Compounded equity at 0.5% risk per trade, leverage-capped. IS segments are shaded grey and OOS segments are white. Dashed lines are the ML-filtered variant.

In [22]:
def equity_series(t, cal):
    r = daily_returns(t, cal)
    return pd.Series((1 + r.values).cumprod(), index=pd.to_datetime(pd.Series(cal).astype(str)).values)

names = [s for s in SPLITS if s in EVAL]
fig = make_subplots(rows=(len(names) + 1) // 2, cols=2, subplot_titles=names, vertical_spacing=0.08)
for k, name in enumerate(names):
    E = EVAL[name]; row, col = k // 2 + 1, k % 2 + 1
    full_cal = np.concatenate([E['is_cal'], E['oos_cal']])
    for var, dash in [('rules', 'solid'), ('ml', 'dash')]:
        if E['oos_port'][1.0][var] is None:
            continue
        t = pd.concat([E['is_port'][1.0][var], E['oos_port'][1.0][var]], ignore_index=True)
        eq = equity_series(t, full_cal)
        fig.add_trace(go.Scatter(x=eq.index, y=eq.values, mode='lines', line=dict(dash=dash, width=1.4),
                                 name=f'{var}', legendgroup=var, showlegend=(k == 0),
                                 hovertemplate='%{x|%Y-%m-%d}<br>equity %{y:.3f}<extra>' + f'{name} {var}</extra>'), row, col)
    fig.add_vrect(x0=pd.Timestamp(str(E['is_cal'][0])), x1=pd.Timestamp(str(E['is_cal'][-1])), fillcolor='grey', opacity=0.15,
                  line_width=0, row=row, col=col)
    fig.update_yaxes(title_text='equity (start = 1)', row=row, col=col)
    fig.update_xaxes(title_text='date', row=row, col=col)
fig.update_layout(title='Equity curves by split, 1x costs (grey = in-sample, white = out-of-sample)', height=330 * ((len(names) + 1) // 2))
fig.show()

if WF_OOS.get(('rules', 1.0)) is not None:
    fig = make_subplots(rows=2, cols=1, shared_xaxes=True, row_heights=[0.65, 0.35],
                        subplot_titles=['Anchored walk-forward: stitched OOS equity', 'Drawdown'])
    for (var, cm), t in WF_OOS.items():
        if t is None:
            continue
        eq = equity_series(t, wf_cal)
        dd = eq / eq.cummax() - 1
        lab = f'{var} {cm:g}x costs'
        fig.add_trace(go.Scatter(x=eq.index, y=eq.values, name=lab, mode='lines',
                                 hovertemplate='%{x|%Y-%m-%d}<br>equity %{y:.3f}<extra>' + lab + '</extra>'), 1, 1)
        fig.add_trace(go.Scatter(x=dd.index, y=dd.values * 100, name=lab + ' DD', mode='lines', showlegend=False,
                                 hovertemplate='%{x|%Y-%m-%d}<br>drawdown %{y:.2f}%<extra>' + lab + '</extra>'), 2, 1)
    for k in WF_FOLDS:
        if k in EVAL:
            fig.add_vline(x=pd.Timestamp(str(EVAL[k]['oos_cal'][0])), line_dash='dot', line_color='grey')
    fig.update_yaxes(title_text='equity (start = 1)', row=1, col=1)
    fig.update_yaxes(title_text='drawdown (%)', row=2, col=1)
    fig.update_xaxes(title_text='date (dotted = re-optimization points)', row=2, col=1)
    fig.update_layout(title='Anchored walk-forward cross-check (OOS only)', height=620)
    fig.show()

E = EVAL.get(PRIMARY_SPLIT)
if E is not None:
    fig = go.Figure()
    for (var, cm) in [('rules', 1.0), ('rules', 2.0), ('ml', 1.0), ('ml', 2.0)]:
        t = E['oos_port'][cm][var]
        if t is None:
            continue
        eq = equity_series(t, E['oos_cal'])
        fig.add_trace(go.Scatter(x=eq.index, y=(eq / eq.cummax() - 1).values * 100, mode='lines', name=f'{var} {cm:g}x',
                                 hovertemplate='%{x|%Y-%m-%d}<br>drawdown %{y:.2f}%<extra></extra>'))
    fig.update_layout(title=f'{PRIMARY_SPLIT} split: OOS drawdown by variant and cost level', xaxis_title='date',
                      yaxis_title='drawdown from peak (%)', height=380)
    fig.show()

### 9.3 Annotated example trades (primary split, OOS)
Each chart shows one OOS trade: the swept level (dashed), the sweep extreme, the reclaim/signal bar, the entry at the next open, the stop and target, and the exit. Candles are 1-minute RTH bars, and prices are in adjusted price units.

In [23]:
def plot_trade(tr, before=45, after=15):
    d = int(tr['date'])
    F = load_feats(tr['sym'], ['date', 'minute', 'o', 'h', 'l', 'c', 'v', 'real'], d, d)
    ts = pd.to_datetime(F['date'].astype(str)) + pd.to_timedelta(F['minute'].astype(int), unit='m')
    m_sig = int(tr['entry_min']) - 1
    i_sig = int(np.nonzero(F.minute.values == m_sig)[0][0])
    i_ent, i_ext = i_sig + 1, int(np.nonzero(F.minute.values == int(tr['exit_min']))[0][0])
    i_swp = i_sig - int(tr['bars_to_reclaim'])
    a, b = max(0, i_swp - before), min(len(F) - 1, i_ext + after)
    W = F.iloc[a:b + 1]; tw = ts.iloc[a:b + 1]
    fig = go.Figure(go.Candlestick(x=tw, open=W.o, high=W.h, low=W.l, close=W.c, name='1m bars',
                                   hovertext=[f'vol {v:,.0f}' for v in W.v]))
    lvl, ext = float(tr['sweep_level']), float(tr['sweep_extreme'])
    fig.add_trace(go.Scatter(x=[tw.iloc[0], ts.iloc[i_sig]], y=[lvl, lvl], mode='lines', line=dict(dash='dash', color='#7f7f7f'),
                             name='swept level', hovertemplate='swept level %{y:.3f}<extra></extra>'))
    fig.add_trace(go.Scatter(x=[ts.iloc[i_swp]], y=[ext], mode='markers+text', text=['sweep'], textposition='bottom center' if tr['dir'] > 0 else 'top center',
                             marker=dict(symbol='x', size=11, color='#d62728'), name='sweep extreme', hovertemplate='sweep extreme %{y:.3f}<extra></extra>'))
    fig.add_trace(go.Scatter(x=[ts.iloc[i_sig]], y=[F.c.iloc[i_sig]], mode='markers+text', text=['reclaim'], textposition='top center',
                             marker=dict(symbol='circle-open', size=12, color='#9467bd'), name='reclaim (signal close)', hovertemplate='reclaim close %{y:.3f}<extra></extra>'))
    fig.add_trace(go.Scatter(x=[ts.iloc[i_ent]], y=[tr['entry']], mode='markers', marker=dict(symbol='triangle-up' if tr['dir'] > 0 else 'triangle-down', size=13, color='#2ca02c'),
                             name='entry (next open)', hovertemplate='entry %{y:.3f}<extra></extra>'))
    for y, nm, colr in [(tr['stop'], 'stop', '#d62728'), (tr['target'], 'target', '#2ca02c')]:
        fig.add_trace(go.Scatter(x=[ts.iloc[i_ent], ts.iloc[i_ext]], y=[y, y], mode='lines', line=dict(color=colr, width=1.5),
                                 name=nm, hovertemplate=nm + ' %{y:.3f}<extra></extra>'))
    fig.add_trace(go.Scatter(x=[ts.iloc[i_ext]], y=[tr['exit']], mode='markers', marker=dict(symbol='square', size=10, color='black'),
                             name=f"exit ({EXIT_REASONS[int(tr['reason'])]})", hovertemplate='exit %{y:.3f}<extra></extra>'))
    fig.update_layout(title=f"{tr['sym']} {d} {'long' if tr['dir'] > 0 else 'short'}: {EXIT_REASONS[int(tr['reason'])]} exit, "
                            f"net {tr['R_gross'] - tr['cost_R']:+.2f}R (gross {tr['R_gross']:+.2f}R)",
                      xaxis_title='time (ET)', yaxis_title='price (adjusted units)', xaxis_rangeslider_visible=False, height=430)
    return fig

if E is not None and len(E['oos_raw']):
    ex = E['oos_raw']
    picks = []
    for reason in [1, 0, 2, 3]:
        cand = ex[ex.reason == reason]
        if len(cand):
            picks.append(cand.sample(1, random_state=SEED).iloc[0])
    for tr in picks[:3]:
        plot_trade(tr).show()
mem('evaluation charts')

[mem] evaluation charts                          rss=   703.1 MB   peak=  1698.5 MB


## 10 Overfitting & integrity checks

Each check ends with a **PASS/FAIL** verdict, recorded in `VERDICTS`. The conclusions and the final audit cell read those verdicts. Unless stated otherwise, the checks use the primary split (`HALF`: first half IS, second half OOS) at 1× costs. The multiple-testing checks (DSR, PBO) run on every split.

In [24]:
VERDICTS = {}

def verdict(name, passed, detail):
    VERDICTS[name] = {'verdict': 'PASS' if passed else 'FAIL', 'detail': detail}
    print(f"[{VERDICTS[name]['verdict']}] {name}: {detail}")

P = EVAL[PRIMARY_SPLIT]
P_PARAMS = P['p']
P_OOS0, P_OOS1 = SPLIT_DEF[PRIMARY_SPLIT][2], SPLIT_DEF[PRIMARY_SPLIT][3]
REAL_RULES = P['oos_port'][1.0]['rules']
REAL_ML = P['oos_port'][1.0]['ml']

def hist_with_marker(dist, real, title, xlab, name='placebo'):
    fig = go.Figure(go.Histogram(x=dist, nbinsx=50, name=name, hovertemplate=xlab + ' %{x:.3f}<br>count %{y}<extra></extra>'))
    fig.add_vline(x=real, line_color='#d62728', line_width=3, annotation_text=f'real = {real:.3f}', annotation_position='top right')
    fig.add_vline(x=np.quantile(dist, 0.95), line_dash='dash', line_color='black', annotation_text='95th pct', annotation_position='top left')
    fig.update_layout(title=title, xaxis_title=xlab, yaxis_title='count of runs', height=360)
    return fig

### 10.1 Placebo: random entries with matched count, direction mix, stop/target geometry and exits
For each ETF, every run draws the same number of entry bars as the real OOS strategy at random from eligible bars: tradable day, real print, inside the strategy's entry window. Each draw takes a direction from the real long/short mix and a stop distance (ATR) and target distance (R) from the real trades' distributions. The same engine, time stop, trade cap and costs apply. The real strategy must beat the 95th percentile of placebo expectancy.

In [25]:
def placebo_runs(real_raw, params, d0, d1, n_runs, mode='random', shifts=None, seed=SEED):
    """mode='random': random entry bars; mode='shift': real signal bars shifted by a common offset per run."""
    rng = np.random.default_rng(seed)
    out = [[] for _ in range(n_runs)]
    cols = ['date', 'minute', 'o', 'h', 'l', 'c', 'real', 'adj', 'cost_rt_c', 'atr', 'tradable']
    for sym in SYMBOLS:
        rs = real_raw[real_raw.sym == sym]
        if len(rs) == 0:
            continue
        F = load_feats(sym, cols, d0, d1)
        o, h, l, c = (F[k].values.astype(np.float64) for k in ['o', 'h', 'l', 'c'])
        real_, cost0, cps = F.real.values.astype(np.int8), np.zeros(len(F)), round_trip_cost_ps(sym, F)
        day, minute, atr = F.date.values.astype(np.int64), F.minute.values.astype(np.int64), F.atr.values.astype(np.float64)
        elig = np.nonzero((F.tradable.values == 1) & (F.real.values == 1) & (minute >= OPEN_MIN + params['open_skip'])
                          & (minute <= params['last_entry_min']) & np.isfinite(atr) & (atr > 0))[0]
        elig = elig[elig < len(F) - 1]
        risk_atr = rs['f_risk_atr_sig'].values
        tgt_R = np.full(len(rs), params['R_mult']) if params['target_mode'] == 'R' else rs['f_tgt_R_sig'].values
        dirs = rs['dir'].values
        for k in range(n_runs):
            if mode == 'random':
                idx = np.sort(rng.choice(elig, size=min(len(rs), len(elig)), replace=False))
                j = rng.integers(0, len(rs), size=len(idx))
                d_, ra, tr = rng.choice(dirs, size=len(idx)), risk_atr[j], tgt_R[j]
            else:
                idx = rs['sig_idx'].values + shifts[k]
                ok = (idx > 0) & (idx < len(F) - 1)
                idx, d_, ra, tr = idx[ok], dirs[ok], risk_atr[ok], tgt_R[ok]
                good = np.isin(idx, elig)
                idx, d_, ra, tr = idx[good], d_[good], ra[good], tr[good]
                order = np.argsort(idx); idx, d_, ra, tr = idx[order], d_[order], ra[order], tr[order]
            sig = np.zeros(len(F), np.int8); st = np.full(len(F), np.nan); tg = np.full(len(F), np.nan)
            sig[idx] = d_
            st[idx] = c[idx] - d_ * ra * atr[idx]
            tg[idx] = c[idx] + d_ * tr * ra * atr[idx]
            res = simulate(o, h, l, c, day, minute, sig, st, tg, False, 0.0, 0.0, params['time_stop'],
                           params['max_trades_day'], params['last_entry_min'] + 1, FLAT_MIN, real_, atr, 0.0, cost0, 0.0)
            out[k].append(trades_frame(sym, F, res, cps))
        del F
        gc.collect()
    stats_ = []
    for k in range(n_runs):
        t = pd.concat([x for x in out[k] if len(x)], ignore_index=True) if any(len(x) for x in out[k]) else pd.DataFrame()
        port = apply_portfolio(t, 1.0, params['sizing'])
        m = metrics(port, cal_between(d0, d1))
        stats_.append({'expectancy_R': m['expectancy_R'], 'sharpe': m['sharpe'], 'trades': m['trades']})
    return pd.DataFrame(stats_)

t0 = time.time()
PLACEBO = placebo_runs(P['oos_raw'], P_PARAMS, P_OOS0, P_OOS1, N_PLACEBO, mode='random')
real_exp = metrics(REAL_RULES, P['oos_cal'])['expectancy_R']
p_val = (1 + (PLACEBO.expectancy_R >= real_exp).sum()) / (1 + len(PLACEBO))
print(f'{N_PLACEBO} placebo runs in {time.time() - t0:.0f}s; mean placebo trades {PLACEBO.trades.mean():.0f} vs real {len(REAL_RULES)}; '
      f'placebo expectancy median {PLACEBO.expectancy_R.median():+.3f}R. NOTE: beating the placebo shows the entry timing carries '
      f'information; it does not by itself mean the strategy is profitable after costs.')
hist_with_marker(PLACEBO.expectancy_R, real_exp, f'Placebo (random entries) vs real rules-only OOS expectancy, {PRIMARY_SPLIT}',
                 'OOS expectancy (net R per trade)').show()
verdict('placebo_random_entry', p_val < 0.05, f'real OOS expectancy {real_exp:+.3f}R, placebo 95th pct '
        f'{PLACEBO.expectancy_R.quantile(0.95):+.3f}R, p = {p_val:.3f}')
mem('placebo random')

500 placebo runs in 19s; mean placebo trades 708 vs real 686; placebo expectancy median -0.339R. NOTE: beating the placebo shows the entry timing carries information; it does not by itself mean the strategy is profitable after costs.


[PASS] placebo_random_entry: real OOS expectancy -0.047R, placebo 95th pct -0.226R, p = 0.004
[mem] placebo random                             rss=   750.8 MB   peak=  1698.5 MB


### 10.2 Placebo: time-shifted signals
The real OOS signal bars are shifted by a common random offset of 5–390 bars, either direction, keeping each trade's geometry. If the specific sweep/reclaim bar matters, unshifted signals must beat the shifted ones.

In [26]:
n_shift = max(50, N_PLACEBO // 2)
_rng_sh = np.random.default_rng(SEED + 2)          # own stream: unaffected by RNG use in other cells
_sh = _rng_sh.integers(5, 391, size=n_shift) * _rng_sh.choice([-1, 1], size=n_shift)
SHIFTED = placebo_runs(P['oos_raw'], P_PARAMS, P_OOS0, P_OOS1, n_shift, mode='shift', shifts=_sh)
p_shift = (1 + (SHIFTED.expectancy_R >= real_exp).sum()) / (1 + len(SHIFTED))
print(f'{len(SHIFTED)} shifted runs; mean trades per run {SHIFTED.trades.mean():.0f} (real {len(REAL_RULES)}); '
      f'shifted expectancy median {SHIFTED.expectancy_R.median():+.3f}R, 5-95% [{SHIFTED.expectancy_R.quantile(0.05):+.3f}, {SHIFTED.expectancy_R.quantile(0.95):+.3f}]R')
hist_with_marker(SHIFTED.expectancy_R, real_exp, f'Time-shifted signals vs real rules-only OOS expectancy, {PRIMARY_SPLIT}',
                 'OOS expectancy (net R per trade)', 'shifted').show()
verdict('placebo_time_shift', p_shift < 0.05, f'real {real_exp:+.3f}R vs shifted 95th pct {SHIFTED.expectancy_R.quantile(0.95):+.3f}R, p = {p_shift:.3f}')

250 shifted runs; mean trades per run 499 (real 686); shifted expectancy median -0.137R, 5-95% [-0.357, +0.194]R


[FAIL] placebo_time_shift: real -0.047R vs shifted 95th pct +0.194R, p = 0.223


### 10.3 Placebo: shuffled ML labels
The meta-labeler is refit on IS trades with **randomly permuted labels**, using the same models, purged CV and threshold rule, and then applied OOS. If the real meta-labeler has learned something, its OOS filtered expectancy must beat the shuffled-label distribution.

In [27]:
if P.get('meta') is not None and REAL_ML is not None:
    real_ml_exp = metrics(REAL_ML, P['oos_cal'])['expectancy_R']
    sh = []
    y_true = ml_frame(P['is_raw'])[1]
    for k in range(N_LABEL_SHUFFLES):
        rng_k = np.random.default_rng(SEED + 1000 + k)
        meta_k = fit_meta(P['is_raw'], seed=SEED + k, labels=rng_k.permutation(y_true))
        kept, _ = ml_filter(P['oos_raw'], meta_k)
        sh.append(metrics(apply_portfolio(kept, 1.0, P_PARAMS['sizing']), P['oos_cal'])['expectancy_R'])
    SHUFFLED_ML = np.array(sh, dtype=float)
    p_ml = (1 + np.sum(SHUFFLED_ML >= real_ml_exp)) / (1 + len(SHUFFLED_ML))
    hist_with_marker(SHUFFLED_ML, real_ml_exp, f'Shuffled-label meta-labelers vs real ML-filtered OOS expectancy, {PRIMARY_SPLIT}',
                     'OOS expectancy of ML-kept trades (net R)', 'shuffled labels').show()
    verdict('placebo_shuffled_labels', p_ml < 0.05 and real_ml_exp > real_exp,
            f'real ML OOS {real_ml_exp:+.3f}R vs shuffled 95th pct {np.quantile(SHUFFLED_ML, 0.95):+.3f}R (p = {p_ml:.3f}); rules-only {real_exp:+.3f}R')
else:
    verdict('placebo_shuffled_labels', False, 'ML layer not fit on the primary split (too few IS trades)')
mem('placebo labels')

[FAIL] placebo_shuffled_labels: real ML OOS +0.005R vs shuffled 95th pct +0.088R (p = 0.238); rules-only -0.047R
[mem] placebo labels                             rss=   154.9 MB   peak=  1698.5 MB


### 10.4 Deflated Sharpe Ratio (Bailey & López de Prado, 2014) and minimum backtest length
`MinBTL_years` (Bailey, Borwein, López de Prado & Zhu, 2014) is the IS length needed before an annualized Sharpe this large stops being the expected maximum of `N` zero-skill trials. An IS window shorter than MinBTL cannot rule out pure selection luck. The best IS Sharpe is deflated by the expected maximum Sharpe of `N` independent trials, given the cross-trial Sharpe variance, and corrected for non-normal returns (skew, kurtosis) and sample length. `N` is the full number of logged trials for that split, failures included. A second column uses **every trial logged across all splits and folds**, the stricter count. PASS requires DSR ≥ 0.95. The OOS PSR (probabilistic Sharpe ratio vs 0) of the selected candidate is shown alongside; it needs no deflation, because it is a single pre-registered test.

In [28]:
EULER = 0.5772156649

def psr(sr, sr0, T, skew, kurt):
    return stats.norm.cdf((sr - sr0) * np.sqrt(T - 1) / np.sqrt(max(1e-12, 1 - skew * sr + (kurt - 1) / 4 * sr ** 2)))

def dsr(sr, T, skew, kurt, sr_var, N):
    e_max = np.sqrt(sr_var) * ((1 - EULER) * stats.norm.ppf(1 - 1 / N) + EULER * stats.norm.ppf(1 - 1 / (N * np.e)))
    return psr(sr, e_max, T, skew, kurt), e_max

N_GLOBAL = len(TRIAL_LOG)
dsr_rows = []
for name in list(SPLITS) + list(WF_FOLDS):
    if name not in EVAL:
        continue
    log, mat, cal = SEARCH[name]
    ok = (log.status == 'ok').values
    srs = np.array([r.mean() / r.std() if r.std() > 0 else np.nan for r in mat[ok]])
    best_i = int(np.nanargmax(np.where(ok, log.objective.values, -np.inf)))
    r = mat[best_i].astype(float)
    sr, T = r.mean() / r.std(), len(r)
    sk, ku = stats.skew(r), stats.kurtosis(r, fisher=False)
    d_split, emax = dsr(sr, T, sk, ku, np.nanvar(srs), len(log))
    d_glob, _ = dsr(sr, T, sk, ku, np.nanvar(srs), N_GLOBAL)
    ro = daily_returns(EVAL[name]['oos_port'][1.0]['rules'], EVAL[name]['oos_cal']).values
    sr_o = ro.mean() / ro.std() if ro.std() > 0 else 0.0
    z_n = (1 - EULER) * stats.norm.ppf(1 - 1 / len(log)) + EULER * stats.norm.ppf(1 - 1 / (len(log) * np.e))
    min_btl = (z_n / (sr * np.sqrt(252))) ** 2 if sr > 0 else np.inf     # Bailey et al. (2014), years
    dsr_rows.append({'split': name, 'N_trials': len(log), 'N_ok': int(ok.sum()), 'IS_years': T / 252, 'MinBTL_years': min_btl,
                     'IS_sharpe_ann': sr * np.sqrt(252),
                     'E[max SR]_ann': emax * np.sqrt(252), 'DSR (N split)': d_split, 'DSR (N global)': d_glob,
                     'OOS_sharpe_ann': sr_o * np.sqrt(252), 'OOS PSR(>0)': psr(sr_o, 0, len(ro), stats.skew(ro), stats.kurtosis(ro, fisher=False))})
DSR_TABLE = pd.DataFrame(dsr_rows).set_index('split')
display(DSR_TABLE.round(3))
fig = go.Figure()
fig.add_trace(go.Bar(x=DSR_TABLE.index, y=DSR_TABLE['DSR (N split)'], name='DSR (split trial count)', hovertemplate='%{x}: %{y:.3f}<extra></extra>'))
fig.add_trace(go.Bar(x=DSR_TABLE.index, y=DSR_TABLE['OOS PSR(>0)'], name='OOS PSR vs 0', hovertemplate='%{x}: %{y:.3f}<extra></extra>'))
fig.add_hline(y=0.95, line_dash='dash', annotation_text='0.95 threshold')
fig.update_layout(title='Deflated Sharpe Ratio (IS, multiple-testing adjusted) and OOS probabilistic Sharpe by split', barmode='group',
                  xaxis_title='split', yaxis_title='probability', yaxis_range=[0, 1.05], height=380)
fig.show()
n_pass = int((DSR_TABLE['DSR (N split)'] >= 0.95).sum())
verdict('deflated_sharpe', bool(DSR_TABLE.loc[PRIMARY_SPLIT, 'DSR (N split)'] >= 0.95),
        f"{PRIMARY_SPLIT} DSR = {DSR_TABLE.loc[PRIMARY_SPLIT, 'DSR (N split)']:.3f} (N={DSR_TABLE.loc[PRIMARY_SPLIT, 'N_trials']}); "
        f"{n_pass}/{len(DSR_TABLE)} splits/folds have DSR >= 0.95")

,N_trials,N_ok,IS_years,MinBTL_years,IS_sharpe_ann,E[max SR]_ann,DSR (N split),DSR (N global),OOS_sharpe_ann,OOS PSR(>0)
split,,,,,,,,,,
IS1y,250,198,1.008,1.799,2.115,4.556,0.001,0.0,-1.170,0.000
IS2y,250,161,2.008,11.476,0.838,3.652,0.000,0.0,-0.851,0.000
IS3y,250,194,3.008,6.205,1.139,2.806,0.001,0.0,-0.934,0.000
IS4y,250,180,4.004,60.327,0.365,2.896,0.000,0.0,-0.316,0.085
IS5y,250,167,5.000,21.106,0.618,2.389,0.000,0.0,-0.695,0.001
HALF,250,190,11.841,97.001,0.288,2.617,0.000,0.0,-0.203,0.245
WF2008,100,60,5.000,122.876,0.228,2.663,0.000,0.0,-2.011,0.000
WF2011,100,63,8.004,73.629,0.295,2.574,0.000,0.0,0.428,0.778
WF2014,100,64,10.996,4055.769,0.040,2.605,0.000,0.0,0.429,0.779


[FAIL] deflated_sharpe: HALF DSR = 0.000 (N=250); 0/12 splits/folds have DSR >= 0.95


### 10.4b Data-snooping tests: White's Reality Check, Hansen's SPA, Lo (2002) Sharpe
White (2000) and Hansen (2005) ask whether the **best of all tried configurations** beats a zero-return benchmark once the search is accounted for. They bootstrap the full IS trials × days matrix with a stationary bootstrap (Politis & Romano, 1994; mean block 10 days). SPA is the studentized, recentred version and is less distorted by poor trials. The Lo (2002) column re-annualizes the OOS Sharpe with the autocorrelation-corrected factor η(q) (lags ≤ 10) instead of √252. PASS requires the primary split's SPA p-value < 0.05.

In [29]:
def sb_indices(n, mean_block, rng):
    """Stationary-bootstrap index vector (vectorized): geometric block lengths, circular wrap."""
    new = rng.random(n) < 1.0 / mean_block
    new[0] = True
    bid = np.cumsum(new) - 1
    starts = rng.integers(0, n, size=bid[-1] + 1)
    pos = np.arange(n) - np.nonzero(new)[0][bid]
    return (starts[bid] + pos) % n

def reality_check_spa(mat, n_boot=1000, mean_block=10, seed=SEED):
    M = mat[np.isfinite(mat).all(axis=1) & (mat.std(axis=1) > 0)].astype(np.float64)
    if len(M) < 2:
        return np.nan, np.nan
    K, T = M.shape
    rng = np.random.default_rng(seed)
    mu = M.mean(axis=1)
    boots = np.empty((n_boot, K))
    for b in range(n_boot):
        boots[b] = M[:, sb_indices(T, mean_block, rng)].mean(axis=1)
    omega = np.sqrt(T) * boots.std(axis=0) + 1e-12
    V = np.max(np.sqrt(T) * mu)
    p_rc = float(np.mean(np.max(np.sqrt(T) * (boots - mu), axis=1) >= V))
    t = np.sqrt(T) * mu / omega
    mu_c = np.where(t >= -np.sqrt(2 * np.log(np.log(T))), mu, 0.0)
    spa_obs = max(0.0, t.max())
    spa_boot = np.maximum(0.0, np.max(np.sqrt(T) * (boots - mu_c) / omega, axis=1))
    return p_rc, float(np.mean(spa_boot >= spa_obs))

def lo_sharpe_annualized(r, q=252, max_lag=10):
    r = np.asarray(r, float)
    if r.std() == 0:
        return np.nan
    rho = [pd.Series(r).autocorr(k) for k in range(1, max_lag + 1)]
    eta = q / np.sqrt(q + 2 * sum((q - k) * rk for k, rk in enumerate(rho, 1) if np.isfinite(rk)))
    return r.mean() / r.std() * eta

rc_rows = []
for name in DSR_TABLE.index:
    log, mat, _ = SEARCH[name]
    p_rc, p_spa = reality_check_spa(mat[(log.status == 'ok').values], n_boot=max(200, N_BOOT // 2))
    ro = daily_returns(EVAL[name]['oos_port'][1.0]['rules'], EVAL[name]['oos_cal']).values
    rc_rows.append({'split': name, 'RealityCheck p': p_rc, 'SPA p': p_spa, 'OOS Sharpe (sqrt252)': DSR_TABLE.loc[name, 'OOS_sharpe_ann'],
                    'OOS Sharpe (Lo 2002 adj.)': lo_sharpe_annualized(ro)})
SNOOP_TABLE = pd.DataFrame(rc_rows).set_index('split')
display(SNOOP_TABLE.round(3))
verdict('reality_check_spa', bool(SNOOP_TABLE.loc[PRIMARY_SPLIT, 'SPA p'] < 0.05),
        f"{PRIMARY_SPLIT}: Reality Check p = {SNOOP_TABLE.loc[PRIMARY_SPLIT, 'RealityCheck p']:.3f}, SPA p = {SNOOP_TABLE.loc[PRIMARY_SPLIT, 'SPA p']:.3f}")

,RealityCheck p,SPA p,OOS Sharpe (sqrt252),OOS Sharpe (Lo 2002 adj.)
split,,,,
IS1y,0.907,0.426,-1.170,-1.091
IS2y,0.999,0.894,-0.851,-0.820
IS3y,0.577,0.506,-0.934,-0.928
IS4y,1.000,0.993,-0.316,-0.292
IS5y,0.999,0.879,-0.695,-0.724
HALF,1.000,0.962,-0.203,-0.192
WF2008,1.000,0.999,-2.011,-2.525
WF2011,0.999,0.960,0.428,0.441
WF2014,1.000,0.991,0.429,0.441


[FAIL] reality_check_spa: HALF: Reality Check p = 1.000, SPA p = 0.962


### 10.5 Probability of Backtest Overfitting (CSCV; Bailey, Borwein, López de Prado & Zhu)
The IS trials × days return matrix of each split is cut into S = 16 time blocks. For each of the C(16, 8) = 12,870 half/half combinations, the trial with the best Sharpe on one half is located on the other half, and its relative rank there gives a logit λ. PBO = P(λ ≤ 0), the probability that the IS-best configuration is below the median out of sample. PASS requires PBO < 0.5. Below 0.2 counts as strong evidence against overfitting.

In [30]:
def cscv_pbo(mat, S=16):
    M = mat[np.isfinite(mat).all(axis=1) & (mat.std(axis=1) > 0)]
    if len(M) < 5:
        return np.nan, np.array([]), np.array([]), np.array([])
    T = M.shape[1] - M.shape[1] % S
    blocks = np.array_split(np.arange(T), S)
    s1 = np.stack([M[:, b].sum(axis=1) for b in blocks], 1)
    s2 = np.stack([(M[:, b] ** 2).sum(axis=1) for b in blocks], 1)
    nb = np.array([len(b) for b in blocks])
    lam, is_sr, oos_sr = [], [], []
    for comb in itertools.combinations(range(S), S // 2):
        mask = np.zeros(S, bool); mask[list(comb)] = True
        def sr(m):
            n = nb[m].sum(); mu = s1[:, m].sum(1) / n
            return mu / np.sqrt(np.maximum(s2[:, m].sum(1) / n - mu ** 2, 1e-18))
        a, b = sr(mask), sr(~mask)
        i = int(np.argmax(a))
        w = (np.argsort(np.argsort(b))[i] + 1) / (len(b) + 1)
        lam.append(np.log(w / (1 - w))); is_sr.append(a[i]); oos_sr.append(b[i])
    lam = np.array(lam)
    return float((lam <= 0).mean()), lam, np.array(is_sr) * np.sqrt(252), np.array(oos_sr) * np.sqrt(252)

PBO = {}
for name in list(SPLITS) + list(WF_FOLDS):
    if name in SEARCH:
        log, mat, _ = SEARCH[name]
        PBO[name] = cscv_pbo(mat[(log.status == 'ok').values])
PBO_TABLE = pd.Series({k: v[0] for k, v in PBO.items()}, name='PBO').to_frame()
display(PBO_TABLE.round(3))
pb, lam, iss, oss = PBO[PRIMARY_SPLIT]
fig = make_subplots(rows=1, cols=2, subplot_titles=[f'{PRIMARY_SPLIT}: logit distribution (PBO = {pb:.2f})', 'IS-best Sharpe: IS half vs OOS half'])
fig.add_trace(go.Histogram(x=lam, nbinsx=40, name='λ', hovertemplate='λ %{x:.2f}<br>count %{y}<extra></extra>'), 1, 1)
fig.add_vline(x=0, line_dash='dash', row=1, col=1)
fig.add_trace(go.Scattergl(x=iss, y=oss, mode='markers', marker=dict(size=3, opacity=0.4), name='combinations',
                           hovertemplate='IS SR %{x:.2f}<br>OOS SR %{y:.2f}<extra></extra>'), 1, 2)
fig.update_xaxes(title_text='logit of OOS relative rank (λ)', row=1, col=1)
fig.update_yaxes(title_text='count of combinations', row=1, col=1)
fig.update_xaxes(title_text='IS Sharpe of selected trial (annualized)', row=1, col=2)
fig.update_yaxes(title_text='OOS Sharpe of same trial (annualized)', row=1, col=2)
fig.update_layout(title='Probability of Backtest Overfitting (CSCV, S=16)', height=400, showlegend=False)
fig.show()
fig = go.Figure(go.Bar(x=PBO_TABLE.index, y=PBO_TABLE.PBO, hovertemplate='%{x}: PBO %{y:.3f}<extra></extra>'))
fig.add_hline(y=0.5, line_dash='dash', annotation_text='0.5 (coin flip)')
fig.update_layout(title='PBO by split / walk-forward fold', xaxis_title='split', yaxis_title='PBO (probability)', yaxis_range=[0, 1], height=340)
fig.show()
verdict('pbo_cscv', bool(pb < 0.5), f'{PRIMARY_SPLIT} PBO = {pb:.3f}; splits with PBO < 0.5: {int((PBO_TABLE.PBO < 0.5).sum())}/{len(PBO_TABLE)}')
mem('DSR/PBO')

,PBO
IS1y,0.377
IS2y,0.516
IS3y,0.400
IS4y,0.660
IS5y,0.671
HALF,0.246
WF2008,0.216
WF2011,0.254
WF2014,0.202
WF2017,0.041


[PASS] pbo_cscv: HALF PBO = 0.246; splits with PBO < 0.5: 9/12
[mem] DSR/PBO                                    rss=   221.1 MB   peak=  1698.5 MB


### 10.6 Parameter stability (IS only)
Pairs of parameters are varied on a grid around the selected configuration, on the **IS window only**, so OOS stays untouched. A robust edge shows a plateau: neighbours of the selected cell keep most of its Sharpe. A spike means the optimizer found noise. PASS requires the selected IS Sharpe to be positive and at least 60% of the immediate neighbours to keep ≥ 50% of it.

In [31]:
def _grid(v, step, lo, hi, n=2):
    """Grid centred EXACTLY on the selected value v (so the selected cell is always on the grid), clipped to bounds."""
    vals = [v + k * step for k in range(-n, n + 1)]
    return sorted(set(round(float(x), 6) for x in vals if lo - 1e-9 <= x <= hi + 1e-9) | {round(float(v), 6)})

def _idx(vals, v):
    hits = [i for i, x in enumerate(vals) if np.isclose(float(x), float(v), atol=1e-6)]
    return hits[0] if hits else None

pp = P_PARAMS
tgt_key, tgt_step, tgt_lo, tgt_hi = ('R_mult', 0.25, 1.0, 4.0) if pp['target_mode'] == 'R' else ('min_R', 0.25, 0.75, 2.0)
PAIRS = [
    ('sd_max', _grid(pp['sd_max'], 0.25, 0.5, 2.5), 'm_reclaim', sorted(set(int(x) for x in _grid(pp['m_reclaim'], 1, 0, 10)))),
    (tgt_key, _grid(pp[tgt_key], tgt_step, tgt_lo, tgt_hi), 'stop_buf', _grid(pp['stop_buf'], 0.05, 0.0, 0.5)),
    ('trend_tf', list(TREND_TFS), 'trend_len', list(TREND_LENS)),
    ('z', _grid(pp['z'], 0.25, 1.5, 5.0), 'rr_max', _grid(pp['rr_max'], 0.05, 0.45, 1.0)),
]
is0, is1 = SPLIT_DEF[PRIMARY_SPLIT][:2]
grid_cfgs, grid_keys = [], []
for a, av, b, bv in PAIRS:
    for x in dict.fromkeys(av):
        for y in dict.fromkeys(bv):
            cfg = dict(pp, **{a: x, b: y})
            if 'risk_min' in (a, b):
                cfg['risk_max'] = cfg['risk_min'] + (pp['risk_max'] - pp['risk_min'])
            grid_cfgs.append(cfg); grid_keys.append((a, x, b, y))
t0 = time.time()
grid_res = evaluate_params(grid_cfgs, is0, is1)
cal_is = cal_between(is0, is1)
grid_sh = [metrics(apply_portfolio(r, 1.0, c['sizing']), cal_is)['sharpe'] if not isinstance(r, Exception) and len(r) else np.nan
           for r, c in zip(grid_res, grid_cfgs)]
print(f'{len(grid_cfgs)} IS grid evaluations in {time.time() - t0:.0f}s')
best_sh = metrics(P['is_port'][1.0]['rules'], P['is_cal'])['sharpe']
fig = make_subplots(rows=2, cols=2, subplot_titles=[f'{a} × {b}' for a, _, b, _ in PAIRS], horizontal_spacing=0.12, vertical_spacing=0.15)
neigh = []
for k, (a, av, b, bv) in enumerate(PAIRS):
    av, bv = list(dict.fromkeys(av)), list(dict.fromkeys(bv))
    Z = np.full((len(bv), len(av)), np.nan)
    for (ka, x, kb, y), s_ in zip(grid_keys, grid_sh):
        if ka == a and kb == b:
            Z[_idx(bv, y), _idx(av, x)] = s_
    ia, ib = _idx(av, pp[a]), _idx(bv, pp[b])
    if ia is None or ib is None:
        print(f'WARNING: selected value not on the {a} x {b} grid')
    if ia is not None and ib is not None:
        for da in (-1, 0, 1):
            for db in (-1, 0, 1):
                if (da or db) and 0 <= ia + da < len(av) and 0 <= ib + db < len(bv):
                    neigh.append(Z[ib + db, ia + da])
    fig.add_trace(go.Heatmap(z=Z, x=[str(v) for v in av], y=[str(v) for v in bv], coloraxis='coloraxis',
                             hovertemplate=f'{a}=%{{x}}<br>{b}=%{{y}}<br>IS Sharpe %{{z:.2f}}<extra></extra>'), k // 2 + 1, k % 2 + 1)
    if ia is not None and ib is not None:
        fig.add_trace(go.Scatter(x=[str(av[ia])], y=[str(bv[ib])], mode='markers', marker=dict(symbol='star', size=14, color='white', line=dict(color='black', width=1)),
                                 showlegend=False, hovertemplate='selected<extra></extra>'), k // 2 + 1, k % 2 + 1)
    fig.update_xaxes(title_text=a, row=k // 2 + 1, col=k % 2 + 1)
    fig.update_yaxes(title_text=b, row=k // 2 + 1, col=k % 2 + 1)
fig.update_layout(title=f'Parameter stability around the selected configuration ({PRIMARY_SPLIT} IS Sharpe, annualized; ★ = selected)',
                  coloraxis=dict(colorscale='RdBu', cmid=0, colorbar=dict(title='IS Sharpe')), height=760)
fig.show()
neigh = np.array(neigh, dtype=float)
frac = float(np.mean(neigh >= 0.5 * best_sh)) if best_sh > 0 and len(neigh) else 0.0
verdict('parameter_plateau', bool(best_sh > 0 and frac >= 0.6),
        f'selected IS Sharpe {best_sh:.2f}; {frac:.0%} of {len(neigh)} neighbours keep >= 50% of it (median neighbour {np.nanmedian(neigh):.2f})')
free('grid_res'); mem('parameter stability')

59 IS grid evaluations in 69s


[PASS] parameter_plateau: selected IS Sharpe 0.29; 78% of 23 neighbours keep >= 50% of it (median neighbour 0.23)
[mem] parameter stability                        rss=   785.6 MB   peak=  1698.5 MB


### 10.7 Monte Carlo: trade-order reshuffle and stationary bootstrap
(a) The OOS trade sequence is reshuffled 2,000 times: the total stays the same, so this shows the drawdown distribution that luck in ordering produces. (b) A stationary block bootstrap of OOS daily returns (Politis & Romano, mean block 10 days) gives confidence intervals for annual return, Sharpe and max drawdown. PASS requires the bootstrap 5th-percentile Sharpe to be above 0.

In [32]:
def stationary_bootstrap(r, n_boot, mean_block=10, seed=SEED):
    rng = np.random.default_rng(seed)
    n = len(r)
    out = np.empty((n_boot, 3))
    for b in range(n_boot):
        x = r[sb_indices(n, mean_block, rng)]
        eq = np.cumprod(1 + x)
        out[b] = [eq[-1] ** (252 / n) - 1, x.mean() / x.std() * np.sqrt(252) if x.std() > 0 else 0, (eq / np.maximum.accumulate(eq) - 1).min()]
    return pd.DataFrame(out, columns=['ann_return', 'sharpe', 'max_dd'])

MC = {}
for var, t in [('rules', REAL_RULES), ('ml', REAL_ML)]:
    if t is None or len(t) == 0:
        continue
    rets = t['ret'].values
    dds = []
    rng = np.random.default_rng(SEED)
    for _ in range(N_BOOT):
        eq = np.cumprod(1 + rng.permutation(rets))
        dds.append((eq / np.maximum.accumulate(eq) - 1).min())
    MC[var] = {'shuffle_dd': np.array(dds), 'boot': stationary_bootstrap(daily_returns(t, P['oos_cal']).values, N_BOOT)}
mc_rows = []
for var, d in MC.items():
    b = d['boot']
    mc_rows.append({'variant': var, 'reshuffle maxDD p5': np.quantile(d['shuffle_dd'], 0.05), 'reshuffle maxDD median': np.median(d['shuffle_dd']),
                    'boot ann.ret p5': b.ann_return.quantile(0.05), 'boot ann.ret p50': b.ann_return.median(), 'boot ann.ret p95': b.ann_return.quantile(0.95),
                    'boot Sharpe p5': b.sharpe.quantile(0.05), 'boot Sharpe p50': b.sharpe.median(), 'boot Sharpe p95': b.sharpe.quantile(0.95),
                    'boot maxDD p5': b.max_dd.quantile(0.05), 'boot maxDD p50': b.max_dd.median()})
MC_TABLE = pd.DataFrame(mc_rows).set_index('variant')
display(MC_TABLE.round(3))
fig = make_subplots(rows=1, cols=2, subplot_titles=['Trade-order reshuffle: max drawdown', 'Stationary bootstrap: annualized Sharpe'])
for var, d in MC.items():
    fig.add_trace(go.Histogram(x=d['shuffle_dd'] * 100, name=f'{var} maxDD', opacity=0.6, nbinsx=50, hovertemplate='maxDD %{x:.2f}%<br>count %{y}<extra></extra>'), 1, 1)
    fig.add_trace(go.Histogram(x=d['boot'].sharpe, name=f'{var} Sharpe', opacity=0.6, nbinsx=50, hovertemplate='Sharpe %{x:.2f}<br>count %{y}<extra></extra>'), 1, 2)
fig.add_vline(x=0, line_dash='dash', row=1, col=2)
fig.update_xaxes(title_text='max drawdown (%)', row=1, col=1); fig.update_xaxes(title_text='annualized Sharpe', row=1, col=2)
fig.update_yaxes(title_text='count of resamples')
fig.update_layout(title=f'Monte Carlo on {PRIMARY_SPLIT} OOS (1x costs)', barmode='overlay', height=380)
fig.show()
verdict('monte_carlo', bool(MC_TABLE.loc['rules', 'boot Sharpe p5'] > 0),
        f"rules-only bootstrap Sharpe 90% CI [{MC_TABLE.loc['rules', 'boot Sharpe p5']:.2f}, {MC_TABLE.loc['rules', 'boot Sharpe p95']:.2f}]"
        + (f"; ML [{MC_TABLE.loc['ml', 'boot Sharpe p5']:.2f}, {MC_TABLE.loc['ml', 'boot Sharpe p95']:.2f}]" if 'ml' in MC_TABLE.index else ''))
mem('monte carlo')

,reshuffle maxDD p5,reshuffle maxDD median,boot ann.ret p5,boot ann.ret p50,boot ann.ret p95,boot Sharpe p5,boot Sharpe p50,boot Sharpe p95,boot maxDD p5,boot maxDD p50
variant,,,,,,,,,,
rules,-0.272,-0.202,-0.031,-0.010,0.013,-0.700,-0.201,0.300,-0.356,-0.209
ml,-0.214,-0.151,-0.023,-0.002,0.019,-0.569,-0.035,0.451,-0.295,-0.162


[FAIL] monte_carlo: rules-only bootstrap Sharpe 90% CI [-0.70, 0.30]; ML [-0.57, 0.45]
[mem] monte carlo                                rss=   784.7 MB   peak=  1698.5 MB


### 10.8 Cross-sectional consistency
How many ETFs are profitable out of sample on their own (net R > 0)? A real, general effect should show up across most of the universe, not in one or two names. PASS requires at least two-thirds of the ETFs that traded to be profitable OOS, with a one-sided sign test p < 0.05.

In [33]:
def per_symbol(t):
    g = t.groupby('sym', observed=True)['R']
    out = g.agg(trades='size', expectancy_R='mean', total_R='sum')
    out['ci_lo'] = g.apply(lambda x: np.quantile(np.random.default_rng(SEED).choice(x.values, (500, len(x))).mean(1), 0.025))
    out['ci_hi'] = g.apply(lambda x: np.quantile(np.random.default_rng(SEED).choice(x.values, (500, len(x))).mean(1), 0.975))
    return out

XS = per_symbol(REAL_RULES)
xs_ml = per_symbol(REAL_ML) if REAL_ML is not None and len(REAL_ML) else None
display(XS.round(3))
fig = go.Figure()
fig.add_trace(go.Bar(x=XS.index.astype(str), y=XS.expectancy_R, name='rules-only', error_y=dict(type='data', symmetric=False,
                     array=XS.ci_hi - XS.expectancy_R, arrayminus=XS.expectancy_R - XS.ci_lo),
                     customdata=XS.trades, hovertemplate='%{x}: %{y:+.3f}R/trade<br>%{customdata} trades<extra>rules</extra>'))
if xs_ml is not None:
    fig.add_trace(go.Bar(x=xs_ml.index.astype(str), y=xs_ml.expectancy_R, name='ML-filtered', customdata=xs_ml.trades,
                         hovertemplate='%{x}: %{y:+.3f}R/trade<br>%{customdata} trades<extra>ML</extra>'))
fig.add_hline(y=0, line_color='black')
fig.update_layout(title=f'Per-ETF OOS expectancy, {PRIMARY_SPLIT} split, 1x costs (95% bootstrap CI)', xaxis_title='ETF',
                  yaxis_title='net R per trade', barmode='group', height=400)
fig.show()
xs_split = {}
for name, E_ in EVAL.items():
    t_ = E_['oos_port'][1.0]['rules']
    if len(t_):
        s_ = t_.groupby('sym', observed=True)['R'].sum()
        xs_split[name] = f'{int((s_ > 0).sum())}/{len(s_)}'
print('ETFs profitable OOS (rules-only, 1x) by split:', xs_split)
n_pos, n_tot = int((XS.total_R > 0).sum()), len(XS)
p_sign = stats.binomtest(n_pos, n_tot, 0.5, alternative='greater').pvalue if n_tot else 1.0
verdict('cross_sectional', bool(n_tot and n_pos / n_tot >= 2 / 3 and p_sign < 0.05), f'{n_pos}/{n_tot} ETFs profitable OOS; sign-test p = {p_sign:.3f}')

,trades,expectancy_R,total_R,ci_lo,ci_hi
sym,,,,,
DIA,68,0.021,1.408,-0.355,0.380
GLD,50,-0.506,-25.290,-0.816,-0.182
IWM,201,-0.195,-39.114,-0.406,0.002
QQQ,153,-0.056,-8.543,-0.274,0.202
SLV,13,0.525,6.830,-0.254,1.396
SPY,169,0.248,41.949,-0.005,0.505
TLT,31,-0.264,-8.177,-0.788,0.278
USO,1,-1.112,-1.112,-1.112,-1.112


ETFs profitable OOS (rules-only, 1x) by split: {'IS1y': '0/8', 'IS2y': '2/9', 'IS3y': '0/8', 'IS4y': '1/9', 'IS5y': '1/8', 'HALF': '3/8', 'WF2008': '2/8', 'WF2011': '6/8', 'WF2014': '2/4', 'WF2017': '2/4', 'WF2020': '2/6', 'WF2023': '2/7'}
[FAIL] cross_sectional: 3/8 ETFs profitable OOS; sign-test p = 0.855


### 10.9 Lookahead audit (automated)
1. **Feature truncation test.** For several ETFs and random intraday cut points, all features are rebuilt *from bars truncated at the cut*, including the daily table. They must equal the cached full-history features at every bar ≤ cut.
2. **Signal truncation test.** The selected configuration's signals, stops and targets on truncated data must equal the full-data values at every bar ≤ cut.
3. **Engine truncation test.** Every trade that has exited by the cut must be identical in both runs.
4. **Structural checks.** Entry is always the bar after the signal. No ML feature is a post-entry quantity. The HTF trend of every sampled bar matches a from-scratch recomputation using only HTF bars completed before it.

In [34]:
LOOKAHEAD = []
rng_la = np.random.default_rng(SEED + 7)
audit_syms = [s for s in ['SPY', 'TLT', 'GLD', 'IWM'] if s in SYMBOLS][:3]
for sym in audit_syms:
    g_full = pd.read_parquet(DIRS['bars'] / f'{sym}.parquet', filters=[('year', '>=', 2013), ('year', '<=', 2015)])
    f_full = compute_features(g_full)
    for _ in range(3):
        cut = int(rng_la.integers(len(g_full) // 3, len(g_full) - 10))
        f_cut = compute_features(g_full.iloc[:cut + 1].copy())
        num = [c for c in f_full.columns if c not in ('year',)]
        a, b = f_full[num].iloc[:cut + 1].to_numpy(np.float64), f_cut[num].to_numpy(np.float64)
        same = np.isclose(a, b, rtol=1e-5, atol=1e-6, equal_nan=True)
        bad_cols = [num[j] for j in np.nonzero(~same.all(axis=0))[0]]
        LOOKAHEAD.append({'test': 'features', 'sym': sym, 'cut': cut, 'mismatched_columns': ','.join(bad_cols) or '-', 'pass': not bad_cols})
        s_full = generate_signals(sym, f_full, P_PARAMS)
        s_cut = generate_signals(sym, f_cut, P_PARAMS)
        ok_sig = (np.array_equal(s_full[0][:cut + 1], s_cut[0]) and np.allclose(s_full[1][:cut + 1], s_cut[1], equal_nan=True)
                  and np.allclose(s_full[2][:cut + 1], s_cut[2], equal_nan=True))
        LOOKAHEAD.append({'test': 'signals', 'sym': sym, 'cut': cut, 'mismatched_columns': '-' if ok_sig else 'sig/stop/tgt', 'pass': ok_sig})
        t_full, t_cut = run_symbol(sym, f_full, P_PARAMS), run_symbol(sym, f_cut, P_PARAMS)
        cols_cmp = ['sig_idx', 'entry_idx', 'exit_idx', 'dir', 'entry', 'exit', 'stop', 'target']
        a_ = t_full[t_full.exit_idx < cut][cols_cmp].reset_index(drop=True)
        b_ = t_cut[t_cut.exit_idx < cut][cols_cmp].reset_index(drop=True)
        ok_tr = a_.shape == b_.shape and np.allclose(a_.to_numpy(float), b_.to_numpy(float))
        LOOKAHEAD.append({'test': 'engine', 'sym': sym, 'cut': cut, 'mismatched_columns': '-' if ok_tr else f'{len(a_)} vs {len(b_)} trades', 'pass': ok_tr})
    # HTF trend from scratch (independent pandas implementation): bar i must see the direction of the last HTF bar
    # that COMPLETED before i's own HTF bucket, computed only from bars in completed buckets.
    tf, n_ = P_PARAMS['trend_tf'], P_PARAMS['trend_len']
    col = f'tdir_{tf}_{n_}'
    samp = rng_la.choice(np.arange(5000, len(g_full)), size=25, replace=False)
    good = 0
    for i in samp:
        gi = g_full.iloc[:i + 1]
        key = gi.date.values.astype(np.int64) * 1000 + (gi.minute.values.astype(np.int64) - OPEN_MIN) // tf
        prior = gi[key < key[-1]]
        grp = prior.groupby(key[key < key[-1]], sort=True)
        hc = grp.c.last().values.astype(float); hh = grp.h.max().values.astype(float); hl = grp.l.min().values.astype(float)
        pc = np.r_[hc[0], hc[:-1]]
        hatr = pd.Series(np.maximum(hh - hl, np.maximum(abs(hh - pc), abs(hl - pc)))).ewm(alpha=1 / ATR_N, adjust=False, min_periods=ATR_N).mean().values
        ema = pd.Series(hc).ewm(span=n_, adjust=False, min_periods=n_).mean().values
        slope = (ema[-1] - ema[-4]) / hatr[-1] if len(ema) >= 4 else np.nan
        d_scratch = 0 if not np.isfinite(slope) else (1 if (hc[-1] > ema[-1] and slope > 0) else (-1 if (hc[-1] < ema[-1] and slope < 0) else 0))
        good += int(d_scratch == int(f_full[col].iloc[i]))
    LOOKAHEAD.append({'test': 'htf_trend', 'sym': sym, 'cut': -1, 'mismatched_columns': f'{good}/{len(samp)} match', 'pass': good == len(samp)})
    del g_full, f_full, f_cut
    gc.collect()

# structural checks over every trade log produced in §8-§9
post_entry = {'R_gross', 'cost_R', 'exit', 'exit_idx', 'exit_min', 'reason', 'bars_held', 'risk_ps', 'risk_atr', 'lev_req', 'entry', 'R', 'ret'}
LOOKAHEAD.append({'test': 'no post-entry ML features', 'sym': '-', 'cut': -1,
                  'mismatched_columns': ','.join(sorted(post_entry & set(ML_FEATURES))) or '-', 'pass': not (post_entry & set(ML_FEATURES))})
bad = sum(int((E_['oos_raw'].entry_idx != E_['oos_raw'].sig_idx + 1).sum() + (E_['is_raw'].entry_idx != E_['is_raw'].sig_idx + 1).sum())
          for E_ in EVAL.values() if len(E_['oos_raw']))
LOOKAHEAD.append({'test': 'entry = signal bar + 1 (all logs)', 'sym': '-', 'cut': -1, 'mismatched_columns': f'{bad} violations', 'pass': bad == 0})
LOOKAHEAD = pd.DataFrame(LOOKAHEAD)
display(LOOKAHEAD)
verdict('lookahead_audit', bool(LOOKAHEAD['pass'].all()), f"{int(LOOKAHEAD['pass'].sum())}/{len(LOOKAHEAD)} audit checks passed")
mem('lookahead audit')

,test,sym,cut,mismatched_columns,pass
0,features,SPY,292853,-,True
1,signals,SPY,292853,-,True
2,engine,SPY,292853,-,True
3,features,SPY,273934,-,True
4,signals,SPY,273934,-,True
5,engine,SPY,273934,-,True
6,features,SPY,110902,-,True
7,signals,SPY,110902,-,True
8,engine,SPY,110902,-,True
9,htf_trend,SPY,-1,25/25 match,True


[PASS] lookahead_audit: 32/32 audit checks passed
[mem] lookahead audit                            rss=  1136.7 MB   peak=  1698.5 MB


## 11 VIX regime analysis

VIX is never traded. Each session is labelled with the **prior day's VIX close**, since today's close is unknown intraday, and with that close's percentile within the previous 252 VIX closes. OOS trades of every split are bucketed by absolute level (<15, 15–20, 20–30, >30) and by percentile quintile. Expectancy gets a 95% bootstrap CI, and any bucket with fewer than 30 trades is flagged as unreliable.

In [35]:
def bucket_stats(t, bucket_days, label):
    """Trade + daily stats for one bucket; bucket_days = the calendar days that belong to the bucket."""
    n = 0 if t is None else len(t)
    R = t['R'].values if n else np.array([])
    days = np.array(sorted(bucket_days))
    r = daily_returns(t, days) if len(days) and n else pd.Series(np.zeros(len(days)))
    eq = (1 + r).cumprod() if len(r) else pd.Series([1.0])
    if n:
        boot = np.random.default_rng(SEED).choice(R, size=(N_BOOT, n), replace=True).mean(axis=1)
        lo, hi = np.quantile(boot, [0.025, 0.975])
    else:
        lo = hi = np.nan
    wins, losses = R[R > 0], R[R <= 0]
    return {'bucket': label, 'trades': n, 'days': len(days), 'win_rate': len(wins) / n if n else np.nan,
            'expectancy_R': R.mean() if n else np.nan, 'exp_ci_lo': lo, 'exp_ci_hi': hi,
            'profit_factor': wins.sum() / -losses.sum() if len(losses) and losses.sum() < 0 else np.nan,
            'sharpe': r.mean() / r.std() * np.sqrt(252) if len(r) > 2 and r.std() > 0 else np.nan,
            'max_dd': float((eq / eq.cummax() - 1).min()), 'reliable': n >= 30}

VIX_DAY = pd.DataFrame({'date': CALENDAR})
_vx = VIX[['date', 'close']].copy()
_vx['pct'] = _vx.close.rolling(252, min_periods=60).apply(lambda a: (a[:-1] < a[-1]).mean(), raw=True)
_sess = pd.DataFrame({'dt': pd.to_datetime(pd.Series(CALENDAR).astype(str))})
_m = pd.merge_asof(_sess, _vx.rename(columns={'date': 'dt_v'}), left_on='dt', right_on='dt_v', direction='backward', allow_exact_matches=False)
VIX_DAY['vix_prev'] = _m['close'].values
VIX_DAY['vix_pct_prev'] = _m['pct'].values
VIX_DAY = VIX_DAY.set_index('date')
VIX_LEVELS = [(-np.inf, 15, '<15'), (15, 20, '15-20'), (20, 30, '20-30'), (30, np.inf, '>30')]
VIX_PCTS = [(0, 0.2, 'P0-20'), (0.2, 0.4, 'P20-40'), (0.4, 0.6, 'P40-60'), (0.6, 0.8, 'P60-80'), (0.8, 1.01, 'P80-100')]

def vix_breakdown(t, cal, split, variant):
    if t is None or len(t) == 0:
        return []
    dv = VIX_DAY.reindex(cal)
    tv = VIX_DAY.reindex(t['date'].values)
    rows = []
    for col, bins, dim in [('vix_prev', VIX_LEVELS, 'VIX level'), ('vix_pct_prev', VIX_PCTS, 'VIX percentile')]:
        for lo, hi, lab in bins:
            mt = (tv[col].values >= lo) & (tv[col].values < hi)
            md = (dv[col].values >= lo) & (dv[col].values < hi)
            rows.append({'split': split, 'variant': variant, 'dimension': dim, **bucket_stats(t[mt], dv.index[md], lab)})
    return rows

vix_rows = []
for name, E_ in EVAL.items():
    for var in ['rules', 'ml']:
        vix_rows += vix_breakdown(E_['oos_port'][1.0][var], E_['oos_cal'], name, var)
for var in ['rules', 'ml']:
    if WF_OOS.get((var, 1.0)) is not None:
        vix_rows += vix_breakdown(WF_OOS[(var, 1.0)], wf_cal, 'WF stitched', var)
VIX_TABLE = pd.DataFrame(vix_rows)
VIX_TABLE.to_parquet(DIRS['results'] / 'vix_regimes.parquet', index=False)
display(VIX_TABLE[(VIX_TABLE.split == PRIMARY_SPLIT)].drop(columns=['split']).round(3))

# mechanism check: is the regime difference gross edge, or costs measured in R (stops are ATR-scaled, costs are cents)?
_t = EVAL[PRIMARY_SPLIT]['oos_port'][1.0]['rules']
_lv = VIX_DAY.reindex(_t['date'].values)['vix_prev'].values
VIX_COSTS = pd.DataFrame([{'bucket': lab, 'trades': int(((_lv >= lo) & (_lv < hi)).sum()),
                           'mean_gross_R': _t['R_gross'].values[(_lv >= lo) & (_lv < hi)].mean(),
                           'mean_cost_R': _t['cost_R'].values[(_lv >= lo) & (_lv < hi)].mean(),
                           'mean_net_R': _t['R'].values[(_lv >= lo) & (_lv < hi)].mean(),
                           'median_risk_atr': np.median(_t['risk_atr'].values[(_lv >= lo) & (_lv < hi)])}
                          for lo, hi, lab in VIX_LEVELS]).set_index('bucket')
print(f'{PRIMARY_SPLIT} rules-only OOS: gross R, cost in R and net R by prior-day VIX level')
display(VIX_COSTS.round(3))

,variant,dimension,bucket,trades,days,win_rate,expectancy_R,exp_ci_lo,exp_ci_hi,profit_factor,sharpe,max_dd,reliable
54,rules,VIX level,<15,70,1074,0.286,-0.360,-0.659,-0.054,0.535,-1.139,-0.093,True
55,rules,VIX level,15-20,256,1043,0.383,-0.015,-0.195,0.167,0.976,-0.055,-0.076,True
56,rules,VIX level,20-30,298,710,0.349,-0.064,-0.238,0.102,0.907,-0.312,-0.115,True
57,rules,VIX level,>30,62,158,0.387,0.260,-0.164,0.694,1.400,1.174,-0.031,True
58,rules,VIX percentile,P0-20,81,805,0.346,-0.123,-0.438,0.225,0.832,-0.253,-0.033,True
59,rules,VIX percentile,P20-40,88,526,0.330,-0.205,-0.488,0.087,0.717,-1.314,-0.085,True
60,rules,VIX percentile,P40-60,97,500,0.289,-0.335,-0.586,-0.064,0.580,-1.646,-0.116,True
61,rules,VIX percentile,P60-80,159,570,0.421,0.105,-0.120,0.366,1.175,0.781,-0.038,True
62,rules,VIX percentile,P80-100,261,584,0.360,0.045,-0.144,0.227,1.069,0.203,-0.076,True
63,ml,VIX level,<15,61,1074,0.295,-0.331,-0.640,-0.003,0.553,-1.027,-0.080,True


HALF rules-only OOS: gross R, cost in R and net R by prior-day VIX level


,trades,mean_gross_R,mean_cost_R,mean_net_R,median_risk_atr
bucket,,,,,
<15,70,-0.165,0.195,-0.360,1.842
15-20,256,0.129,0.144,-0.015,1.798
20-30,298,0.069,0.133,-0.064,1.760
>30,62,0.356,0.096,0.260,1.521


In [36]:
def bucket_bar_chart(df, title, xlab):
    fig = go.Figure()
    for var, colr in [('rules', '#2a6fdb'), ('ml', '#e07b39')]:
        d = df[df.variant == var]
        if d.empty:
            continue
        fig.add_trace(go.Bar(x=d.bucket, y=d.expectancy_R, name='rules-only' if var == 'rules' else 'ML-filtered', marker_color=colr,
                             marker_pattern_shape=np.where(d.reliable, '', 'x'),
                             error_y=dict(type='data', symmetric=False, array=d.exp_ci_hi - d.expectancy_R, arrayminus=d.expectancy_R - d.exp_ci_lo),
                             customdata=np.c_[d.trades, d.win_rate, d.profit_factor],
                             hovertemplate='%{x}<br>expectancy %{y:+.3f}R<br>trades %{customdata[0]}<br>win %{customdata[1]:.1%}<br>PF %{customdata[2]:.2f}<extra></extra>'))
    fig.add_hline(y=0, line_color='black')
    fig.update_layout(title=title + ' (hatched = fewer than 30 trades, unreliable; bars = 95% bootstrap CI)', xaxis_title=xlab,
                      yaxis_title='OOS net R per trade', barmode='group', height=400)
    return fig

for dim, xlab in [('VIX level', 'prior-day VIX close'), ('VIX percentile', 'prior-day VIX percentile (trailing 252d)')]:
    bucket_bar_chart(VIX_TABLE[(VIX_TABLE.split == PRIMARY_SPLIT) & (VIX_TABLE.dimension == dim)],
                     f'{PRIMARY_SPLIT} OOS expectancy by {dim}', xlab).show()

piv = VIX_TABLE[(VIX_TABLE.variant == 'rules') & (VIX_TABLE.dimension == 'VIX level')].pivot(index='split', columns='bucket', values='expectancy_R')
piv = piv[[b for _, _, b in VIX_LEVELS if b in piv.columns]]
cnt = VIX_TABLE[(VIX_TABLE.variant == 'rules') & (VIX_TABLE.dimension == 'VIX level')].pivot(index='split', columns='bucket', values='trades')[piv.columns]
fig = go.Figure(go.Heatmap(z=piv.values, x=piv.columns, y=piv.index, colorscale='RdBu', zmid=0, customdata=cnt.values,
                           colorbar=dict(title='net R/trade'),
                           hovertemplate='%{y} | VIX %{x}<br>expectancy %{z:+.3f}R<br>trades %{customdata}<extra></extra>'))
fig.update_layout(title='Rules-only OOS expectancy by VIX regime across all splits (1x costs)', xaxis_title='prior-day VIX close',
                  yaxis_title='split', height=420)
fig.show()

def regime_spread(df):
    d = df[df.reliable]
    return (d.expectancy_R.max() - d.expectancy_R.min(), d.loc[d.expectancy_R.idxmax(), 'bucket'], d.loc[d.expectancy_R.idxmin(), 'bucket']) if len(d) >= 2 else (np.nan, None, None)

VIX_SUMMARY = {}
for var in ['rules', 'ml']:
    d = VIX_TABLE[(VIX_TABLE.split == PRIMARY_SPLIT) & (VIX_TABLE.variant == var) & (VIX_TABLE.dimension == 'VIX level')]
    if len(d):
        spread, best_b, worst_b = regime_spread(d)
        pos = d[d.reliable & (d.exp_ci_lo > 0)].bucket.tolist()
        VIX_SUMMARY[var] = {'best_bucket': best_b, 'worst_bucket': worst_b, 'spread_R': spread,
                            'buckets_ci_above_zero': pos, 'reliable_buckets': int(d.reliable.sum())}
print(json.dumps(VIX_SUMMARY, indent=1, default=str))
mem('VIX regimes')

{
 "rules": {
  "best_bucket": ">30",
  "worst_bucket": "<15",
  "spread_R": 0.6197051420494621,
  "buckets_ci_above_zero": [],
  "reliable_buckets": 4
 },
 "ml": {
  "best_bucket": ">30",
  "worst_bucket": "<15",
  "spread_R": 0.46116615380403014,
  "buckets_ci_above_zero": [],
  "reliable_buckets": 4
 }
}
[mem] VIX regimes                                rss=  1088.8 MB   peak=  1698.5 MB


### 11.1 Verdict: where the edge thrives or decays
<!-- audit:claims -->
* **Decays in calm markets.** In the primary split, rules-only OOS trades on sessions after a VIX close below 15 lost −0.360R per trade (70 trades, 95% CI −0.659R to −0.054R). That is the only VIX-level bucket whose interval excludes zero. ML-filtered trades in the same bucket lost −0.331R.
* **Best in stressed markets, but not significantly.** With VIX above 30 the rules-only expectancy was +0.260R (62 trades, CI −0.164R to +0.694R). VIX 15–20 and 20–30 came in at −0.015R and −0.064R.
* **Mechanism (measured above, not assumed).** Costs are fixed in cents while stops scale with ATR, so cost per trade in R falls as volatility rises: 0.195R below VIX 15 versus 0.096R above 30. Gross expectancy also rises with VIX, from −0.165R to +0.356R. Both effects favour high-volatility sessions.
* **Not a usable filter.** No VIX bucket has an interval entirely above zero. Picking a VIX filter from this table would also be a choice made on OOS data, which this notebook does not allow. The pattern is descriptive only.

## 12 Macro events, news uncertainty & geopolitical risk (analysis only)

This section **does not change any rule, parameter or search result.** It tags the existing trade logs from §9 and asks where the edge thrives, decays, or shows no difference.

**Sources.**
* **FRED** (St. Louis Fed). The API key is read from the `FRED_API_KEY` environment variable and is never written to the notebook.
  * Release calendars come from FRED's `release/dates` endpoint, for CPI, Employment Situation (NFP), Personal Income & Outlays (PCE), GDP, Advance Retail Sales and PPI. `fredapi` has no wrapper for release calendars, so the endpoint is called directly with the same key.
  * `USEPUINDXD`, the daily Economic Policy Uncertainty index (Baker, Bloom & Davis), via `fredapi`.
  * `EMVOVERALLEMV`, the Equity Market Volatility tracker, via `fredapi`. **It is monthly, not daily.** Each month's value is used only from the first day of the month after next (conservative publication lag).
* **FOMC statement dates** come from the Federal Reserve's own FOMC calendar pages (federalreserve.gov, **non-FRED**). FRED's "FOMC Press Release" release is updated almost daily with policy-rate data (thousands of dates), so it is not a meeting calendar and is not used. Scheduled statements get the era release time. Intermeeting actions (conference calls, unscheduled meetings, notation votes) tag the session but get no ±30-minute window unless their time is documented.
* **Geopolitical Risk (GPR) daily index**, Caldara & Iacoviello, from matteoiacoviello.com. **Non-FRED source.** The file is the current vintage, not point-in-time, so historical values may include revisions.

**Timing.** Events are aligned to their **release date and time (ET)**, never the observation period. A release on a non-trading day (e.g. the Sunday 2020-03-15 FOMC action, Good Friday NFP) tags the next session as pre-open news. CPI, NFP, PCE, GDP, retail sales and PPI are 08:30 ET releases, before the open, so they tag the whole session as an event day. Their ±30-minute window (08:00–09:00) has no RTH bars and is reported as structurally empty. FOMC statements: 14:15 ET until 2013-03-19 (12:30 ET on the 2011–2012 press-conference days), 14:00 ET from the March 2013 meeting on (Federal Reserve press release, 13 Mar 2013). Known unscheduled intermeeting actions in the sample have their actual announcement times overridden from the public record. Uncertainty and GPR use the **prior day's value**, as specified. **Caveat:** EPU and GPR are themselves published with a lag of a day or more, and the downloaded GPR file is the current vintage, not point-in-time. So the prior-day alignment is optimistic by up to a day, and a 2-session-lag sensitivity is reported alongside. EMV is monthly, and its ALFRED first-vintage dates are back-fills, not publication dates, so month *m* is used only from the first day of month *m*+2. Terciles are cut **on each split's in-sample days only**.

**Memory.** Only daily/event series are loaded. They are merged onto the trade logs, never onto 1-minute bars.

In [37]:
EXT = CACHE / 'external'
EXT.mkdir(exist_ok=True)
FRED_KEY = os.environ.get('FRED_API_KEY')
EVENT_AVAILABILITY = {}

RELEASE_NAMES = {   # event type -> FRED release name (matched case-insensitively), ET release time in minutes
    'CPI': ('Consumer Price Index', 8 * 60 + 30),
    'NFP': ('Employment Situation', 8 * 60 + 30),
    'FOMC': ('FOMC Press Release', None),                # time by era, see fomc_time()
    'PCE': ('Personal Income and Outlays', 8 * 60 + 30),
    'GDP': ('Gross Domestic Product', 8 * 60 + 30),
    'RETAIL': ('Advance Monthly Sales for Retail and Food Services', 8 * 60 + 30),
    'PPI': ('Producer Price Index', 8 * 60 + 30),
}
FOMC_PRESSCONF_1230 = {20110427, 20110622, 20111102, 20120125, 20120425, 20120620, 20120913, 20121212}
FOMC_UNSCHEDULED_TIME = {20080122: 8 * 60 + 20, 20081008: 7 * 60, 20200303: 10 * 60, 20200315: 17 * 60}

def fomc_time(d):
    if d in FOMC_UNSCHEDULED_TIME:
        return FOMC_UNSCHEDULED_TIME[d]
    if d in FOMC_PRESSCONF_1230:
        return 12 * 60 + 30
    return 14 * 60 + 15 if d < 20130320 else 14 * 60

def _redact(msg):
    """Never let the API key reach an output cell (request URLs in exception messages contain it)."""
    msg = str(msg)
    return msg.replace(FRED_KEY, '***') if FRED_KEY else msg

def _fred_get(path, **params):
    """FRED REST call with retry on rate limits. Errors carry FRED's own message and status, never the request URL."""
    import requests
    for attempt in range(4):
        r = requests.get(f'https://api.stlouisfed.org/fred/{path}', params={**params, 'api_key': FRED_KEY, 'file_type': 'json'}, timeout=30)
        if r.status_code == 429 and attempt < 3:
            time.sleep(2 ** attempt)
            continue
        if r.status_code != 200:
            try:
                why = r.json().get('error_message', '')
            except ValueError:
                why = r.text[:200]
            raise RuntimeError(_redact(f'FRED API {path}: HTTP {r.status_code} {why}'))
        return r.json()

def fetch_event_calendar():
    path = EXT / 'fred_release_calendar.parquet'
    if path.exists() and not REBUILD_CACHE:
        cal = pd.read_parquet(path)
        for ev, n in cal.groupby('event', observed=True).size().items():
            EVENT_AVAILABILITY[ev] = f'FRED release calendar: {n} dates (cached)'
        return cal
    if not FRED_KEY:
        print('FRED_API_KEY not set -> release calendars unavailable; event-day analysis skipped.')
        return pd.DataFrame(columns=['date', 'event', 'release_min'])
    releases = []
    offset = 0
    try:
        while True:
            js = _fred_get('releases', limit=1000, offset=offset)
            releases += js['releases']
            if len(js['releases']) < 1000:
                break
            offset += 1000
    except Exception as e:
        msg = _redact(e)
        print(f'FRED release list unavailable: {msg}')
        for ev in RELEASE_NAMES:
            EVENT_AVAILABILITY[ev] = f'unavailable: {msg}'[:200]
        return pd.DataFrame(columns=['date', 'event', 'release_min'])
    by_name = {r['name'].lower(): r['id'] for r in releases}
    rows = []
    for ev, (name, tmin) in RELEASE_NAMES.items():
        rid = by_name.get(name.lower())
        if rid is None:
            EVENT_AVAILABILITY[ev] = f'unavailable: no FRED release named "{name}"'
            continue
        try:
            js = _fred_get('release/dates', release_id=rid, realtime_start='2002-01-01', realtime_end='9999-12-31',
                           include_release_dates_with_no_data='false', limit=10000, sort_order='asc')
        except Exception as e:
            EVENT_AVAILABILITY[ev] = f'unavailable: {_redact(e)}'[:200]
            continue
        ds = [int(x['date'].replace('-', '')) for x in js.get('release_dates', [])]
        EVENT_AVAILABILITY[ev] = f'FRED release_id={rid}, {len(ds)} dates'
        for d in ds:
            rows.append({'date': d, 'event': ev, 'release_min': fomc_time(d) if ev == 'FOMC' else tmin})
    cal = pd.DataFrame(rows)
    if len(cal):
        cal['event'] = cal['event'].astype('category')
        cal.to_parquet(path, index=False)
    return cal

def fetch_fred_series(series_id, realtime=False):
    path = EXT / f'fred_{series_id}{"_rt" if realtime else ""}.parquet'
    if path.exists() and not REBUILD_CACHE:
        df = pd.read_parquet(path)
        EVENT_AVAILABILITY[series_id] = f'FRED {series_id}: {len(df)} obs (cached)'
        return df
    if not FRED_KEY:
        EVENT_AVAILABILITY[series_id] = 'unavailable: FRED_API_KEY not set'
        return None
    try:
        from fredapi import Fred
        fred = Fred(api_key=FRED_KEY)
        if realtime:   # first-release vintages: when each observation became public
            s = fred.get_series_all_releases(series_id)
            s = s.sort_values('realtime_start').drop_duplicates('date', keep='first')
            df = pd.DataFrame({'obs_date': pd.to_datetime(s['date']), 'avail_date': pd.to_datetime(s['realtime_start']),
                               'value': pd.to_numeric(s['value'], errors='coerce')})
        else:
            s = fred.get_series(series_id)
            df = pd.DataFrame({'obs_date': pd.to_datetime(s.index), 'value': s.values.astype(float)})
    except Exception as e:
        EVENT_AVAILABILITY[series_id] = f'unavailable: {_redact(repr(e))}'[:200]
        return None
    EVENT_AVAILABILITY[series_id] = f'FRED {series_id}: {len(df)} obs'
    df.to_parquet(path, index=False)
    return df

def fetch_gpr_daily():
    path = EXT / 'gpr_daily.parquet'
    if path.exists() and not REBUILD_CACHE:
        df = pd.read_parquet(path)
        EVENT_AVAILABILITY['GPR'] = f'Caldara-Iacoviello GPRD (non-FRED): {len(df)} obs, {df.obs_date.min().date()} -> {df.obs_date.max().date()} (cached)'
        return df
    import requests
    url = 'https://www.matteoiacoviello.com/gpr_files/data_gpr_daily_recent.dta'
    try:
        raw = EXT / 'data_gpr_daily_recent.dta'
        r = requests.get(url, timeout=60)
        r.raise_for_status()
        raw.write_bytes(r.content)
        g = pd.read_stata(raw)
    except Exception as e:
        EVENT_AVAILABILITY['GPR'] = f'unavailable: {e!r}'[:200]
        return None
    dcol = next(c for c in g.columns if c.lower() in ('date', 'day'))
    vcol = next(c for c in g.columns if c.upper() == 'GPRD')
    df = pd.DataFrame({'obs_date': pd.to_datetime(g[dcol]), 'value': pd.to_numeric(g[vcol], errors='coerce')}).dropna()
    EVENT_AVAILABILITY['GPR'] = f'Caldara-Iacoviello GPRD (non-FRED): {len(df)} obs, {df.obs_date.min().date()} -> {df.obs_date.max().date()}'
    df.to_parquet(path, index=False)
    return df

def fetch_fomc_statements():
    """FOMC statement dates from the Federal Reserve's own calendar pages (NON-FRED source).
    FRED's 'FOMC Press Release' (release 101) is updated almost daily with policy-rate data, so its dates are not meeting dates.
    Rows marked 'Conference Call', '(unscheduled)' or '(notation vote)' on the Fed pages are intermeeting actions."""
    path = EXT / 'fed_fomc_statements.parquet'
    if path.exists() and not REBUILD_CACHE:
        df = pd.read_parquet(path)
        EVENT_AVAILABILITY['FOMC'] = f'Federal Reserve FOMC calendars (non-FRED): {len(df)} statements, {int((df.kind == "scheduled").sum())} scheduled (cached)'
        return df
    import requests
    pages = ['fomccalendars'] + [f'fomchistorical{y}' for y in range(2002, pd.Timestamp.today().year - 4)]
    pat = re.compile(r'(?:boarddocs/press/(?:monetary|general)/\d{4}/(\d{8})/|monetary/?(\d{8})[ab]\.htm)')
    rows = {}
    try:
        for pg in pages:
            r = requests.get(f'https://www.federalreserve.gov/monetarypolicy/{pg}.htm', headers={'User-Agent': 'Mozilla/5.0 (research)'}, timeout=60)
            if r.status_code != 200:
                continue
            html = r.text
            for m in pat.finditer(html):
                d = int(m.group(1) or m.group(2))
                if d < 20020101:
                    continue
                ctx = re.sub(r'<[^>]+>', ' ', html[max(0, m.start() - 400):m.start()])[-200:].lower()
                kind = 'intermeeting' if any(w in ctx for w in ('conference call', 'unscheduled', 'notation vote')) else 'scheduled'
                rows[d] = 'intermeeting' if rows.get(d) == 'intermeeting' else kind
    except Exception as e:
        EVENT_AVAILABILITY['FOMC'] = f'unavailable: Fed calendar download failed ({type(e).__name__})'
        return pd.DataFrame(columns=['date', 'kind'])
    df = pd.DataFrame(sorted(rows.items()), columns=['date', 'kind'])
    EVENT_AVAILABILITY['FOMC'] = f'Federal Reserve FOMC calendars (non-FRED): {len(df)} statements, {int((df.kind == "scheduled").sum())} scheduled'
    if len(df):
        df.to_parquet(path, index=False)
    return df

EVENT_CAL = fetch_event_calendar()
FOMC_STATEMENTS = fetch_fomc_statements()
if len(EVENT_CAL):
    n_fred_fomc = int((EVENT_CAL.event == 'FOMC').sum())
    EVENT_CAL = EVENT_CAL[EVENT_CAL.event != 'FOMC']
    print(f"FRED 'FOMC Press Release' has {n_fred_fomc} dates (near-daily policy-rate updates), so it is NOT used as a meeting calendar")
if len(FOMC_STATEMENTS):
    fo = FOMC_STATEMENTS.copy()
    # scheduled statements: era release time; intermeeting actions: known time if documented, otherwise day-level tag only
    fo['release_min'] = [fomc_time(d) if k == 'scheduled' else FOMC_UNSCHEDULED_TIME.get(d, np.nan) for d, k in zip(fo.date, fo.kind)]
    fo['event'] = 'FOMC'
    EVENT_CAL = pd.concat([EVENT_CAL, fo[['date', 'event', 'release_min']]], ignore_index=True)
if len(EVENT_CAL):
    EVENT_CAL['event'] = EVENT_CAL['event'].astype(str)
EPU = fetch_fred_series('USEPUINDXD')
EMV = fetch_fred_series('EMVOVERALLEMV')
if EMV is not None:
    # ALFRED's first vintage back-fills decades of history on one date, so its realtime_start is not a publication date.
    # Conservative fixed lag instead: month m is used only from the first day of month m+2.
    EMV['avail_date'] = EMV.obs_date.values.astype('datetime64[M]') + np.timedelta64(2, 'M')
GPR = fetch_gpr_daily()
print(json.dumps(EVENT_AVAILABILITY, indent=1))
if len(EVENT_CAL):
    display(EVENT_CAL.groupby('event', observed=True).agg(n=('date', 'size'), first=('date', 'min'), last=('date', 'max')))
mem('event/uncertainty data')

FRED 'FOMC Press Release' has 3822 dates (near-daily policy-rate updates), so it is NOT used as a meeting calendar
{
 "CPI": "FRED release calendar: 315 dates (cached)",
 "FOMC": "Federal Reserve FOMC calendars (non-FRED): 211 statements, 198 scheduled (cached)",
 "GDP": "FRED release calendar: 334 dates (cached)",
 "NFP": "FRED release calendar: 304 dates (cached)",
 "PCE": "FRED release calendar: 305 dates (cached)",
 "PPI": "FRED release calendar: 314 dates (cached)",
 "RETAIL": "FRED release calendar: 319 dates (cached)",
 "USEPUINDXD": "FRED USEPUINDXD: 15270 obs (cached)",
 "EMVOVERALLEMV": "FRED EMVOVERALLEMV: 501 obs (cached)",
 "GPR": "Caldara-Iacoviello GPRD (non-FRED): 15267 obs, 1985-01-01 -> 2026-10-19 (cached)"
}


,n,first,last
event,,,
CPI,315,20020116,20261014
FOMC,211,20020130,20260916
GDP,334,20020130,20260930
NFP,304,20020104,20261002
PCE,305,20020131,20260930
PPI,314,20020111,20261015
RETAIL,319,20020115,20261015


[mem] event/uncertainty data                     rss=  1091.8 MB   peak=  1698.5 MB


In [38]:
def _prior_value(days, obs_dates, values, avail_dates=None, lag_sessions=1):
    """For each session date (int yyyymmdd): the latest observation dated strictly before the session that is
    `lag_sessions` back (1 = the prior day's value). With avail_dates, the value must have been available before then."""
    ref = pd.Series(days).shift(lag_sessions - 1).bfill().values if lag_sessions > 1 else days
    sess = pd.DataFrame({'date': days, 'dt': pd.to_datetime(pd.Series(ref).astype(int).astype(str))}).sort_values('dt')
    src = pd.DataFrame({'key': pd.to_datetime(np.asarray(avail_dates if avail_dates is not None else obs_dates)).astype('datetime64[ns]'),
                        'obs': pd.to_datetime(np.asarray(obs_dates)).astype('datetime64[ns]'), 'value': np.asarray(values, dtype=float)})
    sess['dt'] = sess['dt'].astype('datetime64[ns]')
    src = src.dropna().sort_values(['key', 'obs'], kind='stable').drop_duplicates('key', keep='last')
    out = pd.merge_asof(sess, src, left_on='dt', right_on='key', direction='backward', allow_exact_matches=False)
    return out.set_index('date')['value'].reindex(days).values

DAY_CONTEXT = pd.DataFrame({'date': CALENDAR})
DAY_CONTEXT['epu_prev'] = _prior_value(CALENDAR, EPU.obs_date, EPU.value) if EPU is not None else np.nan
DAY_CONTEXT['emv_avail'] = _prior_value(CALENDAR, EMV.obs_date, EMV.value, EMV.avail_date) if EMV is not None else np.nan
DAY_CONTEXT['gpr_prev'] = _prior_value(CALENDAR, GPR.obs_date, GPR.value) if GPR is not None else np.nan
# sensitivity: EPU and GPR are published with a lag of a day or more and are not point-in-time -> also a 2-session lag
DAY_CONTEXT['epu_lag2'] = _prior_value(CALENDAR, EPU.obs_date, EPU.value, lag_sessions=2) if EPU is not None else np.nan
DAY_CONTEXT['gpr_lag2'] = _prior_value(CALENDAR, GPR.obs_date, GPR.value, lag_sessions=2) if GPR is not None else np.nan
if len(EVENT_CAL):
    # a release on a non-trading day (Sunday FOMC 2020-03-15, Good Friday NFP) tags the NEXT session, as pre-open news
    # releases outside the bar data are dropped; a shift is allowed only to a session at most 4 calendar days later
    EVENT_CAL = EVENT_CAL[(EVENT_CAL.date >= CALENDAR[0]) & (EVENT_CAL.date <= CALENDAR[-1])].copy()
    nxt = np.searchsorted(CALENDAR, EVENT_CAL.date.values)
    ok_ = nxt < len(CALENDAR)
    gap = (pd.to_datetime(pd.Series(CALENDAR[np.minimum(nxt, len(CALENDAR) - 1)]).astype(str)).values
           - pd.to_datetime(EVENT_CAL.date.astype(str)).values) / np.timedelta64(1, 'D')
    ok_ &= gap <= 4
    EVENT_CAL = EVENT_CAL[ok_].copy()
    sess = CALENDAR[nxt[ok_]]
    shifted = sess != EVENT_CAL.date.values
    EVENT_CAL['release_min'] = np.where(shifted, OPEN_MIN - 1, EVENT_CAL['release_min'].astype(float))
    EVENT_CAL['session'] = sess
    print(f'{int(shifted.sum())} releases fell on non-trading days and tag the next session as pre-open news')
for ev in RELEASE_NAMES:
    DAY_CONTEXT[f'ev_{ev}'] = DAY_CONTEXT.date.isin(EVENT_CAL.loc[EVENT_CAL.event == ev, 'session']) if len(EVENT_CAL) else False
DAY_CONTEXT['event_day'] = DAY_CONTEXT[[f'ev_{e}' for e in RELEASE_NAMES]].any(axis=1)
DAY_CONTEXT = DAY_CONTEXT.set_index('date')
FOMC_TIME = (EVENT_CAL[(EVENT_CAL.event == 'FOMC') & (EVENT_CAL.release_min >= OPEN_MIN)].groupby('session')['release_min'].max().to_dict()
             if len(EVENT_CAL) else {})       # only intraday FOMC releases get a ±30 min window

def tag_trades(t):
    """Add event/uncertainty context to a trade log (daily merge; no 1m bars touched)."""
    if t is None or len(t) == 0:
        return t
    t = t.copy()
    ctx = DAY_CONTEXT.reindex(t['date'].values)
    for c in ctx.columns:
        t[c] = ctx[c].values
    rel = t['date'].map(FOMC_TIME)
    # overlap of [entry bar open, exit bar close] with [release - 30, release + 30]
    t['fomc_window'] = rel.notna() & (t['entry_min'] < rel + 30) & (t['exit_min'] + 1 > rel - 30)
    t['fomc_after'] = rel.notna() & (t['entry_min'] >= rel)
    return t

def is_terciles(values_is):
    v = pd.Series(values_is).dropna()
    return (np.nan, np.nan) if len(v) < 30 else tuple(np.quantile(v, [1 / 3, 2 / 3]))

def tercile_label(x, cuts):
    lo, hi = cuts
    return np.where(np.isnan(x), 'n/a', np.where(x <= lo, 'T1 low', np.where(x <= hi, 'T2 mid', 'T3 high')))

def event_breakdown(split, variant):
    """Bucket table for OOS trades of one split/variant (1x costs)."""
    t = tag_trades(EVAL[split]['oos_port'][1.0][variant])
    if t is None or len(t) == 0:
        return pd.DataFrame()
    oos_days = EVAL[split]['oos_cal']
    is_days = EVAL[split]['is_cal']
    dc = DAY_CONTEXT.reindex(oos_days)
    rows = []
    if not len(EVENT_CAL):
        for dim in ['event', 'intraday']:
            rows.append({'dimension': dim, 'bucket': 'unavailable', 'trades': 0, 'reliable': False,
                         'note': 'FRED release calendars unavailable (FRED_API_KEY not set or API error) - skipped, not guessed'})
    else:
        rows += _event_rows(t, dc)
    rows += _tercile_rows(t, dc, is_days)
    df = pd.DataFrame(rows)
    df.insert(0, 'variant', variant)
    df.insert(0, 'split', split)
    return df

def _event_rows(t, dc):
    rows = []
    # event day vs not, and by type
    rows.append({'dimension': 'event', **bucket_stats(t[~t.event_day], dc.index[~dc.event_day.values], 'non-event day')})
    rows.append({'dimension': 'event', **bucket_stats(t[t.event_day], dc.index[dc.event_day.values], 'any event day')})
    for ev in RELEASE_NAMES:
        m = t[f'ev_{ev}'].values.astype(bool)
        rows.append({'dimension': 'event', **bucket_stats(t[m], dc.index[dc[f'ev_{ev}'].values.astype(bool)], ev)})
    fomc_days = dc.index[dc['ev_FOMC'].values.astype(bool)]
    rows.append({'dimension': 'intraday', **bucket_stats(t[t.fomc_window], fomc_days, 'FOMC ±30min')})
    rows.append({'dimension': 'intraday', **bucket_stats(t[t.ev_FOMC.astype(bool) & ~t.fomc_window], fomc_days, 'FOMC day, outside ±30min')})
    rows.append({'dimension': 'intraday', **bucket_stats(t[t.fomc_after], fomc_days, 'FOMC day, after release')})
    rows.append({'dimension': 'intraday', 'bucket': '08:30 releases ±30min', 'trades': 0, 'reliable': False,
                 'note': 'structurally empty: window 08:00-09:00 is before the RTH open'})
    return rows

def _tercile_rows(t, dc, is_days):
    rows = []
    # uncertainty terciles: cut on IS days, applied to OOS
    for col, dim in [('epu_prev', 'EPU tercile'), ('emv_avail', 'EMV tercile'), ('gpr_prev', 'GPR tercile'),
                     ('epu_lag2', 'EPU tercile (lag 2)'), ('gpr_lag2', 'GPR tercile (lag 2)')]:
        cuts = is_terciles(DAY_CONTEXT.reindex(is_days)[col].values)
        if np.isnan(cuts[0]):
            rows.append({'dimension': dim, 'bucket': 'unavailable', 'trades': 0, 'reliable': False,
                         'note': EVENT_AVAILABILITY.get({'epu_prev': 'USEPUINDXD', 'emv_avail': 'EMVOVERALLEMV', 'gpr_prev': 'GPR',
                                                         'epu_lag2': 'USEPUINDXD', 'gpr_lag2': 'GPR'}[col], 'insufficient IS coverage')})
            continue
        lab_t = tercile_label(t[col].values.astype(float), cuts)
        lab_d = tercile_label(dc[col].values.astype(float), cuts)
        for b in ['T1 low', 'T2 mid', 'T3 high']:
            rows.append({'dimension': dim, **bucket_stats(t[lab_t == b], dc.index[lab_d == b], b),
                         'cut_lo': cuts[0], 'cut_hi': cuts[1]})
    return rows

17 releases fell on non-trading days and tag the next session as pre-open news


### 12.1 Performance by event type, uncertainty tercile and GPR tercile (OOS, 1× costs)

In [39]:
ev_tables = [event_breakdown(name, var) for name in EVAL for var in ['rules', 'ml'] if EVAL[name]['oos_port'][1.0][var] is not None]
EVENT_TABLE = pd.concat([x for x in ev_tables if len(x)], ignore_index=True) if any(len(x) for x in ev_tables) else pd.DataFrame()
if len(EVENT_TABLE):
    EVENT_TABLE.to_parquet(DIRS['results'] / 'event_regimes.parquet', index=False)
    prim = EVENT_TABLE[EVENT_TABLE.split == PRIMARY_SPLIT]
    for dim, xlab, ttl in [('event', 'release type (event = release on that session date)', 'by macro event type'),
                           ('intraday', 'FOMC intraday window (release time ET by era)', 'around FOMC statements'),
                           ('EPU tercile', 'prior-day EPU tercile (cut on IS days)', 'by Economic Policy Uncertainty tercile'),
                           ('EMV tercile', 'latest published EMV tercile (cut on IS days)', 'by Equity Market Volatility tracker tercile'),
                           ('GPR tercile', 'prior-day GPR tercile (cut on IS days; non-FRED)', 'by Geopolitical Risk tercile'),
                           ('EPU tercile (lag 2)', 'EPU tercile, 2-session lag (sensitivity)', 'by EPU tercile, 2-session lag'),
                           ('GPR tercile (lag 2)', 'GPR tercile, 2-session lag (sensitivity)', 'by GPR tercile, 2-session lag')]:
        d = prim[(prim.dimension == dim) & (prim.trades > 0)]
        if len(d):
            bucket_bar_chart(d, f'{PRIMARY_SPLIT} OOS expectancy {ttl}', xlab).show()
        else:
            print(f'{dim}: no data (see availability above)')
    # consistency across splits: in how many splits is each bucket's (reliable) OOS expectancy > 0?
    cons = (EVENT_TABLE[EVENT_TABLE.variant == 'rules'].assign(pos=lambda x: (x.expectancy_R > 0) & x.reliable.astype(bool))
            .groupby(['dimension', 'bucket'], sort=False).agg(splits_reliable=('reliable', 'sum'), splits_positive=('pos', 'sum')))
    keep_cols = ['trades', 'win_rate', 'expectancy_R', 'exp_ci_lo', 'exp_ci_hi', 'profit_factor', 'sharpe', 'max_dd', 'reliable']
    EVENT_SUMMARY = (prim.pivot_table(index=['dimension', 'bucket'], columns='variant', values=keep_cols, aggfunc='first', sort=False)
                     .swaplevel(0, 1, axis=1).sort_index(axis=1, level=0))
    cons.columns = pd.MultiIndex.from_product([['all splits (rules)'], cons.columns])
    EVENT_SUMMARY = EVENT_SUMMARY.join(cons)
    EVENT_SUMMARY['flag'] = np.where(EVENT_SUMMARY[('rules', 'trades')].fillna(0) < 30, 'UNRELIABLE (<30 trades)', '')
    print(f'Summary table: {PRIMARY_SPLIT} OOS by bucket (rules / ml), plus cross-split consistency of the rules-only variant')
    display(EVENT_SUMMARY.round(3))
mem('event analysis')

Summary table: HALF OOS by bucket (rules / ml), plus cross-split consistency of the rules-only variant


ml                                                                                 rules                                                       \
                                             exp_ci_hi exp_ci_lo expectancy_R max_dd profit_factor reliable sharpe trades win_rate exp_ci_hi exp_ci_lo expectancy_R max_dd profit_factor reliable   
dimension           bucket                                                                                                                                                                          
event               non-event day                0.049    -0.247       -0.102 -0.197         0.854     True -0.493    377    0.350     0.014    -0.263       -0.129 -0.224         0.821     True   
                    any event day                0.454    -0.002        0.221 -0.067         1.395     True  0.999    187    0.422     0.330    -0.074        0.129 -0.068         1.219     True   
                    CPI                          0.514    -0.664       -0.099 -0.022         0.855    False -0.589     23    0.348     0.529    -0.568       -0.040 -0.019         0.937    False   
                    NFP                          0.681    -0.348        0.165 -0.018         1.280     True  0.903     34    0.441     0.551    -0.428        0.030 -0.034         1.047     True   
                    FOMC                         0.484    -0.720       -0.161 -0.018         0.773    False -0.834     23    0.304     0.404    -0.662       -0.169 -0.020         0.761    False   
                    PCE                          1.058    -0.043        0.492 -0.018         2.159     True  2.327     34    0.441     0.724    -0.247        0.246 -0.018         1.462     True   
                    GDP                          0.879    -0.421        0.210 -0.021         1.328    False  0.586     29    0.379     0.830    -0.387        0.206 -0.021         1.323     True   
                    RETAIL                       0.918    -0.131        0.366 -0.011         1.951    False  1.572     26    0.538     0.647    -0.308        0.150 -0.016         1.304     True   
                    PPI                          0.766    -0.232        0.265 -0.016         1.474     True  1.474     36    0.472     0.689    -0.215        0.241 -0.021         1.440     True   
intraday            FOMC ±30min                  1.947    -1.215       -0.145 -0.006         0.838    False -0.558      4    0.250     1.947    -1.215       -0.145 -0.006         0.838    False   
                    FOMC day, outside ±30min     0.491    -0.744       -0.164 -0.016         0.755    False -0.637     19    0.316     0.449    -0.693       -0.174 -0.018         0.742    False   
                    FOMC day, after release     -1.149    -1.149       -1.149 -0.005         0.000    False -1.572      1    0.000    -1.149    -1.149       -1.149 -0.005         0.000    False   
                    08:30 releases ±30min          NaN       NaN          NaN    NaN           NaN    False    NaN      0      NaN       NaN       NaN          NaN    NaN           NaN    False   
EPU tercile         T1 low                       0.505    -0.093        0.191 -0.048         1.325     True  0.788    109    0.422     0.442    -0.165        0.129 -0.054         1.211     True   
                    T2 mid                      -0.017    -0.414       -0.227 -0.158         0.686     True -1.233    181    0.320    -0.032    -0.397       -0.219 -0.181         0.701     True   
                    T3 high                      0.281    -0.098        0.084 -0.065         1.134     True  0.239    274    0.391     0.170    -0.142        0.004 -0.090         1.006     True   
EMV tercile         T1 low                       0.182    -0.390       -0.102 -0.073         0.847     True -0.187     93    0.376     0.096    -0.430       -0.179 -0.092         0.744     True   
                    T2 mid                       0.306    -0.173        0.064 -0.077         1.097     True  0.0

[mem] event analysis                             rss=   614.2 MB   peak=  1698.5 MB


### 12.2 Event filters as hypotheses: chosen in-sample, confirmed out-of-sample
A candidate filter skips trades in one bucket, for example FOMC ±30 min, event days, or the top EPU tercile. For each split, the filter that most improves **IS** expectancy is selected; it must remove at least 30 IS trades. It then counts as a result only if it **also** improves OOS expectancy and the OOS difference between kept and removed trades has a 95% bootstrap CI above zero, with at least 30 OOS trades removed. Choosing the best of about 20 candidates in-sample is itself a multiple test, which is why the OOS confirmation is required.

In [40]:
def candidate_masks(t, is_days):
    m = {}
    if len(EVENT_CAL):
        m.update({'skip FOMC ±30min': t.fomc_window.values, 'skip FOMC after release': t.fomc_after.values,
                  'skip all event days': t.event_day.values.astype(bool)})
        for ev in RELEASE_NAMES:
            m[f'skip {ev} days'] = t[f'ev_{ev}'].values.astype(bool)
    for col, nm in [('epu_prev', 'EPU'), ('emv_avail', 'EMV'), ('gpr_prev', 'GPR')]:
        cuts = is_terciles(DAY_CONTEXT.reindex(is_days)[col].values)
        if np.isnan(cuts[0]):
            continue
        lab = tercile_label(t[col].values.astype(float), cuts)
        m[f'skip {nm} high tercile'] = lab == 'T3 high'
        m[f'skip {nm} low tercile'] = lab == 'T1 low'
    return m

filt_rows = []
for name, E_ in EVAL.items():
    t_is, t_oos = tag_trades(E_['is_port'][1.0]['rules']), tag_trades(E_['oos_port'][1.0]['rules'])
    if t_is is None or len(t_is) < 60 or t_oos is None or len(t_oos) == 0 or (not len(EVENT_CAL) and EPU is None and GPR is None):
        continue
    m_is, m_oos = candidate_masks(t_is, E_['is_cal']), candidate_masks(t_oos, E_['is_cal'])
    base_is = t_is.R.mean()
    scored = [(t_is.R.values[~mk].mean() - base_is, k) for k, mk in m_is.items() if mk.sum() >= 30 and (~mk).sum() >= 30]
    if not scored:
        continue
    gain_is, best = max(scored)
    mk = m_oos[best]
    kept, removed = t_oos.R.values[~mk], t_oos.R.values[mk]
    gain_oos = kept.mean() - t_oos.R.mean() if len(kept) else np.nan
    if len(removed) >= 30 and len(kept) >= 30:
        diff = np.random.default_rng(SEED).choice(kept, (N_BOOT, len(kept))).mean(1) - np.random.default_rng(SEED + 1).choice(removed, (N_BOOT, len(removed))).mean(1)
        ci = np.quantile(diff, [0.025, 0.975])
    else:
        ci = (np.nan, np.nan)
    confirmed = bool(gain_is > 0 and gain_oos > 0 and np.isfinite(ci[0]) and ci[0] > 0)
    filt_rows.append({'split': name, 'IS-chosen filter': best, 'IS gain (R/trade)': gain_is, 'IS trades removed': int(m_is[best].sum()),
                      'OOS gain (R/trade)': gain_oos, 'OOS trades removed': int(mk.sum()), 'OOS kept-removed CI lo': ci[0],
                      'OOS kept-removed CI hi': ci[1], 'confirmed OOS': confirmed})
FILTER_TESTS = pd.DataFrame(filt_rows)
display(FILTER_TESTS.round(3) if len(FILTER_TESTS) else 'no filter tests possible (event/uncertainty data unavailable)')
EVENT_VERDICT_DATA = {
    'n_filter_tests': len(FILTER_TESTS), 'n_confirmed': int(FILTER_TESTS['confirmed OOS'].sum()) if len(FILTER_TESTS) else 0,
    'availability': EVENT_AVAILABILITY,
}
print(json.dumps(EVENT_VERDICT_DATA, indent=1, default=str))

,split,IS-chosen filter,IS gain (R/trade),IS trades removed,OOS gain (R/trade),OOS trades removed,OOS kept-removed CI lo,OOS kept-removed CI hi,confirmed OOS
0,IS3y,skip EPU low tercile,0.150,66,0.004,405,-0.164,0.239,False
1,IS4y,skip EPU high tercile,0.055,45,-0.047,1322,-0.292,0.158,False
2,IS5y,skip EMV low tercile,0.072,44,0.012,193,-0.088,0.269,False
3,HALF,skip GPR high tercile,0.058,105,0.084,365,-0.068,0.379,False
4,WF2008,skip EMV high tercile,0.087,50,-0.124,279,-0.479,0.218,False
5,WF2011,skip EPU low tercile,0.102,74,0.014,8,NaN,NaN,False
6,WF2014,skip EMV high tercile,0.063,207,0.058,22,NaN,NaN,False
7,WF2017,skip GPR low tercile,0.060,199,0.114,92,-0.279,0.760,False
8,WF2020,skip GPR high tercile,0.056,110,0.033,127,-0.315,0.501,False
9,WF2023,skip EPU high tercile,0.126,210,0.018,78,NaN,NaN,False


{
 "n_filter_tests": 10,
 "n_confirmed": 0,
 "availability": {
  "CPI": "FRED release calendar: 315 dates (cached)",
  "FOMC": "Federal Reserve FOMC calendars (non-FRED): 211 statements, 198 scheduled (cached)",
  "GDP": "FRED release calendar: 334 dates (cached)",
  "NFP": "FRED release calendar: 304 dates (cached)",
  "PCE": "FRED release calendar: 305 dates (cached)",
  "PPI": "FRED release calendar: 314 dates (cached)",
  "RETAIL": "FRED release calendar: 319 dates (cached)",
  "USEPUINDXD": "FRED USEPUINDXD: 15270 obs (cached)",
  "EMVOVERALLEMV": "FRED EMVOVERALLEMV: 501 obs (cached)",
  "GPR": "Caldara-Iacoviello GPRD (non-FRED): 15267 obs, 1985-01-01 -> 2026-10-19 (cached)"
 }
}


### 12.3 Verdict
<!-- audit:claims -->
* **Data.** FRED release calendars (CPI, NFP, PCE, GDP, retail sales, PPI), daily EPU and monthly EMV were all available. FOMC statement dates come from the Federal Reserve's own calendar pages (non-FRED), because FRED's FOMC release is a near-daily data feed, not a meeting calendar.
* **Release days vs. other days: suggestive, not significant.** In the primary split, rules-only OOS trades on release days earned +0.129R per trade (219 trades, 95% CI −0.074R to +0.330R), against −0.129R on other days (467 trades, CI −0.263R to +0.014R). The ML-filtered variant shows the same pattern, +0.221R on release days, but its interval also starts below zero (−0.002R).
* **By event type: no reliable difference.** Every per-type interval includes zero. CPI days (27 trades) and FOMC days (26 trades) have fewer than 30 trades each and are flagged unreliable.
* **FOMC ±30 minutes: too few trades to judge.** Only 4 primary-split OOS trades overlapped the ±30-minute window around a scheduled statement. The same window around the pre-open releases (CPI, NFP, PCE, GDP, retail sales, PPI) falls before the open, so it is structurally empty.
* **News uncertainty (EPU, EMV) and geopolitical risk (GPR): no consistent effect.** The middle EPU tercile is the only primary-split bucket whose interval lies below zero: −0.219R, CI −0.397R to −0.032R, and also below zero with a 2-session lag. It sits between a low tercile at +0.129R and a high tercile at +0.004R, so it is not a monotone uncertainty effect. With 22 reliable buckets tested, one or two such intervals are expected by chance. All EMV and GPR terciles include zero.
* **Filter hypotheses: none confirmed.** In all 10 testable splits and folds, the IS-selected filter (an EPU, EMV or GPR tercile skip, depending on the split) improved IS expectancy, but 0 of 10 were confirmed out of sample.
* **Verdict.** Release days are where the edge looks least bad, but the difference is not significant, and no IS-chosen event or uncertainty filter survives out of sample. Macro and news context do not make this strategy profitable.

## 13 Conclusions

### 13.1 Metrics: best rules-only vs best ML-filtered, every IS/OOS split
"Best" means the configuration and meta-labeler selected **in-sample** for that split. All numbers below are generated by this cell from the trade logs. Expectancy is net R per trade after costs. Sharpe and Sortino are annualized from daily portfolio returns, with zero-return days included. Max DD is on compounded equity at 0.5% risk per trade.

In [41]:
COLS_OUT = ['trades', 'win_rate', 'expectancy_R', 'profit_factor', 'sharpe', 'sortino', 'max_dd', 'avg_R_gross', 'avg_win_R',
            'avg_loss_R', 'trades_per_day']
ALL_METRICS = pd.concat([METRICS, WF_METRICS.assign(variant=WF_METRICS.variant.map({'rules': 'rules-only', 'ml': 'ML-filtered'}))], ignore_index=True)
ALL_METRICS['variant_kind'] = np.where(ALL_METRICS.variant.str.startswith('ML'), 'ML-filtered', 'rules-only')
CONCL = ALL_METRICS[ALL_METRICS.split.isin(list(SPLITS) + ['WF stitched'])].set_index(['split', 'variant_kind', 'period', 'costs'])[COLS_OUT]
CONCL = CONCL.reindex(sorted(CONCL.index, key=lambda k: (list(SPLITS) + ['WF stitched']).index(k[0]) * 100 + (k[1] != 'rules-only') * 10
                                + (k[2] == 'OOS') * 2 + (k[3] == '2x')))
CONCL.to_csv(DIRS['results'] / 'conclusions_metrics.csv')
fmt = {'win_rate': '{:.1%}', 'expectancy_R': '{:+.3f}', 'profit_factor': '{:.2f}', 'sharpe': '{:+.2f}', 'sortino': '{:+.2f}',
       'max_dd': '{:.1%}', 'avg_R_gross': '{:+.3f}', 'avg_win_R': '{:+.2f}', 'avg_loss_R': '{:+.2f}', 'trades_per_day': '{:.3f}'}
display(CONCL.style.format(fmt, na_rep='—').set_caption('IS and OOS metrics by split, variant and cost level'))

oos1 = CONCL.xs(('OOS', '1x'), level=('period', 'costs'))
HEADLINE = {
    'oos_rules_1x_expectancy': {s: float(oos1.loc[(s, 'rules-only'), 'expectancy_R']) for s in oos1.index.get_level_values(0).unique() if (s, 'rules-only') in oos1.index},
    'oos_ml_1x_expectancy': {s: float(oos1.loc[(s, 'ML-filtered'), 'expectancy_R']) for s in oos1.index.get_level_values(0).unique() if (s, 'ML-filtered') in oos1.index},
    'oos_rules_1x_sharpe': {s: float(oos1.loc[(s, 'rules-only'), 'sharpe']) for s in oos1.index.get_level_values(0).unique() if (s, 'rules-only') in oos1.index},
    'oos_ml_1x_sharpe': {s: float(oos1.loc[(s, 'ML-filtered'), 'sharpe']) for s in oos1.index.get_level_values(0).unique() if (s, 'ML-filtered') in oos1.index},
    'splits_rules_oos_positive_1x': int(sum(v > 0 for v in oos1.xs('rules-only', level=1)['expectancy_R'])),
    'splits_ml_oos_positive_1x': int(sum(v > 0 for v in oos1.xs('ML-filtered', level=1)['expectancy_R'])) if 'ML-filtered' in oos1.index.get_level_values(1) else 0,
    'n_splits': int(oos1.index.get_level_values(0).nunique()),
    'total_trials_logged': int(len(TRIAL_LOG)),
    'verdicts': {k: v['verdict'] for k, v in VERDICTS.items()},
    'peak_rss_mb': float(max(m['peak_mb'] for m in MEM_LOG)),
}
(DIRS['results'] / 'headline.json').write_text(json.dumps(HEADLINE, indent=1))
print(json.dumps(HEADLINE, indent=1))

{
 "oos_rules_1x_expectancy": {
  "IS1y": -0.15754136519505846,
  "IS2y": -0.1145090647680105,
  "IS3y": -0.1637554765971411,
  "IS4y": -0.11267723060348368,
  "IS5y": -0.09865831131857603,
  "HALF": -0.04672011828496687,
  "WF stitched": -0.06714073006429132
 },
 "oos_ml_1x_expectancy": {
  "IS3y": -0.1637554765971411,
  "HALF": 0.005027476190681749,
  "WF stitched": 0.10506497901549781
 },
 "oos_rules_1x_sharpe": {
  "IS1y": -1.1697342793227914,
  "IS2y": -0.8506348984631887,
  "IS3y": -0.9342279056169892,
  "IS4y": -0.3159324028457093,
  "IS5y": -0.6949383230958373,
  "HALF": -0.20296480610544618,
  "WF stitched": -0.2951723265988261
 },
 "oos_ml_1x_sharpe": {
  "IS3y": -0.9342279056169892,
  "HALF": -0.035050875308515354,
  "WF stitched": 0.2823452857561664
 },
 "splits_rules_oos_positive_1x": 0,
 "splits_ml_oos_positive_1x": 2,
 "n_splits": 7,
 "total_trials_logged": 2100,
 "verdicts": {
  "placebo_random_entry": "PASS",
  "placebo_time_shift": "FAIL",
  "placebo_shuffled_labels":

### 13.2 Overfitting & integrity verdicts

In [42]:
VERDICT_TABLE = pd.DataFrame(VERDICTS).T
display(VERDICT_TABLE)
(DIRS['results'] / 'verdicts.json').write_text(json.dumps(VERDICTS, indent=1))
WF_CI.to_csv(DIRS['results'] / 'wf_stitched_ci.csv', index=False)
WF_FOLD_CMP.to_csv(DIRS['results'] / 'wf_fold_ml_vs_rules.csv', index=False)
(DIRS['results'] / 'event_availability.json').write_text(json.dumps(EVENT_VERDICT_DATA, indent=1, default=str))
print(f"{(VERDICT_TABLE.verdict == 'PASS').sum()}/{len(VERDICT_TABLE)} checks passed")
mem_df = pd.DataFrame(MEM_LOG)
print(f"Peak resident memory over the whole run: {mem_df.peak_mb.max():.0f} MB (budget 4096 MB)")
fig = go.Figure(go.Scatter(x=np.arange(len(mem_df)), y=mem_df.rss_mb, mode='lines+markers', text=mem_df.stage, name='RSS',
                           hovertemplate='%{text}<br>RSS %{y:.0f} MB<extra></extra>'))
fig.add_trace(go.Scatter(x=np.arange(len(mem_df)), y=mem_df.peak_mb, mode='lines', name='peak RSS', line=dict(dash='dot'),
                         hovertemplate='peak %{y:.0f} MB<extra></extra>'))
fig.add_hline(y=4096, line_color='#d62728', annotation_text='4 GB budget')
fig.update_layout(title='Memory per stage (resident set size)', xaxis_title='stage # (hover for name)', yaxis_title='MB', height=360)
fig.show()

,verdict,detail
placebo_random_entry,PASS,"real OOS expectancy -0.047R, placebo 95th pct ..."
placebo_time_shift,FAIL,"real -0.047R vs shifted 95th pct +0.194R, p = ..."
placebo_shuffled_labels,FAIL,real ML OOS +0.005R vs shuffled 95th pct +0.08...
deflated_sharpe,FAIL,HALF DSR = 0.000 (N=250); 0/12 splits/folds ha...
reality_check_spa,FAIL,"HALF: Reality Check p = 1.000, SPA p = 0.962"
pbo_cscv,PASS,HALF PBO = 0.246; splits with PBO < 0.5: 9/12
parameter_plateau,PASS,selected IS Sharpe 0.29; 78% of 23 neighbours ...
monte_carlo,FAIL,"rules-only bootstrap Sharpe 90% CI [-0.70, 0.3..."
cross_sectional,FAIL,3/8 ETFs profitable OOS; sign-test p = 0.855
lookahead_audit,PASS,32/32 audit checks passed


4/10 checks passed
Peak resident memory over the whole run: 1698 MB (budget 4096 MB)


### 13.3 Conclusions
<!-- audit:claims -->
**Verdict: no tradable edge after costs.** The metrics table (§13.1) and the verdict table (§13.2) are generated from the trade logs. This text quotes them, and §13.4 checks every number in it.

1. **Rules-only fails out of sample everywhere.** At 1× costs, OOS expectancy is negative in all 6 IS/OOS splits, ranging from −0.164R to −0.047R per trade. The stitched anchored walk-forward is negative too: −0.067R over 1319 trades, 95% CI −0.160R to +0.025R. At 2× costs every split is worse, from −0.420R to −0.187R.
2. **There is some signal, but it is smaller than the costs.** In the primary split, OOS trades earned +0.093R gross and paid about 0.14R in costs. Real entries beat random entries with the same count, direction mix and bracket geometry (placebo median −0.339R, p = 0.004), so the sweep-and-reclaim timing carries some information. But the exact bar matters little (time-shifted signals, p = 0.223), and the information is worth less than the round trip of a 1-minute bracket on these ETFs.
3. **The in-sample winners are selection artefacts.** Across 2100 logged trials, every selected configuration has a Deflated Sharpe Ratio of at most 0.001 against a threshold of 0.95. In the primary split, the minimum backtest length implied by the trial count is 97 years, against 11.8 years of IS data, and Hansen's SPA p-value is 0.962. Short IS windows show the collapse most clearly: the IS1y winner had an IS Sharpe of 2.11 on 38 trades and an OOS Sharpe of −1.17. PBO is 0.246 in the primary split (3 of 12 splits and folds are at or above 0.5), and the parameter plateau check passes (78% of neighbours). So the optimizer finds a broad region rather than a spike, but that region is not profitable after costs.
4. **The ML meta-labeler does not reliably help.** Out-of-fold AUCs ranged from 0.42 to 0.55, where chance is 0.5. ML was not fit on IS1y, IS2y, IS4y or IS5y, which had fewer than 150 IS trades; on IS3y it kept 100% of trades, so its row equals rules-only. In the primary split, ML-filtered OOS expectancy was +0.005R against −0.047R for rules-only, and it does not beat meta-labelers trained on shuffled labels (p = 0.238). In the walk-forward, ML beat rules-only in 5 of 6 folds (sign test p = 0.109) and the stitched ML OOS expectancy is +0.105R. But its 95% CI runs from −0.044R to +0.259R and includes zero, much of the gain comes from one fold (WF2020: +0.452R against +0.020R), and at 2× costs it is −0.080R. An ML improvement is **not established**.
5. **Not cross-sectionally consistent.** Only 3 of the 8 ETFs that traded were profitable OOS in the primary split (sign test p = 0.855). SPY did best at +0.248R per trade, with a 95% CI of −0.005R to +0.505R. CPER never met the coverage rule, and IEF and SHY placed no primary-split OOS trades.
6. **The "best" rules change from split to split.** Of the 12 selected configurations, 9 are short-only. The selected trend timeframe takes 4 different values, and 5 different volume-confirmation recipes were chosen. This instability is what fitting noise looks like, not a stable Wyckoff effect.
7. **VIX regime (§11.1).** Losses concentrate after calm sessions: below VIX 15, expectancy is −0.360R, the only bucket whose interval lies below zero. Part of the reason is that fixed costs are a larger fraction of a tight 1-minute stop. Above 30, expectancy is +0.260R but not significant.
8. **Macro events, news uncertainty and geopolitical risk (§12).** Release days (CPI, NFP, PCE, GDP, retail sales, PPI, FOMC) were the least-bad sessions in the primary split, +0.129R per trade against −0.129R on other days, but both intervals include zero. Trades inside the FOMC ±30-minute window were too few to judge (4 trades). EPU, EMV and GPR terciles show no consistent effect: the one bucket with an interval below zero is the middle EPU tercile, which is not monotone and is consistent with chance among 22 buckets. None of the 10 IS-chosen event or uncertainty filters was confirmed out of sample. Event context does not rescue the strategy.
9. **Caveats.** All order-flow quantities are OHLCV **proxies**, not bid/ask, tape or footprint data. Volume after the 2022-03-07 source switch is IEX-only, with relative volume computed within each source segment. Spreads are assumed by causal liquidity tier rather than measured. Pre-switch prices were rescaled at the source seam. The universe is the 11 ETFs available, none delisted. Each caveat could move the true after-cost result somewhat in either direction, but none plausibly turns a loss of −0.047R to −0.164R per trade in every split into a robust profit.
10. **Bottom line.** Do not trade this system as specified. Any future version should be pre-registered, tested on a market where costs are small relative to a 1-minute bracket (or on longer brackets), and judged first against the placebo, DSR and walk-forward tests above. The notebook ran end to end within the 4 GB memory budget; the measured peak is in §13.2.

In [43]:
# Registry for the §13.4 claim audit: every number in the narrative cells marked audit:claims must equal one of these,
# formatted exactly as written (sign and precision). Expressions are evaluated on THIS run's outputs.
def _cc(split, variant, period, costs, col):
    return CONCL.loc[(split, variant, period, costs), col]

def _vx(var, bucket, col, dim='VIX level'):
    r = VIX_TABLE[(VIX_TABLE.split == PRIMARY_SPLIT) & (VIX_TABLE.variant == var) & (VIX_TABLE.dimension == dim) & (VIX_TABLE.bucket == bucket)]
    return r[col].iloc[0]

def _ev(dim, bucket, col, var='rules'):
    r = EVENT_TABLE[(EVENT_TABLE.split == PRIMARY_SPLIT) & (EVENT_TABLE.variant == var) & (EVENT_TABLE.dimension == dim) & (EVENT_TABLE.bucket == bucket)]
    return r[col].iloc[0]

_oos_rules_1x = [_cc(s, 'rules-only', 'OOS', '1x', 'expectancy_R') for s in SPLITS]
_oos_rules_2x = [_cc(s, 'rules-only', 'OOS', '2x', 'expectancy_R') for s in SPLITS]
_wf = WF_CI.set_index(['variant', 'costs'])
_sel = [v[0] for v in BEST.values() if v[0] is not None]
_half_rel = EVENT_TABLE[(EVENT_TABLE.split == PRIMARY_SPLIT) & (EVENT_TABLE.variant == 'rules') & (EVENT_TABLE.trades > 0) & EVENT_TABLE.reliable.astype(bool)]
CLAIMS = {
    # constants and design parameters quoted in the text
    'cost_1x': ('COST_MULTS[0]', '{:g}'), 'cost_2x': ('COST_MULTS[1]', '{:g}'), 'ci_level': ('0.95', '{:.0%}'),
    'dsr_threshold': ('0.95', '{}'), 'pbo_half': ('0.5', '{}'), 'n_splits': ('len(SPLITS)', '{}'),
    'n_split_folds': ('len(DSR_TABLE)', '{}'), 'fomc_window': ('30', '{}'), 'n_symbols': ('len(SYMBOLS)', '{}'),
    'mem_budget_gb': ('4096 // 1024', '{}'), 'min_ml_trades': ('MIN_ML_TRADES', '{}'),
    'vix_15': ('VIX_LEVELS[0][1]', '{:g}'), 'vix_20': ('VIX_LEVELS[1][1]', '{:g}'), 'vix_30': ('VIX_LEVELS[2][1]', '{:g}'),
    # 13.3 point 1
    'oos_rules_1x_min': ('min(_oos_rules_1x)', '{:+.3f}'), 'oos_rules_1x_max': ('max(_oos_rules_1x)', '{:+.3f}'),
    'oos_rules_2x_min': ('min(_oos_rules_2x)', '{:+.3f}'), 'oos_rules_2x_max': ('max(_oos_rules_2x)', '{:+.3f}'),
    'wf_rules_exp': ("_wf.loc[('rules', '1x'), 'expectancy_R']", '{:+.3f}'), 'wf_rules_n': ("_wf.loc[('rules', '1x'), 'trades']", '{:.0f}'),
    'wf_rules_lo': ("_wf.loc[('rules', '1x'), 'ci_lo']", '{:+.3f}'), 'wf_rules_hi': ("_wf.loc[('rules', '1x'), 'ci_hi']", '{:+.3f}'),
    # point 2
    'half_gross': ("_cc(PRIMARY_SPLIT, 'rules-only', 'OOS', '1x', 'avg_R_gross')", '{:+.3f}'),
    'half_cost': ("_cc(PRIMARY_SPLIT, 'rules-only', 'OOS', '1x', 'avg_R_gross') - _cc(PRIMARY_SPLIT, 'rules-only', 'OOS', '1x', 'expectancy_R')", '{:.2f}'),
    'placebo_median': ('PLACEBO.expectancy_R.median()', '{:+.3f}'), 'placebo_p': ('p_val', '{:.3f}'), 'shift_p': ('p_shift', '{:.3f}'),
    # point 3
    'n_trials_total': ('len(TRIAL_LOG)', '{}'), 'dsr_max': ("DSR_TABLE['DSR (N split)'].max()", '{:.3f}'),
    'minbtl_half': ("DSR_TABLE.loc[PRIMARY_SPLIT, 'MinBTL_years']", '{:.0f}'), 'is_years_half': ("DSR_TABLE.loc[PRIMARY_SPLIT, 'IS_years']", '{:.1f}'),
    'spa_half': ("SNOOP_TABLE.loc[PRIMARY_SPLIT, 'SPA p']", '{:.3f}'),
    'is1y_is_sharpe': ("_cc('IS1y', 'rules-only', 'IS', '1x', 'sharpe')", '{:.2f}'), 'is1y_is_trades': ("_cc('IS1y', 'rules-only', 'IS', '1x', 'trades')", '{:.0f}'),
    'is1y_oos_sharpe': ("_cc('IS1y', 'rules-only', 'OOS', '1x', 'sharpe')", '{:+.2f}'),
    'pbo_primary': ("PBO_TABLE.loc[PRIMARY_SPLIT, 'PBO']", '{:.3f}'), 'pbo_n_bad': ('int((PBO_TABLE.PBO >= 0.5).sum())', '{}'),
    'plateau_frac': ('frac', '{:.0%}'),
    # point 4
    'auc_min': ('ML_REPORT.oof_auc.min()', '{:.2f}'), 'auc_max': ('ML_REPORT.oof_auc.max()', '{:.2f}'),
    'is3y_keep': ("ML_REPORT.loc[('IS3y', EVAL['IS3y']['meta']['chosen']), 'keep_frac']", '{:.0%}'),
    'half_ml_exp': ("_cc(PRIMARY_SPLIT, 'ML-filtered', 'OOS', '1x', 'expectancy_R')", '{:+.3f}'),
    'half_rules_exp': ("_cc(PRIMARY_SPLIT, 'rules-only', 'OOS', '1x', 'expectancy_R')", '{:+.3f}'),
    'shuffle_p': ('p_ml', '{:.3f}'), 'wf_ml_wins': ('WF_ML_WINS', '{}'), 'wf_folds': ('len(WF_FOLD_CMP)', '{}'),
    'wf_sign_p': ('WF_ML_SIGN_P', '{:.3f}'), 'wf_ml_exp': ("_wf.loc[('ml', '1x'), 'expectancy_R']", '{:+.3f}'),
    'wf_ml_lo': ("_wf.loc[('ml', '1x'), 'ci_lo']", '{:+.3f}'), 'wf_ml_hi': ("_wf.loc[('ml', '1x'), 'ci_hi']", '{:+.3f}'),
    'wf2020_ml': ("WF_FOLD_CMP.set_index('fold').loc['WF2020', 'ml_exp']", '{:+.3f}'),
    'wf2020_rules': ("WF_FOLD_CMP.set_index('fold').loc['WF2020', 'rules_exp']", '{:+.3f}'),
    'wf_ml_2x': ("_wf.loc[('ml', '2x'), 'expectancy_R']", '{:+.3f}'),
    # point 5
    'xs_pos': ('n_pos', '{}'), 'xs_tot': ('n_tot', '{}'), 'xs_p': ('p_sign', '{:.3f}'),
    'spy_exp': ("XS.loc['SPY', 'expectancy_R']", '{:+.3f}'), 'spy_lo': ("XS.loc['SPY', 'ci_lo']", '{:+.3f}'), 'spy_hi': ("XS.loc['SPY', 'ci_hi']", '{:+.3f}'),
    # point 6
    'n_selected': ('len(_sel)', '{}'), 'n_short_only': ("sum(p['side'] == 2 for p in _sel)", '{}'),
    'n_trend_tf': ("len({p['trend_tf'] for p in _sel})", '{}'), 'n_confirm': ("len({p['confirm'] for p in _sel})", '{}'),
    # 11.1 VIX
    'vix_lt15_exp': ("_vx('rules', '<15', 'expectancy_R')", '{:+.3f}'), 'vix_lt15_n': ("_vx('rules', '<15', 'trades')", '{:.0f}'),
    'vix_lt15_lo': ("_vx('rules', '<15', 'exp_ci_lo')", '{:+.3f}'), 'vix_lt15_hi': ("_vx('rules', '<15', 'exp_ci_hi')", '{:+.3f}'),
    'vix_lt15_ml': ("_vx('ml', '<15', 'expectancy_R')", '{:+.3f}'),
    'vix_gt30_exp': ("_vx('rules', '>30', 'expectancy_R')", '{:+.3f}'), 'vix_gt30_n': ("_vx('rules', '>30', 'trades')", '{:.0f}'),
    'vix_gt30_lo': ("_vx('rules', '>30', 'exp_ci_lo')", '{:+.3f}'), 'vix_gt30_hi': ("_vx('rules', '>30', 'exp_ci_hi')", '{:+.3f}'),
    'vix_1520': ("_vx('rules', '15-20', 'expectancy_R')", '{:+.3f}'), 'vix_2030': ("_vx('rules', '20-30', 'expectancy_R')", '{:+.3f}'),
    'vix_cost_lt15': ("VIX_COSTS.loc['<15', 'mean_cost_R']", '{:.3f}'), 'vix_cost_gt30': ("VIX_COSTS.loc['>30', 'mean_cost_R']", '{:.3f}'),
    'vix_gross_lt15': ("VIX_COSTS.loc['<15', 'mean_gross_R']", '{:+.3f}'), 'vix_gross_gt30': ("VIX_COSTS.loc['>30', 'mean_gross_R']", '{:+.3f}'),
    # 12.3 events
    'ev_any': ("_ev('event', 'any event day', 'expectancy_R')", '{:+.3f}'), 'ev_any_n': ("_ev('event', 'any event day', 'trades')", '{:.0f}'),
    'ev_any_lo': ("_ev('event', 'any event day', 'exp_ci_lo')", '{:+.3f}'), 'ev_any_hi': ("_ev('event', 'any event day', 'exp_ci_hi')", '{:+.3f}'),
    'ev_non': ("_ev('event', 'non-event day', 'expectancy_R')", '{:+.3f}'), 'ev_non_n': ("_ev('event', 'non-event day', 'trades')", '{:.0f}'),
    'ev_non_lo': ("_ev('event', 'non-event day', 'exp_ci_lo')", '{:+.3f}'), 'ev_non_hi': ("_ev('event', 'non-event day', 'exp_ci_hi')", '{:+.3f}'),
    'ev_any_ml': ("_ev('event', 'any event day', 'expectancy_R', 'ml')", '{:+.3f}'), 'ev_any_ml_lo': ("_ev('event', 'any event day', 'exp_ci_lo', 'ml')", '{:+.3f}'),
    'cpi_n': ("_ev('event', 'CPI', 'trades')", '{:.0f}'), 'fomc_n': ("_ev('event', 'FOMC', 'trades')", '{:.0f}'),
    'fomc_win_n': ("_ev('intraday', 'FOMC ±30min', 'trades')", '{:.0f}'),
    'epu_mid': ("_ev('EPU tercile', 'T2 mid', 'expectancy_R')", '{:+.3f}'), 'epu_mid_lo': ("_ev('EPU tercile', 'T2 mid', 'exp_ci_lo')", '{:+.3f}'),
    'epu_mid_hi': ("_ev('EPU tercile', 'T2 mid', 'exp_ci_hi')", '{:+.3f}'), 'epu_low': ("_ev('EPU tercile', 'T1 low', 'expectancy_R')", '{:+.3f}'),
    'epu_high': ("_ev('EPU tercile', 'T3 high', 'expectancy_R')", '{:+.3f}'), 'lag_sessions': ('2', '{}'),
    'n_reliable_buckets': ('len(_half_rel)', '{}'),
    'n_filter_tests': ("EVENT_VERDICT_DATA['n_filter_tests']", '{}'), 'n_filter_confirmed': ("EVENT_VERDICT_DATA['n_confirmed']", '{}'),
}
# sanity: the qualitative statements in the text that are not numbers
assert all(v < 0 for v in _oos_rules_1x), 'text says rules-only OOS is negative in every split'
assert (DSR_TABLE['DSR (N split)'] < 0.95).all()
assert _vx('rules', '<15', 'exp_ci_hi') < 0 and all(_vx('rules', b, 'exp_ci_lo') <= 0 <= _vx('rules', b, 'exp_ci_hi') for b in ['15-20', '20-30', '>30'])
assert not any(_vx('rules', b, 'exp_ci_lo') > 0 for _, _, b in VIX_LEVELS) and not any(_vx('ml', b, 'exp_ci_lo') > 0 for _, _, b in VIX_LEVELS)
assert EVENT_TABLE[(EVENT_TABLE.split == PRIMARY_SPLIT) & EVENT_TABLE.dimension.str.match(r'(GPR|EMV)') & EVENT_TABLE.reliable.astype(bool)].pipe(
    lambda d: ((d.exp_ci_lo <= 0) & (d.exp_ci_hi >= 0)).all()), 'text says every EMV/GPR tercile CI includes zero'
_excl = _half_rel[(_half_rel.exp_ci_lo > 0) | (_half_rel.exp_ci_hi < 0)]
assert set(zip(_excl.dimension, _excl.bucket)) <= {('EPU tercile', 'T2 mid'), ('EPU tercile (lag 2)', 'T2 mid')} and _ev('EPU tercile (lag 2)', 'T2 mid', 'exp_ci_hi') < 0
_types = _half_rel[_half_rel.dimension == 'event']
assert ((_types.exp_ci_lo <= 0) & (_types.exp_ci_hi >= 0)).all(), 'text says every event bucket interval includes zero'
assert _ev('event', 'CPI', 'trades') < 30 and _ev('event', 'FOMC', 'trades') < 30 and _ev('event', 'any event day', 'exp_ci_lo', 'ml') < 0
assert (FILTER_TESTS['IS gain (R/trade)'] > 0).all() and len(FILTER_TESTS) == EVENT_VERDICT_DATA['n_filter_tests']
assert _wf.loc[('ml', '1x'), 'ci_lo'] < 0 < _wf.loc[('ml', '1x'), 'ci_hi']
print(f'{len(CLAIMS)} claims registered; qualitative assertions hold')

100 claims registered; qualitative assertions hold


### 13.4 Result-claim audit ("hallucination check")
The final cell does two things:
1. It **re-derives the headline numbers from scratch**. Every split's OOS trade log is reloaded from the parquet files written in §9, and costs, the daily loss limit, the leverage caps and the metrics are recomputed. The results must equal the §13.1 table.
2. It **audits every number written in the narrative cells**, i.e. markdown cells containing the marker `audit:claims`. It reads this notebook file and extracts every numeric token, stripping only structural numbers: section references (§x.y), heading and list numbers, calendar years/dates, and split/bucket labels such as IS3y or WF2020. Each token must equal, **as text including its sign and precision**, the formatted value of a registered expression in `CLAIMS`. The expressions are evaluated now, on this run's outputs. A number in the prose that no registered output produces fails the audit, and so does a registered claim whose value is no longer what the prose says.

In [44]:
import nbformat
# (1) re-derive OOS rules-only metrics from the saved trade logs
rederived = []
for name, E_ in EVAL.items():
    raw = pd.read_parquet(DIRS['trades'] / f'{name}_oos_raw.parquet')
    raw['sym'] = raw['sym'].astype('category')
    for cm in COST_MULTS:
        m_ = metrics(apply_portfolio(raw, cm, E_['p']['sizing']), E_['oos_cal'])
        ref = METRICS[(METRICS.split == name) & (METRICS.period == 'OOS') & (METRICS.costs == f'{cm:g}x') & (METRICS.variant == 'rules-only')].iloc[0]
        for k in ['trades', 'expectancy_R', 'sharpe', 'max_dd', 'profit_factor', 'win_rate']:
            ok = (pd.isna(ref[k]) and pd.isna(m_[k])) or np.isclose(ref[k], m_[k], rtol=1e-9, atol=1e-12)
            rederived.append({'split': name, 'costs': f'{cm:g}x', 'metric': k, 'table': ref[k], 'rederived': m_[k], 'match': bool(ok)})
REDERIVED = pd.DataFrame(rederived)
print(f"(1) re-derived OOS rules-only metrics: {int(REDERIVED.match.sum())}/{len(REDERIVED)} equal the §13.1 table")

# (2) narrative-claim registry: token as written  <-  (expression on this run's outputs, format)
def _c(key, col, fmt):
    return (f"CONCL.loc[{key!r}, {col!r}]", fmt)

CLAIMS = CLAIMS if 'CLAIMS' in globals() else {}

def _tok(s):
    return s.replace('−', '-').replace('–', '-').strip()

NB_PATH = PROJECT / 'research.ipynb'
cells = []
if NB_PATH.exists():
    cells = [c.source for c in nbformat.read(NB_PATH, as_version=4).cells if c.cell_type == 'markdown' and 'audit:claims' in c.source]
values = {}
for name, (expr, fmt) in CLAIMS.items():
    try:
        values[name] = _tok(fmt.format(eval(expr)))
    except Exception as e:
        values[name] = f'<error {e!r}>'
value_set = {}
for k, v in values.items():
    value_set.setdefault(v, []).append(k)
rows = []
for src in cells:
    text = re.sub(r'<!--.*?-->', ' ', src, flags=re.S)
    text = re.sub(r'^\s*#+\s*[\d.]+', ' ', text, flags=re.M)                       # heading numbers (### 13.3)
    text = re.sub(r'^\s*\d+\.\s', ' ', text, flags=re.M)                          # ordered-list markers
    text = re.sub(r'§\s?\d+(\.\d+)*[a-z]?', ' ', text)                              # section references
    text = re.sub(r'\b(19[89]\d|20[0-2]\d)(-\d\d){0,2}\b', ' ', text)                # years / ISO dates
    text = re.sub(r'\b(IS\dy|WF\d{4}|P\d+-\d+|T[123])\b', ' ', text)                 # split / bucket labels
    for mt in re.finditer(r'(?<![\w.])([-+−]?\d+(?:\.\d+)?%?)', text):
        tok = _tok(mt.group(1))
        hit = value_set.get(tok) or value_set.get(tok.lstrip('+'))
        rows.append({'token': mt.group(1), 'claims': ','.join(hit) if hit else '', 'verified': bool(hit),
                     'context': ' '.join(text[max(0, mt.start() - 50):mt.end() + 25].split())})
CLAIM_AUDIT = pd.DataFrame(rows, columns=['token', 'claims', 'verified', 'context'])
quoted = set(_tok(t) for t in CLAIM_AUDIT.token) | set(_tok(t).lstrip('+') for t in CLAIM_AUDIT.token)
orphans = [k for k, v in values.items() if v not in quoted and v.lstrip('+') not in quoted]
errors = [k for k, v in values.items() if v.startswith('<error')]
print(f"(2) {len(CLAIM_AUDIT)} numeric tokens in {len(cells)} audited narrative cells; "
      f"{int(CLAIM_AUDIT.verified.sum())} verified against {len(CLAIMS)} registered claims; "
      f"{int((~CLAIM_AUDIT.verified).sum())} unverified; {len(orphans)} registered-but-unquoted; {len(errors)} expression errors")
if (~CLAIM_AUDIT.verified).any():
    display(CLAIM_AUDIT[~CLAIM_AUDIT.verified])
if orphans or errors:
    print('orphans:', orphans, 'errors:', errors)
AUDIT_PASS = bool(REDERIVED.match.all() and len(cells) > 0 and len(CLAIM_AUDIT) > 0 and CLAIM_AUDIT.verified.all() and not errors)
print('RESULT-CLAIM AUDIT:', 'PASS' if AUDIT_PASS else 'FAIL')
RUN_INFO['finished'] = time.strftime('%Y-%m-%d %H:%M:%S')
print(json.dumps(RUN_INFO, indent=1))
mem('end of notebook')

(1) re-derived OOS rules-only metrics: 144/144 equal the §13.1 table
(2) 129 numeric tokens in 4 audited narrative cells; 129 verified against 100 registered claims; 0 unverified; 0 registered-but-unquoted; 0 expression errors
RESULT-CLAIM AUDIT: PASS
{
 "seed": 20261025,
 "fast_mode": false,
 "python": "3.13.9",
 "platform": "macOS-27.0.1-arm64-arm-64bit-Mach-O",
 "numpy": "2.3.5",
 "pandas": "2.3.3",
 "numba": "0.62.1",
 "pyarrow": "21.0.0",
 "scipy": "1.16.3",
 "plotly": "6.3.0",
 "started": "2026-10-25 23:00:38",
 "optuna": "5.0.0",
 "sklearn": "1.7.2",
 "xgboost": "3.4.1",
 "shap": "0.54.1",
 "finished": "2026-10-25 23:05:24"
}
[mem] end of notebook                            rss=   723.1 MB   peak=  1698.5 MB
